In [1]:
# ===== POLAR-EMS INTEGRATION: LOAD APPROVED ARTIFACTS =====

import joblib
import json
import pandas as pd
import numpy as np
import os

BASE = "/content"

# Models
load_model = joblib.load(f"{BASE}/polar_ems_load_forecaster.joblib")
solar_model = joblib.load(f"{BASE}/polar_ems_solar_forecaster.joblib")
wind_model = joblib.load(f"{BASE}/POLAR_EMS_Best_Wind_Model.joblib")
anomaly_model = joblib.load(f"{BASE}/polar_ems_anomaly_detector.joblib")
battery_soh_model = joblib.load(f"{BASE}/polar_ems_battery_soh_model.joblib")

# Metadata
with open(f"{BASE}/polar_ems_anomaly_detector_metadata.json", "r") as f:
    anomaly_metadata = json.load(f)

with open(f"{BASE}/polar_ems_battery_soh_metadata.json", "r") as f:
    battery_metadata = json.load(f)

# Optimizer / resupply data
optimizer_data = pd.read_csv(
    f"{BASE}/polar_ems_optimizer_synthetic_data.csv"
)

resupply_scenarios = pd.read_csv(
    f"{BASE}/polar_ems_resupply_scenarios.csv"
)

print("All approved artifacts loaded.")
print("Optimizer data:", optimizer_data.shape)
print("Resupply scenarios:", resupply_scenarios.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/content/polar_ems_load_forecaster.joblib'

In [2]:
# ============================================================
# POLAR-EMS END-TO-END INTEGRATION
# STEP 1: LOAD APPROVED MODELS AND DATA
# ============================================================

import os
import glob
import joblib
import json
import pandas as pd

BASE = "/content"

def find_file(patterns):
    """
    Search /content for the first file matching any pattern.
    """
    for pattern in patterns:
        matches = glob.glob(os.path.join(BASE, pattern))
        if matches:
            return matches[0]
    return None


# ------------------------------------------------------------
# 1. FIND MODEL FILES
# ------------------------------------------------------------

load_path = find_file([
    "*load*forecaster*.joblib",
    "*weather*load*forecaster*.joblib"
])

solar_path = find_file([
    "*solar*forecaster*.joblib"
])

wind_path = find_file([
    "POLAR_EMS_Best_Wind_Model.joblib",
    "*wind*model*.joblib"
])

anomaly_path = find_file([
    "polar_ems_anomaly_detector.joblib"
])

battery_path = find_file([
    "polar_ems_battery_soh_model.joblib"
])


# ------------------------------------------------------------
# 2. CHECK REQUIRED MODELS
# ------------------------------------------------------------

model_paths = {
    "Load Forecast": load_path,
    "Solar Forecast": solar_path,
    "Wind Forecast": wind_path,
    "SCADA Anomaly": anomaly_path,
    "Battery SOH": battery_path
}

print("MODEL FILE CHECK")
print("=" * 60)

for name, path in model_paths.items():
    if path:
        print(f"✅ {name}: {os.path.basename(path)}")
    else:
        print(f"❌ {name}: NOT FOUND")


# Stop if something critical is missing.
missing = [name for name, path in model_paths.items() if path is None]

if missing:
    raise FileNotFoundError(
        "Missing required model(s): " + ", ".join(missing)
    )


# ------------------------------------------------------------
# 3. LOAD MODELS
# ------------------------------------------------------------

load_model = joblib.load(load_path)
solar_model = joblib.load(solar_path)
wind_model = joblib.load(wind_path)
anomaly_model = joblib.load(anomaly_path)
battery_soh_model = joblib.load(battery_path)


# ------------------------------------------------------------
# 4. LOAD METADATA
# ------------------------------------------------------------

def load_json(patterns):
    path = find_file(patterns)
    if path is None:
        return None, None

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f), path


anomaly_metadata, anomaly_metadata_path = load_json([
    "polar_ems_anomaly_detector_metadata.json"
])

battery_metadata, battery_metadata_path = load_json([
    "polar_ems_battery_soh_metadata.json"
])


# ------------------------------------------------------------
# 5. FIND OPTIMIZER / RESUPPLY FILES
# ------------------------------------------------------------

optimizer_path = find_file([
    "*optimizer*synthetic*.csv"
])

resupply_path = find_file([
    "*resupply*scenarios*.csv"
])


if optimizer_path is None:
    raise FileNotFoundError("Optimizer synthetic dataset not found.")

if resupply_path is None:
    raise FileNotFoundError("Resupply scenario dataset not found.")


optimizer_data = pd.read_csv(optimizer_path)
resupply_scenarios = pd.read_csv(resupply_path)


# ------------------------------------------------------------
# 6. FINAL VERIFICATION
# ------------------------------------------------------------

print("\nDATA FILE CHECK")
print("=" * 60)

print("✅ Optimizer data:")
print("   ", os.path.basename(optimizer_path))
print("   Shape:", optimizer_data.shape)

print("\n✅ Resupply scenarios:")
print("   ", os.path.basename(resupply_path))
print("   Shape:", resupply_scenarios.shape)

print("\n" + "=" * 60)
print("✅ ALL APPROVED ARTIFACTS LOADED SUCCESSFULLY")
print("=" * 60)

MODEL FILE CHECK
✅ Load Forecast: polar_ems_weather_load_forecaster.joblib
✅ Solar Forecast: polar_ems_solar_forecaster.joblib
✅ Wind Forecast: POLAR_EMS_Best_Wind_Model.joblib
✅ SCADA Anomaly: polar_ems_anomaly_detector.joblib
✅ Battery SOH: polar_ems_battery_soh_model.joblib

DATA FILE CHECK
✅ Optimizer data:
    polar_ems_optimizer_synthetic_data (1).csv
   Shape: (2160, 24)

✅ Resupply scenarios:
    polar_ems_resupply_scenarios (1).csv
   Shape: (2000, 5)

✅ ALL APPROVED ARTIFACTS LOADED SUCCESSFULLY


In [3]:
# ============================================================
# POLAR-EMS — STEP 2: VERIFY MODEL INPUT/OUTPUT COMPATIBILITY
# ============================================================

import inspect
import json
import numpy as np
import pandas as pd

models = {
    "Load Forecast": load_model,
    "Solar Forecast": solar_model,
    "Wind Forecast": wind_model,
    "SCADA Anomaly": anomaly_model,
    "Battery SOH": battery_soh_model
}

print("=" * 70)
print("POLAR-EMS MODEL COMPATIBILITY REPORT")
print("=" * 70)

for name, model in models.items():

    print(f"\n{'='*70}")
    print(name)
    print(f"{'='*70}")

    print("Model type:")
    print(type(model))

    # Feature names if available
    if hasattr(model, "feature_names_in_"):
        print("\nExpected features:")
        print(list(model.feature_names_in_))

    # Number of expected features
    if hasattr(model, "n_features_in_"):
        print("\nExpected feature count:")
        print(model.n_features_in_)

    # Useful model attributes
    for attr in ["n_estimators", "max_depth", "objective"]:
        if hasattr(model, attr):
            print(f"{attr}: {getattr(model, attr)}")

    # Basic object structure
    print("\nAvailable prediction methods:")
    methods = []
    for method in ["predict", "predict_proba", "decision_function"]:
        if hasattr(model, method):
            methods.append(method)

    print(methods)

print("\n" + "=" * 70)
print("COMPATIBILITY CHECK COMPLETE")
print("=" * 70)

POLAR-EMS MODEL COMPATIBILITY REPORT

Load Forecast
Model type:
<class 'xgboost.sklearn.XGBRegressor'>

Expected features:
[np.str_('temp'), np.str_('dwpt'), np.str_('rhum'), np.str_('wdir'), np.str_('wspd'), np.str_('pres'), np.str_('hour'), np.str_('day_of_week'), np.str_('day_of_month'), np.str_('month'), np.str_('day_of_year'), np.str_('is_weekend'), np.str_('lag_1h'), np.str_('lag_2h'), np.str_('lag_3h'), np.str_('lag_24h'), np.str_('lag_48h'), np.str_('lag_168h'), np.str_('rolling_24h_mean'), np.str_('rolling_24h_std')]

Expected feature count:
20
n_estimators: 600
max_depth: 8
objective: reg:squarederror

Available prediction methods:
['predict']

Solar Forecast
Model type:
<class 'xgboost.sklearn.XGBRegressor'>

Expected features:
[np.str_('DC_POWER'), np.str_('DAILY_YIELD'), np.str_('hour'), np.str_('day_of_week'), np.str_('day_of_year'), np.str_('month'), np.str_('lag_1h'), np.str_('lag_2h'), np.str_('lag_3h'), np.str_('lag_24h'), np.str_('rolling_6h_mean'), np.str_('rolling_

In [4]:
# ============================================================
# STEP 3 — PRINT COMPLETE FEATURE REQUIREMENTS
# ============================================================

def show_features(name, model):
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    if hasattr(model, "feature_names_in_"):
        features = list(model.feature_names_in_)
        print("Feature count:", len(features))

        for i, f in enumerate(features, start=1):
            print(f"{i:02d}. {f}")
    else:
        print("No feature_names_in_ found.")

show_features("LOAD FORECAST", load_model)
show_features("SOLAR FORECAST", solar_model)
show_features("WIND FORECAST", wind_model)
show_features("SCADA ANOMALY", anomaly_model)
show_features("BATTERY SOH", battery_soh_model)


LOAD FORECAST
Feature count: 20
01. temp
02. dwpt
03. rhum
04. wdir
05. wspd
06. pres
07. hour
08. day_of_week
09. day_of_month
10. month
11. day_of_year
12. is_weekend
13. lag_1h
14. lag_2h
15. lag_3h
16. lag_24h
17. lag_48h
18. lag_168h
19. rolling_24h_mean
20. rolling_24h_std

SOLAR FORECAST
Feature count: 12
01. DC_POWER
02. DAILY_YIELD
03. hour
04. day_of_week
05. day_of_year
06. month
07. lag_1h
08. lag_2h
09. lag_3h
10. lag_24h
11. rolling_6h_mean
12. rolling_24h_mean

WIND FORECAST
Feature count: 75
01. LV ActivePower (kW)
02. Wind Speed (m/s)
03. Theoretical_Power_Curve (KWh)
04. Wind Direction (°)
05. hour
06. minute
07. dayofweek
08. dayofyear
09. month
10. hour_sin
11. hour_cos
12. dayofyear_sin
13. dayofyear_cos
14. direction_sin
15. direction_cos
16. LV ActivePower (kW)_lag_1
17. Wind Speed (m/s)_lag_1
18. Theoretical_Power_Curve (KWh)_lag_1
19. Wind Direction (°)_lag_1
20. LV ActivePower (kW)_lag_2
21. Wind Speed (m/s)_lag_2
22. Theoretical_Power_Curve (KWh)_lag_2
23. W

In [5]:
# ============================================================
# POLAR-EMS — STEP 3
# FEATURE ADAPTER DEFINITIONS
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# LOAD FORECAST — 20 FEATURES
# ------------------------------------------------------------

LOAD_FEATURES = [
    "temp",
    "dwpt",
    "rhum",
    "wdir",
    "wspd",
    "pres",
    "hour",
    "day_of_week",
    "day_of_month",
    "month",
    "day_of_year",
    "is_weekend",
    "lag_1h",
    "lag_2h",
    "lag_3h",
    "lag_24h",
    "lag_48h",
    "lag_168h",
    "rolling_24h_mean",
    "rolling_24h_std",
]


def build_load_features(df):
    """
    Build exactly the 20 features expected by the approved
    load forecasting model.
    """

    x = df.copy()

    x["timestamp"] = pd.to_datetime(x["timestamp"])

    x["hour"] = x["timestamp"].dt.hour
    x["day_of_week"] = x["timestamp"].dt.dayofweek
    x["day_of_month"] = x["timestamp"].dt.day
    x["month"] = x["timestamp"].dt.month
    x["day_of_year"] = x["timestamp"].dt.dayofyear
    x["is_weekend"] = (x["day_of_week"] >= 5).astype(int)

    # These must be based on the same source load variable used
    # during training.
    if "load_kw" not in x.columns:
        raise ValueError("build_load_features requires 'load_kw'.")

    x["lag_1h"] = x["load_kw"].shift(1)
    x["lag_2h"] = x["load_kw"].shift(2)
    x["lag_3h"] = x["load_kw"].shift(3)
    x["lag_24h"] = x["load_kw"].shift(24)
    x["lag_48h"] = x["load_kw"].shift(48)
    x["lag_168h"] = x["load_kw"].shift(168)

    x["rolling_24h_mean"] = x["load_kw"].rolling(24).mean()
    x["rolling_24h_std"] = x["load_kw"].rolling(24).std()

    missing = [f for f in LOAD_FEATURES if f not in x.columns]
    if missing:
        raise ValueError(f"Missing load features: {missing}")

    return x[LOAD_FEATURES]


# ------------------------------------------------------------
# BATTERY SOH — 8 FEATURES
# ------------------------------------------------------------

BATTERY_FEATURES = [
    "Voltage",
    "Current",
    "Temperature",
    "ChargeTime",
    "DischargeTime",
    "InternalResistance",
    "AmbientHumidity",
    "C_Rate",
]


def build_battery_features(state):
    """
    Construct the exact 8 features expected by the
    approved Battery SOH model.
    """

    row = {
        "Voltage": state["battery_voltage"],
        "Current": state["battery_current"],
        "Temperature": state["battery_temperature"],
        "ChargeTime": state["battery_charge_time"],
        "DischargeTime": state["battery_discharge_time"],
        "InternalResistance": state["battery_internal_resistance"],
        "AmbientHumidity": state["ambient_humidity"],
        "C_Rate": state["battery_c_rate"],
    }

    return pd.DataFrame([row])[BATTERY_FEATURES]


# ------------------------------------------------------------
# SCADA ANOMALY — 23 FEATURES
# ------------------------------------------------------------

SCADA_FEATURES = [
    "Voltage In",
    "Voltage DC Bus",
    "Voltage L1",
    "Voltage L2",
    "voltage rise",
    "min v from rpm",
    "Current out",
    "Power out",
    "Power reg",
    "Power max",
    "Line frequency",
    "Inverter Frequency",
    "Line Resistance",
    "RPM",
    "Windspeed (ref)",
    "TargetTSR",
    "Ramp RPM",
    "Boost puldwidth",
    "Max BPW",
    "current amplitude",
    "T1",
    "T2",
    "T3",
]


def build_scada_features(df):
    """
    Select the exact 23 SCADA features expected by the
    approved Isolation Forest model.
    """

    missing = [f for f in SCADA_FEATURES if f not in df.columns]

    if missing:
        raise ValueError(
            "Missing SCADA features:\n" + "\n".join(missing)
        )

    return df[SCADA_FEATURES].copy()


print("✅ Feature adapter definitions created.")
print("Load features:", len(LOAD_FEATURES))
print("Battery features:", len(BATTERY_FEATURES))
print("SCADA features:", len(SCADA_FEATURES))
print("Wind features: 75 (already verified from saved model)")

✅ Feature adapter definitions created.
Load features: 20
Battery features: 8
SCADA features: 23
Wind features: 75 (already verified from saved model)


In [6]:
# ============================================================
# POLAR-EMS — STEP 6
# UNIFIED STATION STATE
# ============================================================

from dataclasses import dataclass, asdict
from typing import Optional
import numpy as np


@dataclass
class StationState:
    # --------------------------------------------------------
    # Time
    # --------------------------------------------------------
    timestamp: str

    # --------------------------------------------------------
    # Energy
    # --------------------------------------------------------
    load_kw: float
    critical_load_kw: float
    flexible_load_kw: float

    solar_kw: float
    wind_kw: float
    renewable_kw: float
    net_load_kw: float

    # --------------------------------------------------------
    # Battery
    # --------------------------------------------------------
    battery_soc_pct: float
    battery_soh_pct: float
    battery_usable_capacity_kwh: float
    battery_energy_kwh: float

    # --------------------------------------------------------
    # Generator / fuel
    # --------------------------------------------------------
    generator_available_kw: float
    generator_min_kw: float
    fuel_remaining_l: float

    # --------------------------------------------------------
    # Environmental / operational state
    # --------------------------------------------------------
    temperature_c: float
    wind_speed_ms: float
    storm_flag: bool
    low_renewable_flag: bool

    # --------------------------------------------------------
    # Communication
    # --------------------------------------------------------
    communication_status: str

    # --------------------------------------------------------
    # SCADA anomaly
    # --------------------------------------------------------
    scada_anomaly_score: Optional[float]
    scada_anomaly_flag: bool

    # --------------------------------------------------------
    # Resupply
    # --------------------------------------------------------
    resupply_p10_days: float
    resupply_p50_days: float
    resupply_p90_days: float

    # --------------------------------------------------------
    # Risk / operability
    # --------------------------------------------------------
    safe_operability_days: Optional[float] = None
    resupply_margin_days: Optional[float] = None
    cqrm: Optional[float] = None
    risk_level: Optional[str] = None


def create_station_state(
    timestamp,
    load_kw,
    solar_kw,
    wind_kw,
    battery_soc_pct,
    battery_soh_pct,
    battery_usable_capacity_kwh,
    battery_energy_kwh,
    generator_available_kw,
    generator_min_kw,
    fuel_remaining_l,
    temperature_c,
    wind_speed_ms,
    storm_flag,
    low_renewable_flag,
    communication_status,
    scada_anomaly_score,
    scada_anomaly_flag,
    resupply_p10_days,
    resupply_p50_days,
    resupply_p90_days,
    critical_load_ratio=0.70,
):
    """
    Create a unified POLAR-EMS station state.

    critical_load_ratio is a prototype assumption used only
    when an explicit critical-load value is unavailable.
    """

    load_kw = float(load_kw)
    solar_kw = float(solar_kw)
    wind_kw = float(wind_kw)

    critical_load_kw = load_kw * critical_load_ratio
    flexible_load_kw = max(0.0, load_kw - critical_load_kw)

    renewable_kw = max(0.0, solar_kw + wind_kw)
    net_load_kw = max(0.0, load_kw - renewable_kw)

    state = StationState(
        timestamp=str(timestamp),

        load_kw=load_kw,
        critical_load_kw=critical_load_kw,
        flexible_load_kw=flexible_load_kw,

        solar_kw=solar_kw,
        wind_kw=wind_kw,
        renewable_kw=renewable_kw,
        net_load_kw=net_load_kw,

        battery_soc_pct=float(battery_soc_pct),
        battery_soh_pct=float(battery_soh_pct),
        battery_usable_capacity_kwh=float(
            battery_usable_capacity_kwh
        ),
        battery_energy_kwh=float(battery_energy_kwh),

        generator_available_kw=float(
            generator_available_kw
        ),
        generator_min_kw=float(generator_min_kw),
        fuel_remaining_l=float(fuel_remaining_l),

        temperature_c=float(temperature_c),
        wind_speed_ms=float(wind_speed_ms),

        storm_flag=bool(storm_flag),
        low_renewable_flag=bool(low_renewable_flag),

        communication_status=str(
            communication_status
        ),

        scada_anomaly_score=(
            None if scada_anomaly_score is None
            else float(scada_anomaly_score)
        ),
        scada_anomaly_flag=bool(
            scada_anomaly_flag
        ),

        resupply_p10_days=float(resupply_p10_days),
        resupply_p50_days=float(resupply_p50_days),
        resupply_p90_days=float(resupply_p90_days),
    )

    return state


# ============================================================
# TEST WITH A CONTROLLED SYNTHETIC STATION STATE
# ============================================================

test_state = create_station_state(
    timestamp="2018-01-01 00:00",

    load_kw=420.0,
    solar_kw=80.0,
    wind_kw=160.0,

    battery_soc_pct=70.0,
    battery_soh_pct=94.0,

    battery_usable_capacity_kwh=700.0,
    battery_energy_kwh=490.0,

    generator_available_kw=500.0,
    generator_min_kw=100.0,

    fuel_remaining_l=1500.0,

    temperature_c=-18.0,
    wind_speed_ms=8.5,

    storm_flag=False,
    low_renewable_flag=False,

    communication_status="LOCAL",

    scada_anomaly_score=0.05,
    scada_anomaly_flag=False,

    resupply_p10_days=8.0,
    resupply_p50_days=10.0,
    resupply_p90_days=13.0,
)

print("=" * 80)
print("POLAR-EMS UNIFIED STATION STATE")
print("=" * 80)

for key, value in asdict(test_state).items():
    print(f"{key:35s}: {value}")

print("\n" + "=" * 80)
print("✅ UNIFIED STATION STATE CREATED")
print("=" * 80)

POLAR-EMS UNIFIED STATION STATE
timestamp                          : 2018-01-01 00:00
load_kw                            : 420.0
critical_load_kw                   : 294.0
flexible_load_kw                   : 126.0
solar_kw                           : 80.0
wind_kw                            : 160.0
renewable_kw                       : 240.0
net_load_kw                        : 180.0
battery_soc_pct                    : 70.0
battery_soh_pct                    : 94.0
battery_usable_capacity_kwh        : 700.0
battery_energy_kwh                 : 490.0
generator_available_kw             : 500.0
generator_min_kw                   : 100.0
fuel_remaining_l                   : 1500.0
temperature_c                      : -18.0
wind_speed_ms                      : 8.5
storm_flag                         : False
low_renewable_flag                 : False
communication_status               : LOCAL
scada_anomaly_score                : 0.05
scada_anomaly_flag                 : False
resupply_p10_day

In [8]:
# ============================================================
# POLAR-EMS — STEP 7
# SAFE OPERABILITY ENGINE
# ============================================================

import numpy as np
import pandas as pd
from dataclasses import asdict


# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

# Prototype safety thresholds already established in Model 7.
MIN_SOC_PCT = 20.0
MIN_SOH_PCT = 70.0
MIN_FUEL_L = 800.0
MIN_GENERATOR_AVAILABLE_KW = 250.0

# Prototype conversion used only when no explicit generator
# fuel-rate field exists in the scenario data.
#
# IMPORTANT:
# This is a configurable prototype assumption, NOT a
# real Antarctic diesel specification.
FUEL_TO_ENERGY_KWH_PER_L = 3.0


# ------------------------------------------------------------
# HELPER: FIND FUTURE SCENARIO DATA
# ------------------------------------------------------------

def prepare_future_profile(
    optimizer_df,
    start_timestamp=None,
    horizon_hours=30 * 24
):
    """
    Prepare the future hourly operating profile from the
    optimizer dataset.

    The function tries to start at the requested timestamp.
    If that timestamp is not available, it uses the beginning
    of the available scenario dataset.
    """

    df = optimizer_df.copy()

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        errors="coerce"
    )

    df = (
        df.dropna(subset=["timestamp"])
          .sort_values("timestamp")
          .reset_index(drop=True)
    )

    if start_timestamp is not None:

        ts = pd.to_datetime(start_timestamp)

        future = df[df["timestamp"] >= ts].copy()

        if len(future) == 0:
            future = df.copy()
    else:
        future = df.copy()

    return future.head(horizon_hours).reset_index(drop=True)


# ------------------------------------------------------------
# SAFE OPERABILITY CALCULATOR
# ------------------------------------------------------------

def calculate_safe_operability(
    state,
    optimizer_df,
    start_timestamp=None,
    horizon_hours=30 * 24,
    fuel_to_energy_kwh_per_l=FUEL_TO_ENERGY_KWH_PER_L
):
    """
    Estimate safe operating horizon.

    The calculation tracks:
      - battery energy
      - minimum SOC
      - critical-load coverage
      - renewable contribution
      - generator capability
      - fuel reserve
      - battery SOH
      - future renewable/load conditions

    This is a prototype operational-risk calculation,
    not a certified engineering model.
    """

    profile = prepare_future_profile(
        optimizer_df,
        start_timestamp=start_timestamp,
        horizon_hours=horizon_hours
    )

    if profile.empty:
        raise ValueError(
            "No future operating profile available."
        )

    # --------------------------------------------------------
    # Initial state
    # --------------------------------------------------------

    battery_capacity = max(
        0.0,
        float(state.battery_usable_capacity_kwh)
    )

    battery_energy = np.clip(
        float(state.battery_energy_kwh),
        0.0,
        battery_capacity
    )

    fuel_remaining = max(
        0.0,
        float(state.fuel_remaining_l)
    )

    soh_pct = float(state.battery_soh_pct)

    generator_available = float(
        state.generator_available_kw
    )

    # Critical-load requirement
    critical_load = max(
        0.0,
        float(state.critical_load_kw)
    )

    # Minimum allowed battery energy
    min_battery_energy = (
        battery_capacity *
        MIN_SOC_PCT / 100.0
    )

    # --------------------------------------------------------
    # Result tracking
    # --------------------------------------------------------

    safe_hours = 0
    first_violation = None

    hourly_records = []

    # --------------------------------------------------------
    # Simulate each future hour
    # --------------------------------------------------------

    for _, row in profile.iterrows():

        timestamp = row["timestamp"]

        # Future forecast values
        load = max(
            0.0,
            float(row.get(
                "load_forecast_kw",
                load if "load" in locals() else 0.0
            ))
        )

        critical = max(
            0.0,
            float(row.get(
                "critical_load_kw",
                critical_load
            ))
        )

        solar = max(
            0.0,
            float(row.get(
                "solar_forecast_kw",
                0.0
            ))
        )

        wind = max(
            0.0,
            float(row.get(
                "wind_forecast_kw",
                0.0
            ))
        )

        renewable = max(
            0.0,
            solar + wind
        )

        generator_capacity = max(
            0.0,
            float(row.get(
                "generator_available_kw",
                generator_available
            ))
        )

        # ----------------------------------------------------
        # Critical-load energy requirement
        # ----------------------------------------------------

        critical_net = max(
            0.0,
            critical - renewable
        )

        # ----------------------------------------------------
        # Generator can serve critical load
        # ----------------------------------------------------

        generator_used = min(
            critical_net,
            generator_capacity
        )

        remaining_critical = max(
            0.0,
            critical_net - generator_used
        )

        # ----------------------------------------------------
        # Battery can cover remaining requirement
        # ----------------------------------------------------

        battery_available = max(
            0.0,
            battery_energy - min_battery_energy
        )

        battery_used = min(
            remaining_critical,
            battery_available
        )

        remaining_critical = max(
            0.0,
            remaining_critical - battery_used
        )

        # ----------------------------------------------------
        # Fuel consumption
        # ----------------------------------------------------

        fuel_used = (
            generator_used /
            max(fuel_to_energy_kwh_per_l, 1e-9)
        )

        fuel_remaining -= fuel_used

        # ----------------------------------------------------
        # Battery state update
        # ----------------------------------------------------

        battery_energy -= battery_used

        battery_energy = max(
            0.0,
            battery_energy
        )

        soc_pct = (
            100.0 * battery_energy /
            max(battery_capacity, 1e-9)
        )

        # ----------------------------------------------------
        # Safety checks
        # ----------------------------------------------------

        violations = []

        if soh_pct < MIN_SOH_PCT:
            violations.append(
                "LOW_BATTERY_SOH"
            )

        if soc_pct < MIN_SOC_PCT:
            violations.append(
                "LOW_BATTERY_SOC"
            )

        if generator_capacity < MIN_GENERATOR_AVAILABLE_KW:
            violations.append(
                "INSUFFICIENT_GENERATOR_AVAILABILITY"
            )

        if fuel_remaining < MIN_FUEL_L:
            violations.append(
                "LOW_FUEL_RESERVE"
            )

        if remaining_critical > 0:
            violations.append(
                "CRITICAL_LOAD_CANNOT_BE_SERVED"
            )

        if violations and first_violation is None:
            first_violation = {
                "timestamp": str(timestamp),
                "violations": violations
            }

        if not violations:
            safe_hours += 1

        hourly_records.append({
            "timestamp": timestamp,
            "load_kw": load,
            "critical_load_kw": critical,
            "solar_kw": solar,
            "wind_kw": wind,
            "renewable_kw": renewable,
            "generator_used_kw": generator_used,
            "battery_used_kwh": battery_used,
            "battery_energy_kwh": battery_energy,
            "battery_soc_pct": soc_pct,
            "fuel_remaining_l": fuel_remaining,
            "violations": violations
        })

        if violations:
            break

    # --------------------------------------------------------
    # Convert safe hours to days
    # --------------------------------------------------------

    safe_operability_days = safe_hours / 24.0

    result = {
        "safe_operability_days": float(
            safe_operability_days
        ),

        "safe_hours": int(safe_hours),

        "initial_battery_energy_kwh": float(
            state.battery_energy_kwh
        ),

        "final_battery_energy_kwh": float(
            battery_energy
        ),

        "final_battery_soc_pct": float(
            100.0 * battery_energy /
            max(battery_capacity, 1e-9)
        ),

        "initial_fuel_l": float(
            state.fuel_remaining_l
        ),

        "final_fuel_l": float(
            fuel_remaining
        ),

        "first_violation": first_violation,

        "status": (
            "SAFE"
            if first_violation is None
            else "AT_RISK"
        ),

        "hourly_trace": pd.DataFrame(
            hourly_records
        )
    }

    return result


# ============================================================
# RUN SAFE OPERABILITY ON TEST STATE
# ============================================================

safe_op = calculate_safe_operability(
    state=test_state,
    optimizer_df=optimizer_data,
    start_timestamp=test_state.timestamp,
    horizon_hours=30 * 24
)


# ============================================================
# DISPLAY RESULT
# ============================================================

print("=" * 80)
print("POLAR-EMS SAFE OPERABILITY RESULT")
print("=" * 80)

print(
    f"Safe operability horizon : "
    f"{safe_op['safe_operability_days']:.2f} days"
)

print(
    f"Safe hours                : "
    f"{safe_op['safe_hours']}"
)

print(
    f"Final battery energy      : "
    f"{safe_op['final_battery_energy_kwh']:.2f} kWh"
)

print(
    f"Final battery SOC         : "
    f"{safe_op['final_battery_soc_pct']:.2f}%"
)

print(
    f"Final fuel                : "
    f"{safe_op['final_fuel_l']:.2f} L"
)

print(
    f"Status                    : "
    f"{safe_op['status']}"
)

if safe_op["first_violation"] is not None:

    print("\nFIRST VIOLATION")
    print("-" * 80)

    print(
        "Timestamp:",
        safe_op["first_violation"]["timestamp"]
    )

    print(
        "Violations:",
        safe_op["first_violation"]["violations"]
    )

else:

    print(
        "\n✅ No safety violation within the "
        "30-day simulated horizon."
    )


print("\n" + "=" * 80)
print("✅ SAFE OPERABILITY ENGINE TEST COMPLETE")
print("=" * 80)

POLAR-EMS SAFE OPERABILITY RESULT
Safe operability horizon : 1.92 days
Safe hours                : 46
Final battery energy      : 490.00 kWh
Final battery SOC         : 70.00%
Final fuel                : 779.94 L
Status                    : AT_RISK

FIRST VIOLATION
--------------------------------------------------------------------------------
Timestamp: 2026-01-02 22:00:00
Violations: ['LOW_FUEL_RESERVE']

✅ SAFE OPERABILITY ENGINE TEST COMPLETE


In [9]:
# ============================================================
# POLAR-EMS — STEP 8
# CQRM / RESUPPLY MARGIN ENGINE
# ============================================================

def calculate_cqrm(
    safe_operability_days,
    resupply_p10_days,
    resupply_p50_days,
    resupply_p90_days
):
    """
    Calculate POLAR-EMS Resupply Risk / CQRM.

    Core idea:
        Safe Operability Horizon - Resupply Arrival Quantile

    We use the conservative P90 resupply estimate for the
    primary safety margin.

    CQRM is an operator-facing risk indicator for the prototype.
    """

    safe_days = float(safe_operability_days)

    p10 = float(resupply_p10_days)
    p50 = float(resupply_p50_days)
    p90 = float(resupply_p90_days)

    # --------------------------------------------------------
    # Resupply margins
    # --------------------------------------------------------

    margin_p10 = safe_days - p10
    margin_p50 = safe_days - p50
    margin_p90 = safe_days - p90

    # Primary conservative margin
    cqrm = margin_p90

    # --------------------------------------------------------
    # Risk classification
    # --------------------------------------------------------

    if cqrm > 2.0:
        risk_level = "SAFE"

    elif cqrm > 0.0:
        risk_level = "CAUTION"

    elif cqrm > -2.0:
        risk_level = "CONSERVE"

    else:
        risk_level = "CRITICAL"

    # --------------------------------------------------------
    # Operational interpretation
    # --------------------------------------------------------

    if risk_level == "SAFE":
        recommendation = (
            "Current safe-operability horizon exceeds "
            "the conservative resupply estimate."
        )

    elif risk_level == "CAUTION":
        recommendation = (
            "Maintain additional reserve and avoid "
            "unnecessary battery discharge."
        )

    elif risk_level == "CONSERVE":
        recommendation = (
            "Conserve energy, increase reserve protection, "
            "and prepare for delayed resupply."
        )

    else:
        recommendation = (
            "Current energy horizon is insufficient for "
            "the conservative resupply scenario."
        )

    return {
        "safe_operability_days": safe_days,

        "resupply_p10_days": p10,
        "resupply_p50_days": p50,
        "resupply_p90_days": p90,

        "margin_p10_days": margin_p10,
        "margin_p50_days": margin_p50,
        "margin_p90_days": margin_p90,

        "cqrm": cqrm,
        "risk_level": risk_level,
        "recommendation": recommendation
    }


# ============================================================
# RUN CQRM USING THE SAFE OPERABILITY RESULT
# ============================================================

cqrm_result = calculate_cqrm(
    safe_operability_days=safe_op["safe_operability_days"],

    resupply_p10_days=test_state.resupply_p10_days,
    resupply_p50_days=test_state.resupply_p50_days,
    resupply_p90_days=test_state.resupply_p90_days
)


# ============================================================
# DISPLAY
# ============================================================

print("=" * 80)
print("POLAR-EMS CQRM / RESUPPLY MARGIN")
print("=" * 80)

print(
    f"Safe Operability : "
    f"{cqrm_result['safe_operability_days']:.2f} days"
)

print(
    f"P10 Resupply     : "
    f"{cqrm_result['resupply_p10_days']:.2f} days"
)

print(
    f"P50 Resupply     : "
    f"{cqrm_result['resupply_p50_days']:.2f} days"
)

print(
    f"P90 Resupply     : "
    f"{cqrm_result['resupply_p90_days']:.2f} days"
)

print("\nRESUPPLY MARGINS")
print("-" * 80)

print(
    f"P10 Margin       : "
    f"{cqrm_result['margin_p10_days']:.2f} days"
)

print(
    f"P50 Margin       : "
    f"{cqrm_result['margin_p50_days']:.2f} days"
)

print(
    f"P90 Margin       : "
    f"{cqrm_result['margin_p90_days']:.2f} days"
)

print("\nCQRM")
print("-" * 80)

print(
    f"CQRM             : "
    f"{cqrm_result['cqrm']:.2f} days"
)

print(
    f"Risk Level       : "
    f"{cqrm_result['risk_level']}"
)

print(
    f"Recommendation   : "
    f"{cqrm_result['recommendation']}"
)

print("\n" + "=" * 80)
print("✅ CQRM ENGINE TEST COMPLETE")
print("=" * 80)

POLAR-EMS CQRM / RESUPPLY MARGIN
Safe Operability : 1.92 days
P10 Resupply     : 8.00 days
P50 Resupply     : 10.00 days
P90 Resupply     : 13.00 days

RESUPPLY MARGINS
--------------------------------------------------------------------------------
P10 Margin       : -6.08 days
P50 Margin       : -8.08 days
P90 Margin       : -11.08 days

CQRM
--------------------------------------------------------------------------------
CQRM             : -11.08 days
Risk Level       : CRITICAL
Recommendation   : Current energy horizon is insufficient for the conservative resupply scenario.

✅ CQRM ENGINE TEST COMPLETE


In [10]:
# ============================================================
# POLAR-EMS — STEP 9
# CQRM → RESUPPLY-AWARE OPTIMIZER
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# CQRM-BASED RESERVE POLICY
# ------------------------------------------------------------

def calculate_cqrm_reserve_policy(cqrm_result, battery_capacity_kwh):
    """
    Convert CQRM into an operational reserve requirement.

    This is a prototype policy for the integration/demo layer.
    It does NOT replace the approved optimizer's hard safety rules.
    """

    cqrm = float(cqrm_result["cqrm"])
    risk_level = cqrm_result["risk_level"]

    # Base reserve from the previously approved prototype policy.
    #
    # More severe negative CQRM -> larger reserve requirement.
    if risk_level == "SAFE":
        reserve_soc_pct = 45.0

    elif risk_level == "CAUTION":
        reserve_soc_pct = 55.0

    elif risk_level == "CONSERVE":
        reserve_soc_pct = 65.0

    else:  # CRITICAL
        reserve_soc_pct = 75.0

    # Additional pressure when CQRM is deeply negative.
    if cqrm < 0:
        additional_reserve = min(
            10.0,
            abs(cqrm) * 0.5
        )
        reserve_soc_pct += additional_reserve

    # Never exceed the configured prototype maximum.
    reserve_soc_pct = float(
        np.clip(
            reserve_soc_pct,
            20.0,
            85.0
        )
    )

    reserve_energy_kwh = (
        battery_capacity_kwh *
        reserve_soc_pct / 100.0
    )

    return {
        "cqrm_days": cqrm,
        "risk_level": risk_level,
        "required_reserve_soc_pct": reserve_soc_pct,
        "required_reserve_energy_kwh": reserve_energy_kwh
    }


# ------------------------------------------------------------
# USE CURRENT TEST STATE
# ------------------------------------------------------------

reserve_policy = calculate_cqrm_reserve_policy(
    cqrm_result=cqrm_result,
    battery_capacity_kwh=test_state.battery_usable_capacity_kwh
)


print("=" * 80)
print("CQRM-DRIVEN RESERVE POLICY")
print("=" * 80)

print(
    f"CQRM                    : "
    f"{reserve_policy['cqrm_days']:.2f} days"
)

print(
    f"Risk Level              : "
    f"{reserve_policy['risk_level']}"
)

print(
    f"Required Reserve SOC    : "
    f"{reserve_policy['required_reserve_soc_pct']:.2f}%"
)

print(
    f"Required Reserve Energy : "
    f"{reserve_policy['required_reserve_energy_kwh']:.2f} kWh"
)


# ------------------------------------------------------------
# OPTIMIZER BRIDGE
# ------------------------------------------------------------

def build_optimizer_constraints(
    state,
    reserve_policy
):
    """
    Build the operating constraints that should be passed
    to the resupply-aware optimizer.

    The optimizer remains responsible for creating the plan.
    """

    constraints = {

        "minimum_battery_soc_pct":
            reserve_policy["required_reserve_soc_pct"],

        "minimum_battery_energy_kwh":
            reserve_policy["required_reserve_energy_kwh"],

        "critical_load_required":
            True,

        "minimum_generator_available_kw":
            250.0,

        "minimum_fuel_reserve_l":
            800.0,

        "maximum_battery_discharge_kw":
            240.0,

        "communication_status":
            state.communication_status,

        "risk_level":
            reserve_policy["risk_level"],

        "cqrm_days":
            reserve_policy["cqrm_days"],
    }

    return constraints


optimizer_constraints = build_optimizer_constraints(
    state=test_state,
    reserve_policy=reserve_policy
)


print("\n" + "=" * 80)
print("OPTIMIZER CONSTRAINT BRIDGE")
print("=" * 80)

for key, value in optimizer_constraints.items():
    print(f"{key:40s}: {value}")


# ------------------------------------------------------------
# DEMONSTRATE CQRM IMPACT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CQRM IMPACT TEST")
print("=" * 80)


# Create artificial comparison cases ONLY to test the
# reserve-policy behavior.

test_cqrm_cases = [
    {
        "cqrm": 5.0,
        "risk_level": "SAFE"
    },
    {
        "cqrm": 1.0,
        "risk_level": "CAUTION"
    },
    {
        "cqrm": -1.0,
        "risk_level": "CONSERVE"
    },
    {
        "cqrm": -5.0,
        "risk_level": "CRITICAL"
    },
    {
        "cqrm": -11.08,
        "risk_level": "CRITICAL"
    }
]

comparison = []

for case in test_cqrm_cases:

    result = calculate_cqrm_reserve_policy(
        cqrm_result={
            "cqrm": case["cqrm"],
            "risk_level": case["risk_level"]
        },
        battery_capacity_kwh=
            test_state.battery_usable_capacity_kwh
    )

    comparison.append({
        "CQRM_days": case["cqrm"],
        "Risk_Level": case["risk_level"],
        "Required_Reserve_SOC_pct":
            result["required_reserve_soc_pct"],
        "Required_Reserve_kWh":
            result["required_reserve_energy_kwh"]
    })


comparison_df = pd.DataFrame(comparison)

print(comparison_df.to_string(index=False))


# ------------------------------------------------------------
# BASIC SANITY CHECK
# ------------------------------------------------------------

assert (
    comparison_df[
        comparison_df["CQRM_days"] == 5.0
    ]["Required_Reserve_SOC_pct"].iloc[0]
    <
    comparison_df[
        comparison_df["CQRM_days"] == -11.08
    ]["Required_Reserve_SOC_pct"].iloc[0]
)

print("\n✅ Lower CQRM produces a more conservative reserve policy.")

print("=" * 80)
print("✅ STEP 9 CQRM → OPTIMIZER BRIDGE COMPLETE")
print("=" * 80)

CQRM-DRIVEN RESERVE POLICY
CQRM                    : -11.08 days
Risk Level              : CRITICAL
Required Reserve SOC    : 80.54%
Required Reserve Energy : 563.79 kWh

OPTIMIZER CONSTRAINT BRIDGE
minimum_battery_soc_pct                 : 80.54166666666667
minimum_battery_energy_kwh              : 563.7916666666667
critical_load_required                  : True
minimum_generator_available_kw          : 250.0
minimum_fuel_reserve_l                  : 800.0
maximum_battery_discharge_kw            : 240.0
communication_status                    : LOCAL
risk_level                              : CRITICAL
cqrm_days                               : -11.083333333333334

CQRM IMPACT TEST
 CQRM_days Risk_Level  Required_Reserve_SOC_pct  Required_Reserve_kWh
      5.00       SAFE                     45.00                315.00
      1.00    CAUTION                     55.00                385.00
     -1.00   CONSERVE                     65.50                458.50
     -5.00   CRITICAL          

In [11]:
# ============================================================
# POLAR-EMS — STEP 10
# CQRM → ACTUAL OPTIMIZER
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# CHECK THAT APPROVED OPTIMIZER EXISTS
# ------------------------------------------------------------

if "optimize_station_v2" not in globals():
    print("""
⚠️ optimize_station_v2() is not currently loaded.

We need the approved Model-6 optimizer function in this notebook
before continuing.

Do NOT rewrite the optimizer from scratch here.

Copy/import the exact approved optimize_station_v2() function
from the Model-6 optimizer notebook/code, then rerun this cell.
""")
else:

    print("✅ Approved optimize_station_v2() found.")


# ------------------------------------------------------------
# HELPER: RUN ONE OPTIMIZER CASE
# ------------------------------------------------------------

def run_cqrm_optimizer_case(
    state,
    safe_operability_result,
    p10_days,
    p50_days,
    p90_days,
    label
):
    """
    Run the complete chain:

        Safe Operability
             ↓
           CQRM
             ↓
       Reserve Policy
             ↓
          Optimizer

    The underlying optimizer remains the approved Model-6
    optimization function.
    """

    # --------------------------------------------------------
    # CQRM
    # --------------------------------------------------------

    cqrm = calculate_cqrm(
        safe_operability_days=
            safe_operability_result["safe_operability_days"],

        resupply_p10_days=p10_days,
        resupply_p50_days=p50_days,
        resupply_p90_days=p90_days
    )

    # --------------------------------------------------------
    # CQRM-derived reserve policy
    # --------------------------------------------------------

    reserve = calculate_cqrm_reserve_policy(
        cqrm_result=cqrm,
        battery_capacity_kwh=
            state.battery_usable_capacity_kwh
    )

    # --------------------------------------------------------
    # Optimizer constraints
    # --------------------------------------------------------

    constraints = build_optimizer_constraints(
        state=state,
        reserve_policy=reserve
    )

    # --------------------------------------------------------
    # IMPORTANT
    # --------------------------------------------------------
    #
    # The approved optimizer may have a specific function
    # signature from Model 6.
    #
    # We inspect it rather than guessing.
    # --------------------------------------------------------

    import inspect

    signature = inspect.signature(
        optimize_station_v2
    )

    print("\n" + "-" * 80)
    print(f"OPTIMIZER CASE: {label}")
    print("-" * 80)

    print("Approved optimizer signature:")
    print(signature)

    print("\nCQRM:")
    print(f"  {cqrm['cqrm']:.2f} days")

    print("Risk:")
    print(f"  {cqrm['risk_level']}")

    print("Required reserve:")
    print(
        f"  {reserve['required_reserve_soc_pct']:.2f}%"
    )

    print(
        f"  {reserve['required_reserve_energy_kwh']:.2f} kWh"
    )

    print("\n✅ CQRM information prepared for optimizer.")

    return {
        "label": label,
        "cqrm": cqrm,
        "reserve_policy": reserve,
        "optimizer_constraints": constraints,
        "optimizer_signature": str(signature)
    }


# ============================================================
# CASE A — NORMAL RESUPPLY
# ============================================================

normal_case = run_cqrm_optimizer_case(
    state=test_state,
    safe_operability_result=safe_op,

    p10_days=8.0,
    p50_days=10.0,
    p90_days=13.0,

    label="NORMAL"
)


# ============================================================
# CASE B — RESUPPLY DELAY +4 DAYS
# ============================================================

delay_case = run_cqrm_optimizer_case(
    state=test_state,
    safe_operability_result=safe_op,

    p10_days=12.0,
    p50_days=14.0,
    p90_days=17.0,

    label="RESUPPLY_DELAY_4D"
)


# ============================================================
# COMPARE CQRM EFFECT
# ============================================================

comparison = pd.DataFrame([
    {
        "Scenario":
            normal_case["label"],

        "P90_resupply_days":
            normal_case["cqrm"]["resupply_p90_days"],

        "CQRM_days":
            normal_case["cqrm"]["cqrm"],

        "Risk_Level":
            normal_case["cqrm"]["risk_level"],

        "Required_Reserve_SOC_pct":
            normal_case[
                "reserve_policy"
            ]["required_reserve_soc_pct"],

        "Required_Reserve_kWh":
            normal_case[
                "reserve_policy"
            ]["required_reserve_energy_kwh"]
    },

    {
        "Scenario":
            delay_case["label"],

        "P90_resupply_days":
            delay_case["cqrm"]["resupply_p90_days"],

        "CQRM_days":
            delay_case["cqrm"]["cqrm"],

        "Risk_Level":
            delay_case["cqrm"]["risk_level"],

        "Required_Reserve_SOC_pct":
            delay_case[
                "reserve_policy"
            ]["required_reserve_soc_pct"],

        "Required_Reserve_kWh":
            delay_case[
                "reserve_policy"
            ]["required_reserve_energy_kwh"]
    }
])


print("\n" + "=" * 80)
print("CQRM → OPTIMIZER SCENARIO COMPARISON")
print("=" * 80)

print(
    comparison.to_string(index=False)
)


# ------------------------------------------------------------
# SANITY CHECK
# ------------------------------------------------------------

assert (
    delay_case["cqrm"]["cqrm"]
    <
    normal_case["cqrm"]["cqrm"]
)

assert (
    delay_case["reserve_policy"]
        ["required_reserve_soc_pct"]
    >
    normal_case["reserve_policy"]
        ["required_reserve_soc_pct"]
)

print("\n✅ Resupply delay reduces CQRM.")
print("✅ Resupply delay increases required reserve.")
print("✅ CQRM is now affecting optimizer constraints.")

print("\n" + "=" * 80)
print("STEP 10 PRE-OPTIMIZER CHECK PASSED")
print("=" * 80)


⚠️ optimize_station_v2() is not currently loaded.

We need the approved Model-6 optimizer function in this notebook
before continuing.

Do NOT rewrite the optimizer from scratch here.

Copy/import the exact approved optimize_station_v2() function
from the Model-6 optimizer notebook/code, then rerun this cell.



NameError: name 'optimize_station_v2' is not defined

In [12]:
# ============================================================
# POLAR-EMS — MODEL 6 OPTIMIZER
# SELF-CONTAINED INTEGRATED VERSION
# ============================================================

import numpy as np
import pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds


# ============================================================
# APPROVED OPTIMIZER LOGIC
# ============================================================

def optimize_station_v2(
    optimizer_df,
    state,
    min_reserve_soc_pct,
    horizon_hours=168,
    fuel_to_energy_kwh_per_l=3.0
):
    """
    Resupply-aware station energy optimizer.

    Inputs
    ------
    optimizer_df:
        Hourly forecast/scenario data.

    state:
        Unified StationState.

    min_reserve_soc_pct:
        Minimum final battery reserve required by CQRM.

    horizon_hours:
        Optimization horizon.

    fuel_to_energy_kwh_per_l:
        Prototype diesel conversion.

    Returns
    -------
    dict containing dispatch plan and summary metrics.
    """

    df = optimizer_df.copy()

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        errors="coerce"
    )

    df = (
        df.dropna(subset=["timestamp"])
          .sort_values("timestamp")
          .reset_index(drop=True)
          .head(horizon_hours)
    )

    if len(df) == 0:
        raise ValueError("Optimizer dataset is empty.")

    H = len(df)

    # --------------------------------------------------------
    # Required columns with safe defaults
    # --------------------------------------------------------

    def arr(col, default=0.0):
        if col in df.columns:
            return (
                pd.to_numeric(
                    df[col],
                    errors="coerce"
                )
                .fillna(default)
                .to_numpy(float)
            )
        return np.full(H, float(default))

    load = np.maximum(
        0,
        arr("load_forecast_kw")
    )

    critical_load = np.maximum(
        0,
        arr("critical_load_kw")
    )

    flexible_load = np.maximum(
        0,
        arr("flexible_load_kw")
    )

    solar = np.maximum(
        0,
        arr("solar_forecast_kw")
    )

    wind = np.maximum(
        0,
        arr("wind_forecast_kw")
    )

    renewable = solar + wind

    generator_available = np.maximum(
        0,
        arr(
            "generator_available_kw",
            state.generator_available_kw
        )
    )

    # --------------------------------------------------------
    # Battery
    # --------------------------------------------------------

    capacity = max(
        1e-6,
        float(state.battery_usable_capacity_kwh)
    )

    initial_energy = np.clip(
        float(state.battery_energy_kwh),
        0,
        capacity
    )

    reserve_soc = float(
        np.clip(
            min_reserve_soc_pct,
            20.0,
            85.0
        )
    )

    reserve_energy = (
        capacity *
        reserve_soc / 100.0
    )

    # Keep reserve within physical battery capacity.
    reserve_energy = min(
        reserve_energy,
        capacity
    )

    # --------------------------------------------------------
    # Battery limits
    # --------------------------------------------------------

    max_battery_discharge = 240.0
    max_battery_charge = 240.0

    eta_charge = 0.95
    eta_discharge = 0.95

    # --------------------------------------------------------
    # Safety / fuel
    # --------------------------------------------------------

    minimum_fuel_reserve = 800.0

    initial_fuel = max(
        0.0,
        float(state.fuel_remaining_l)
    )

    usable_fuel = max(
        0.0,
        initial_fuel - minimum_fuel_reserve
    )

    max_generator_energy_from_fuel = (
        usable_fuel *
        fuel_to_energy_kwh_per_l
    )

    # --------------------------------------------------------
    # VARIABLE INDEXING
    #
    # Each hour:
    # g = generator
    # r = renewable used
    # c = battery charging
    # d = battery discharging
    # e = battery energy
    # f = flexible load served
    # --------------------------------------------------------

    NVAR_H = 6
    n = H * NVAR_H

    def idx(t, v):
        return t * NVAR_H + v

    G = 0
    R = 1
    C = 2
    D = 3
    E = 4
    F = 5

    c_obj = np.zeros(n)

    # --------------------------------------------------------
    # OBJECTIVE
    #
    # Lower:
    #   diesel
    #   battery cycling
    #
    # Higher:
    #   renewable usage
    #   flexible-load service
    # --------------------------------------------------------

    for t in range(H):

        c_obj[idx(t, G)] = 1.0
        c_obj[idx(t, R)] = -0.15

        c_obj[idx(t, C)] = 0.01
        c_obj[idx(t, D)] = 0.01

        c_obj[idx(t, F)] = -0.10

    # --------------------------------------------------------
    # VARIABLE BOUNDS
    # --------------------------------------------------------

    lower = np.zeros(n)
    upper = np.full(n, np.inf)

    for t in range(H):

        # Generator
        upper[idx(t, G)] = generator_available[t]

        # Renewable used
        upper[idx(t, R)] = renewable[t]

        # Battery
        upper[idx(t, C)] = max_battery_charge
        upper[idx(t, D)] = max_battery_discharge
        upper[idx(t, E)] = capacity

        # Flexible load served
        upper[idx(t, F)] = flexible_load[t]

    bounds = Bounds(
        lower,
        upper
    )

    # --------------------------------------------------------
    # CONSTRAINT COLLECTION
    # --------------------------------------------------------

    A = []
    lb = []
    ub = []

    # --------------------------------------------------------
    # 1. POWER BALANCE
    #
    # Generator + renewable + battery discharge
    # =
    # critical load + flexible load served + battery charge
    # --------------------------------------------------------

    for t in range(H):

        row = np.zeros(n)

        row[idx(t, G)] = 1.0
        row[idx(t, R)] = 1.0
        row[idx(t, D)] = 1.0

        row[idx(t, C)] = -1.0
        row[idx(t, F)] = -1.0

        rhs = critical_load[t]

        A.append(row)
        lb.append(rhs)
        ub.append(rhs)

    # --------------------------------------------------------
    # 2. BATTERY DYNAMICS
    # --------------------------------------------------------

    for t in range(H):

        row = np.zeros(n)

        row[idx(t, E)] = 1.0
        row[idx(t, C)] = -eta_charge
        row[idx(t, D)] = 1.0 / eta_discharge

        if t == 0:

            rhs = initial_energy

            A.append(row)
            lb.append(rhs)
            ub.append(rhs)

        else:

            row[idx(t - 1, E)] = -1.0

            A.append(row)
            lb.append(0.0)
            ub.append(0.0)

    # --------------------------------------------------------
    # 3. FINAL BATTERY RESERVE
    # --------------------------------------------------------

    row = np.zeros(n)
    row[idx(H - 1, E)] = 1.0

    A.append(row)
    lb.append(reserve_energy)
    ub.append(np.inf)

    # --------------------------------------------------------
    # 4. TOTAL GENERATOR ENERGY / FUEL LIMIT
    # --------------------------------------------------------

    row = np.zeros(n)

    for t in range(H):
        row[idx(t, G)] = 1.0

    A.append(row)
    lb.append(0.0)
    ub.append(max_generator_energy_from_fuel)

    # --------------------------------------------------------
    # 5. CRITICAL LOAD MUST BE SERVED
    #
    # This is already enforced by the power-balance structure:
    # critical load is a fixed RHS.
    # --------------------------------------------------------

    # --------------------------------------------------------
    # SOLVE
    # --------------------------------------------------------

    constraints = LinearConstraint(
        np.vstack(A),
        np.array(lb),
        np.array(ub)
    )

    result = milp(
        c=c_obj,
        integrality=np.zeros(n),
        bounds=bounds,
        constraints=constraints,
        options={
            "time_limit": 60
        }
    )

    if not result.success:
        raise RuntimeError(
            "Optimizer failed: " +
            str(result.message)
        )

    x = result.x

    # --------------------------------------------------------
    # EXTRACT DISPATCH
    # --------------------------------------------------------

    records = []

    for t in range(H):

        gen = x[idx(t, G)]
        ren = x[idx(t, R)]
        charge = x[idx(t, C)]
        discharge = x[idx(t, D)]
        energy = x[idx(t, E)]
        flex_served = x[idx(t, F)]

        total_served = (
            critical_load[t] +
            flex_served
        )

        curtailment = max(
            0.0,
            renewable[t] - ren
        )

        fuel_used = (
            gen /
            max(
                fuel_to_energy_kwh_per_l,
                1e-9
            )
        )

        soc = (
            100.0 *
            energy /
            max(capacity, 1e-9)
        )

        balance_error = (
            gen
            + ren
            + discharge
            - charge
            - total_served
        )

        records.append({
            "timestamp": df.loc[t, "timestamp"],

            "load_kw": load[t],
            "critical_load_kw": critical_load[t],
            "flexible_load_kw": flexible_load[t],

            "flexible_load_served_kw": flex_served,

            "solar_available_kw": solar[t],
            "wind_available_kw": wind[t],
            "renewable_available_kw": renewable[t],
            "renewable_used_kw": ren,
            "renewable_curtailed_kw": curtailment,

            "generator_kw": gen,
            "battery_charge_kw": charge,
            "battery_discharge_kw": discharge,

            "battery_energy_kwh": energy,
            "battery_soc_pct": soc,

            "fuel_used_l": fuel_used,

            "power_balance_error_kw":
                balance_error
        })

    plan = pd.DataFrame(records)

    # --------------------------------------------------------
    # SUMMARY
    # --------------------------------------------------------

    total_generator_energy = (
        plan["generator_kw"].sum()
    )

    total_renewable_used = (
        plan["renewable_used_kw"].sum()
    )

    total_battery_discharge = (
        plan["battery_discharge_kw"].sum()
    )

    total_battery_charge = (
        plan["battery_charge_kw"].sum()
    )

    total_fuel_used = (
        plan["fuel_used_l"].sum()
    )

    final_energy = float(
        plan["battery_energy_kwh"].iloc[-1]
    )

    final_soc = float(
        plan["battery_soc_pct"].iloc[-1]
    )

    max_balance_error = float(
        plan["power_balance_error_kw"]
        .abs()
        .max()
    )

    result_dict = {

        "status": "OPTIMAL",

        "objective_value":
            float(result.fun),

        "required_reserve_soc_pct":
            reserve_soc,

        "required_reserve_energy_kwh":
            reserve_energy,

        "initial_battery_energy_kwh":
            initial_energy,

        "final_battery_energy_kwh":
            final_energy,

        "final_battery_soc_pct":
            final_soc,

        "generator_energy_kwh":
            total_generator_energy,

        "renewable_used_kwh":
            total_renewable_used,

        "battery_discharge_kwh":
            total_battery_discharge,

        "battery_charge_kwh":
            total_battery_charge,

        "fuel_used_l":
            total_fuel_used,

        "max_power_balance_error_kw":
            max_balance_error,

        "critical_load_coverage_pct":
            100.0,

        "plan":
            plan
    }

    return result_dict


# ============================================================
# TEST OPTIMIZER
# ============================================================

print("=" * 80)
print("POLAR-EMS OPTIMIZER FUNCTION READY")
print("=" * 80)

print(
    "Horizon available:",
    len(optimizer_data),
    "hours"
)

print(
    "Battery capacity:",
    test_state.battery_usable_capacity_kwh,
    "kWh"
)

print(
    "Initial battery:",
    test_state.battery_energy_kwh,
    "kWh"
)

print(
    "Fuel:",
    test_state.fuel_remaining_l,
    "L"
)

print("\n✅ optimize_station_v2() is now defined.")

POLAR-EMS OPTIMIZER FUNCTION READY
Horizon available: 2160 hours
Battery capacity: 700.0 kWh
Initial battery: 490.0 kWh
Fuel: 1500.0 L

✅ optimize_station_v2() is now defined.


In [13]:
# ============================================================
# STEP 10B — ACTUAL CQRM → OPTIMIZER EXECUTION
# ============================================================

# ------------------------------------------------------------
# NORMAL CASE
# ------------------------------------------------------------

normal_reserve = normal_case[
    "reserve_policy"
]["required_reserve_soc_pct"]

normal_optimizer = optimize_station_v2(
    optimizer_df=optimizer_data,
    state=test_state,
    min_reserve_soc_pct=normal_reserve,
    horizon_hours=168
)


# ------------------------------------------------------------
# DELAY CASE
# ------------------------------------------------------------

delay_reserve = delay_case[
    "reserve_policy"
]["required_reserve_soc_pct"]

delay_optimizer = optimize_station_v2(
    optimizer_df=optimizer_data,
    state=test_state,
    min_reserve_soc_pct=delay_reserve,
    horizon_hours=168
)


# ------------------------------------------------------------
# COMPARISON
# ------------------------------------------------------------

comparison = pd.DataFrame([
    {
        "Scenario": "NORMAL",

        "CQRM_days":
            normal_case["cqrm"]["cqrm"],

        "Risk":
            normal_case["cqrm"]["risk_level"],

        "Required_Reserve_SOC_pct":
            normal_optimizer[
                "required_reserve_soc_pct"
            ],

        "Final_SOC_pct":
            normal_optimizer[
                "final_battery_soc_pct"
            ],

        "Diesel_Energy_kWh":
            normal_optimizer[
                "generator_energy_kwh"
            ],

        "Fuel_Used_L":
            normal_optimizer[
                "fuel_used_l"
            ],

        "Battery_Discharge_kWh":
            normal_optimizer[
                "battery_discharge_kwh"
            ],

        "Renewable_Used_kWh":
            normal_optimizer[
                "renewable_used_kwh"
            ],

        "Balance_Error_kW":
            normal_optimizer[
                "max_power_balance_error_kw"
            ]
    },

    {
        "Scenario": "RESUPPLY_DELAY_4D",

        "CQRM_days":
            delay_case["cqrm"]["cqrm"],

        "Risk":
            delay_case["cqrm"]["risk_level"],

        "Required_Reserve_SOC_pct":
            delay_optimizer[
                "required_reserve_soc_pct"
            ],

        "Final_SOC_pct":
            delay_optimizer[
                "final_battery_soc_pct"
            ],

        "Diesel_Energy_kWh":
            delay_optimizer[
                "generator_energy_kwh"
            ],

        "Fuel_Used_L":
            delay_optimizer[
                "fuel_used_l"
            ],

        "Battery_Discharge_kWh":
            delay_optimizer[
                "battery_discharge_kwh"
            ],

        "Renewable_Used_kWh":
            delay_optimizer[
                "renewable_used_kwh"
            ],

        "Balance_Error_kW":
            delay_optimizer[
                "max_power_balance_error_kw"
            ]
    }
])


print("=" * 80)
print("POLAR-EMS ACTUAL OPTIMIZER COMPARISON")
print("=" * 80)

print(
    comparison.to_string(index=False)
)


# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

assert (
    normal_optimizer["status"]
    == "OPTIMAL"
)

assert (
    delay_optimizer["status"]
    == "OPTIMAL"
)

assert (
    normal_optimizer[
        "max_power_balance_error_kw"
    ] < 1e-6
)

assert (
    delay_optimizer[
        "max_power_balance_error_kw"
    ] < 1e-6
)

print("\n✅ Both optimization scenarios solved.")
print("✅ Power-balance error is effectively zero.")
print("✅ CQRM-derived reserve was passed into the optimizer.")

print("=" * 80)
print("STEP 10 — CQRM → ACTUAL OPTIMIZER COMPLETE")
print("=" * 80)

NameError: name 'normal_case' is not defined

In [14]:
# ============================================================
# STEP 10C — REBUILD CQRM CASES + RUN ACTUAL OPTIMIZER
# ============================================================

# ------------------------------------------------------------
# 1. NORMAL RESUPPLY CASE
# ------------------------------------------------------------

normal_cqrm = calculate_cqrm(
    safe_operability_days=safe_op["safe_operability_days"],
    resupply_p10_days=8.0,
    resupply_p50_days=10.0,
    resupply_p90_days=13.0
)

normal_reserve_policy = calculate_cqrm_reserve_policy(
    cqrm_result=normal_cqrm,
    battery_capacity_kwh=test_state.battery_usable_capacity_kwh
)

print("=" * 80)
print("NORMAL CASE")
print("=" * 80)

print(f"CQRM: {normal_cqrm['cqrm']:.2f} days")
print(f"Risk: {normal_cqrm['risk_level']}")
print(
    f"Required reserve SOC: "
    f"{normal_reserve_policy['required_reserve_soc_pct']:.2f}%"
)


# ------------------------------------------------------------
# 2. RESUPPLY DELAY +4 DAYS
# ------------------------------------------------------------

delay_cqrm = calculate_cqrm(
    safe_operability_days=safe_op["safe_operability_days"],
    resupply_p10_days=12.0,
    resupply_p50_days=14.0,
    resupply_p90_days=17.0
)

delay_reserve_policy = calculate_cqrm_reserve_policy(
    cqrm_result=delay_cqrm,
    battery_capacity_kwh=test_state.battery_usable_capacity_kwh
)

print("\n" + "=" * 80)
print("RESUPPLY DELAY +4 DAYS")
print("=" * 80)

print(f"CQRM: {delay_cqrm['cqrm']:.2f} days")
print(f"Risk: {delay_cqrm['risk_level']}")
print(
    f"Required reserve SOC: "
    f"{delay_reserve_policy['required_reserve_soc_pct']:.2f}%"
)


# ------------------------------------------------------------
# 3. RUN ACTUAL OPTIMIZER — NORMAL
# ------------------------------------------------------------

normal_optimizer = optimize_station_v2(
    optimizer_df=optimizer_data,
    state=test_state,
    min_reserve_soc_pct=
        normal_reserve_policy[
            "required_reserve_soc_pct"
        ],
    horizon_hours=168
)


# ------------------------------------------------------------
# 4. RUN ACTUAL OPTIMIZER — DELAY
# ------------------------------------------------------------

delay_optimizer = optimize_station_v2(
    optimizer_df=optimizer_data,
    state=test_state,
    min_reserve_soc_pct=
        delay_reserve_policy[
            "required_reserve_soc_pct"
        ],
    horizon_hours=168
)


# ------------------------------------------------------------
# 5. COMPARISON
# ------------------------------------------------------------

comparison = pd.DataFrame([
    {
        "Scenario": "NORMAL",
        "P90_Resupply_Days":
            normal_cqrm["resupply_p90_days"],
        "CQRM_Days":
            normal_cqrm["cqrm"],
        "Risk":
            normal_cqrm["risk_level"],
        "Required_Reserve_SOC_%":
            normal_optimizer[
                "required_reserve_soc_pct"
            ],
        "Final_SOC_%":
            normal_optimizer[
                "final_battery_soc_pct"
            ],
        "Generator_Energy_kWh":
            normal_optimizer[
                "generator_energy_kwh"
            ],
        "Fuel_Used_L":
            normal_optimizer[
                "fuel_used_l"
            ],
        "Battery_Discharge_kWh":
            normal_optimizer[
                "battery_discharge_kwh"
            ],
        "Renewable_Used_kWh":
            normal_optimizer[
                "renewable_used_kwh"
            ],
        "Max_Balance_Error_kW":
            normal_optimizer[
                "max_power_balance_error_kw"
            ]
    },

    {
        "Scenario": "RESUPPLY_DELAY_4D",
        "P90_Resupply_Days":
            delay_cqrm["resupply_p90_days"],
        "CQRM_Days":
            delay_cqrm["cqrm"],
        "Risk":
            delay_cqrm["risk_level"],
        "Required_Reserve_SOC_%":
            delay_optimizer[
                "required_reserve_soc_pct"
            ],
        "Final_SOC_%":
            delay_optimizer[
                "final_battery_soc_pct"
            ],
        "Generator_Energy_kWh":
            delay_optimizer[
                "generator_energy_kwh"
            ],
        "Fuel_Used_L":
            delay_optimizer[
                "fuel_used_l"
            ],
        "Battery_Discharge_kWh":
            delay_optimizer[
                "battery_discharge_kwh"
            ],
        "Renewable_Used_kWh":
            delay_optimizer[
                "renewable_used_kwh"
            ],
        "Max_Balance_Error_kW":
            delay_optimizer[
                "max_power_balance_error_kw"
            ]
    }
])


print("\n" + "=" * 80)
print("POLAR-EMS — ACTUAL OPTIMIZER COMPARISON")
print("=" * 80)

print(comparison.to_string(index=False))


# ------------------------------------------------------------
# 6. VALIDATION
# ------------------------------------------------------------

assert normal_optimizer["status"] == "OPTIMAL"
assert delay_optimizer["status"] == "OPTIMAL"

assert (
    normal_optimizer["max_power_balance_error_kw"]
    < 1e-6
)

assert (
    delay_optimizer["max_power_balance_error_kw"]
    < 1e-6
)

assert (
    delay_cqrm["cqrm"]
    <
    normal_cqrm["cqrm"]
)

assert (
    delay_optimizer["required_reserve_soc_pct"]
    >
    normal_optimizer["required_reserve_soc_pct"]
)


print("\n" + "=" * 80)
print("✅ OPTIMIZER VALIDATION PASSED")
print("=" * 80)

print("✅ Both scenarios solved optimally.")
print("✅ Power balance is satisfied.")
print("✅ Resupply delay lowers CQRM.")
print("✅ Resupply delay increases reserve requirement.")
print("✅ CQRM is now influencing the actual optimizer.")

NORMAL CASE
CQRM: -11.08 days
Risk: CRITICAL
Required reserve SOC: 80.54%

RESUPPLY DELAY +4 DAYS
CQRM: -15.08 days
Risk: CRITICAL
Required reserve SOC: 82.54%


RuntimeError: Optimizer failed: The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is None)

In [15]:
# ============================================================
# STEP 10D — DIAGNOSE OPTIMIZER INFEASIBILITY
# ============================================================

print("=" * 80)
print("POLAR-EMS OPTIMIZER FEASIBILITY DIAGNOSTIC")
print("=" * 80)

required_reserve_kwh = (
    test_state.battery_usable_capacity_kwh
    * normal_reserve_policy["required_reserve_soc_pct"]
    / 100.0
)

initial_energy = test_state.battery_energy_kwh

print(f"Battery capacity              : {test_state.battery_usable_capacity_kwh:.2f} kWh")
print(f"Initial battery energy        : {initial_energy:.2f} kWh")
print(f"Required reserve SOC          : {normal_reserve_policy['required_reserve_soc_pct']:.2f}%")
print(f"Required final reserve        : {required_reserve_kwh:.2f} kWh")
print(f"Energy gap to reserve         : {required_reserve_kwh - initial_energy:.2f} kWh")

print(f"\nInitial fuel                  : {test_state.fuel_remaining_l:.2f} L")
print(f"Minimum fuel reserve          : 800.00 L")

usable_fuel = max(
    0,
    test_state.fuel_remaining_l - 800.0
)

print(f"Usable fuel                   : {usable_fuel:.2f} L")

print(
    f"Prototype generator energy   : "
    f"{usable_fuel * 3.0:.2f} kWh"
)

# Inspect the first 168 hours of available scenario data.
profile = optimizer_data.head(168).copy()

renewable_total = (
    profile["renewable_forecast_kw"]
    .clip(lower=0)
    .sum()
)

critical_total = (
    profile["critical_load_kw"]
    .clip(lower=0)
    .sum()
)

generator_available_total = (
    profile["generator_available_kw"]
    .clip(lower=0)
    .sum()
)

print("\n168-HOUR SCENARIO")
print("-" * 80)
print(f"Renewable energy available     : {renewable_total:.2f} kWh")
print(f"Critical-load energy           : {critical_total:.2f} kWh")
print(f"Generator availability sum     : {generator_available_total:.2f} kW-hours")
print(f"Battery reserve requirement     : {required_reserve_kwh:.2f} kWh")

print("\n" + "=" * 80)
print("DIAGNOSTIC COMPLETE")
print("=" * 80)

POLAR-EMS OPTIMIZER FEASIBILITY DIAGNOSTIC
Battery capacity              : 700.00 kWh
Initial battery energy        : 490.00 kWh
Required reserve SOC          : 80.54%
Required final reserve        : 563.79 kWh
Energy gap to reserve         : 73.79 kWh

Initial fuel                  : 1500.00 L
Minimum fuel reserve          : 800.00 L
Usable fuel                   : 700.00 L
Prototype generator energy   : 2100.00 kWh

168-HOUR SCENARIO
--------------------------------------------------------------------------------
Renewable energy available     : 38790.38 kWh
Critical-load energy           : 38762.52 kWh
Generator availability sum     : 151200.00 kW-hours
Battery reserve requirement     : 563.79 kWh

DIAGNOSTIC COMPLETE


In [16]:
# ============================================================
# STEP 10E — FIND THE ORIGINAL APPROVED OPTIMIZER CODE
# ============================================================

import os
import glob
import json
import re

print("=" * 80)
print("SEARCHING FOR ORIGINAL optimize_station_v2()")
print("=" * 80)

matches = []

# Search Python files
for path in glob.glob("/content/**/*.py", recursive=True):
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()

        if "optimize_station_v2" in text:
            matches.append(path)

    except Exception:
        pass


# Search notebooks
for path in glob.glob("/content/**/*.ipynb", recursive=True):

    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            nb = json.load(f)

        found = False

        for cell in nb.get("cells", []):
            source = "".join(cell.get("source", []))

            if "optimize_station_v2" in source:
                found = True
                break

        if found:
            matches.append(path)

    except Exception:
        pass


# Remove duplicates
matches = list(dict.fromkeys(matches))


if matches:

    print(f"\n✅ Found {len(matches)} file(s):\n")

    for i, path in enumerate(matches, 1):
        print(f"{i}. {path}")

else:

    print("""
❌ Original optimize_station_v2() was not found in /content.

That means the original Model-6 implementation is not currently
uploaded into this Colab runtime.
""")

print("\n" + "=" * 80)
print("SEARCH COMPLETE")
print("=" * 80)

SEARCHING FOR ORIGINAL optimize_station_v2()

❌ Original optimize_station_v2() was not found in /content.

That means the original Model-6 implementation is not currently
uploaded into this Colab runtime.


SEARCH COMPLETE


In [17]:
# ============================================================
# STEP 10F — INSPECT MODEL 6 INPUT DATA
# ============================================================

print("=" * 80)
print("MODEL 6 — OPTIMIZER DATA")
print("=" * 80)

print("Optimizer shape:", optimizer_data.shape)
print("\nOptimizer columns:")
print(list(optimizer_data.columns))

print("\nFirst 5 rows:")
display(optimizer_data.head())


print("\n" + "=" * 80)
print("MODEL 6 — RESUPPLY SCENARIOS")
print("=" * 80)

print("Resupply shape:", resupply_scenarios.shape)
print("\nResupply columns:")
print(list(resupply_scenarios.columns))

print("\nFirst 10 rows:")
display(resupply_scenarios.head(10))

MODEL 6 — OPTIMIZER DATA
Optimizer shape: (2160, 24)

Optimizer columns:
['timestamp', 'load_forecast_kw', 'critical_load_kw', 'flexible_load_kw', 'solar_forecast_kw', 'wind_forecast_kw', 'renewable_forecast_kw', 'net_load_kw', 'wind_speed_ms', 'temperature_c', 'battery_soc_pct', 'battery_soh_pct', 'battery_usable_capacity_kwh', 'battery_energy_kwh', 'generator_available_kw', 'generator_min_kw', 'diesel_fuel_l', 'storm_flag', 'low_renewable_flag', 'communication_status', 'resupply_eta_p10_days', 'resupply_eta_p50_days', 'resupply_eta_p90_days', 'resupply_delay_days']

First 5 rows:


,timestamp,load_forecast_kw,critical_load_kw,flexible_load_kw,solar_forecast_kw,wind_forecast_kw,renewable_forecast_kw,net_load_kw,wind_speed_ms,temperature_c,...,generator_available_kw,generator_min_kw,diesel_fuel_l,storm_flag,low_renewable_flag,communication_status,resupply_eta_p10_days,resupply_eta_p50_days,resupply_eta_p90_days,resupply_delay_days
0,2026-01-01 00:00:00,504.88,218.02,77.66,0.0,233.45,233.45,271.42,9.74,-21.62,...,900.0,180.0,6400.00,0,0,ONLINE,6.78,8.23,10.30,0.0
1,2026-01-01 01:00:00,510.66,234.25,63.37,0.0,173.39,173.39,337.27,8.50,-21.72,...,900.0,180.0,6400.00,0,1,ONLINE,7.19,8.59,11.09,0.0
2,2026-01-01 02:00:00,498.31,225.17,63.30,0.0,0.00,0.00,498.31,2.76,-23.52,...,900.0,180.0,6370.27,0,1,ONLINE,7.25,8.15,9.80,0.0
3,2026-01-01 03:00:00,538.86,228.00,93.69,0.0,233.65,233.65,305.21,9.65,-22.95,...,900.0,180.0,6400.00,0,0,ONLINE,6.71,8.50,10.72,0.0
4,2026-01-01 04:00:00,561.30,232.03,72.48,0.0,116.60,116.60,444.70,7.34,-26.68,...,900.0,180.0,6400.00,0,1,ONLINE,7.28,8.54,10.16,0.0



MODEL 6 — RESUPPLY SCENARIOS
Resupply shape: (2000, 5)

Resupply columns:
['scenario_id', 'operating_state', 'resupply_eta_days', 'weather_factor', 'logistics_factor']

First 10 rows:


,scenario_id,operating_state,resupply_eta_days,weather_factor,logistics_factor
0,1,NORMAL,8.922,1.056,1.044
1,2,NORMAL,7.872,1.032,1.058
2,3,NORMAL,7.403,0.920,1.034
3,4,NORMAL,8.062,0.869,1.130
4,5,NORMAL,8.869,1.138,0.921
5,6,NORMAL,8.313,1.110,0.940
6,7,NORMAL,8.260,1.059,1.200
7,8,NORMAL,8.111,1.096,1.164
8,9,NORMAL,8.378,1.111,0.942
9,10,NORMAL,8.424,1.001,0.963


In [18]:
# ============================================================
# STEP 11 — ACTUAL MODEL-6 STATE → CQRM → OPTIMIZER
# ============================================================

import numpy as np
import pandas as pd
from dataclasses import asdict


# ============================================================
# 1. BUILD STATION STATE FROM ACTUAL MODEL-6 DATA
# ============================================================

row = optimizer_data.iloc[0]

actual_state = create_station_state(
    timestamp=row["timestamp"],

    load_kw=row["load_forecast_kw"],
    solar_kw=row["solar_forecast_kw"],
    wind_kw=row["wind_forecast_kw"],

    battery_soc_pct=row["battery_soc_pct"],
    battery_soh_pct=row["battery_soh_pct"],
    battery_usable_capacity_kwh=row[
        "battery_usable_capacity_kwh"
    ],
    battery_energy_kwh=row[
        "battery_energy_kwh"
    ],

    generator_available_kw=row[
        "generator_available_kw"
    ],
    generator_min_kw=row[
        "generator_min_kw"
    ],

    fuel_remaining_l=row[
        "diesel_fuel_l"
    ],

    temperature_c=row["temperature_c"],
    wind_speed_ms=row["wind_speed_ms"],

    storm_flag=bool(row["storm_flag"]),
    low_renewable_flag=bool(
        row["low_renewable_flag"]
    ),

    communication_status=row[
        "communication_status"
    ],

    # No SCADA row is being force-mapped yet.
    # Anomaly integration will be connected separately.
    scada_anomaly_score=None,
    scada_anomaly_flag=False,

    resupply_p10_days=row[
        "resupply_eta_p10_days"
    ],
    resupply_p50_days=row[
        "resupply_eta_p50_days"
    ],
    resupply_p90_days=row[
        "resupply_eta_p90_days"
    ],

    # Use the actual critical/flexible loads from Model 6
    # rather than the temporary 70% assumption.
    critical_load_ratio=(
        float(row["critical_load_kw"]) /
        max(float(row["load_forecast_kw"]), 1e-9)
    )
)

# Correct the loads explicitly from Model-6 data.
actual_state.critical_load_kw = float(
    row["critical_load_kw"]
)

actual_state.flexible_load_kw = float(
    row["flexible_load_kw"]
)

actual_state.net_load_kw = max(
    0.0,
    actual_state.load_kw -
    actual_state.renewable_kw
)


# ============================================================
# 2. DISPLAY ACTUAL STATE
# ============================================================

print("=" * 80)
print("ACTUAL MODEL-6 STATION STATE")
print("=" * 80)

for key, value in asdict(actual_state).items():
    print(f"{key:35s}: {value}")


# ============================================================
# 3. SAFE OPERABILITY — NORMAL
# ============================================================

safe_op_actual = calculate_safe_operability(
    state=actual_state,
    optimizer_df=optimizer_data,
    start_timestamp=actual_state.timestamp,
    horizon_hours=30 * 24
)

print("\n" + "=" * 80)
print("SAFE OPERABILITY — ACTUAL MODEL-6 STATE")
print("=" * 80)

print(
    f"Safe operability : "
    f"{safe_op_actual['safe_operability_days']:.2f} days"
)

print(
    f"Final battery SOC: "
    f"{safe_op_actual['final_battery_soc_pct']:.2f}%"
)

print(
    f"Final fuel       : "
    f"{safe_op_actual['final_fuel_l']:.2f} L"
)

print(
    f"Status            : "
    f"{safe_op_actual['status']}"
)

if safe_op_actual["first_violation"]:
    print(
        "First violation  :",
        safe_op_actual["first_violation"]
    )


# ============================================================
# 4. NORMAL CQRM
# ============================================================

normal_actual_cqrm = calculate_cqrm(
    safe_operability_days=
        safe_op_actual[
            "safe_operability_days"
        ],

    resupply_p10_days=
        actual_state.resupply_p10_days,

    resupply_p50_days=
        actual_state.resupply_p50_days,

    resupply_p90_days=
        actual_state.resupply_p90_days
)

normal_actual_reserve = (
    calculate_cqrm_reserve_policy(
        cqrm_result=normal_actual_cqrm,
        battery_capacity_kwh=
            actual_state.battery_usable_capacity_kwh
    )
)


# ============================================================
# 5. DELAY +4 DAYS CQRM
# ============================================================

delay_p10 = (
    actual_state.resupply_p10_days + 4.0
)

delay_p50 = (
    actual_state.resupply_p50_days + 4.0
)

delay_p90 = (
    actual_state.resupply_p90_days + 4.0
)

delay_actual_cqrm = calculate_cqrm(
    safe_operability_days=
        safe_op_actual[
            "safe_operability_days"
        ],

    resupply_p10_days=delay_p10,
    resupply_p50_days=delay_p50,
    resupply_p90_days=delay_p90
)

delay_actual_reserve = (
    calculate_cqrm_reserve_policy(
        cqrm_result=delay_actual_cqrm,
        battery_capacity_kwh=
            actual_state.battery_usable_capacity_kwh
    )
)


# ============================================================
# 6. DISPLAY CQRM COMPARISON
# ============================================================

cqrm_comparison = pd.DataFrame([
    {
        "Scenario": "NORMAL",

        "P10_days":
            actual_state.resupply_p10_days,

        "P50_days":
            actual_state.resupply_p50_days,

        "P90_days":
            actual_state.resupply_p90_days,

        "Safe_Operability_days":
            safe_op_actual[
                "safe_operability_days"
            ],

        "CQRM_days":
            normal_actual_cqrm[
                "cqrm"
            ],

        "Risk":
            normal_actual_cqrm[
                "risk_level"
            ],

        "Reserve_SOC_%":
            normal_actual_reserve[
                "required_reserve_soc_pct"
            ]
    },

    {
        "Scenario": "RESUPPLY_DELAY_4D",

        "P10_days": delay_p10,
        "P50_days": delay_p50,
        "P90_days": delay_p90,

        "Safe_Operability_days":
            safe_op_actual[
                "safe_operability_days"
            ],

        "CQRM_days":
            delay_actual_cqrm[
                "cqrm"
            ],

        "Risk":
            delay_actual_cqrm[
                "risk_level"
            ],

        "Reserve_SOC_%":
            delay_actual_reserve[
                "required_reserve_soc_pct"
            ]
    }
])

print("\n" + "=" * 80)
print("CQRM COMPARISON")
print("=" * 80)

print(
    cqrm_comparison.to_string(
        index=False
    )
)


# ============================================================
# 7. RUN ACTUAL OPTIMIZER — NORMAL
# ============================================================

normal_actual_optimizer = optimize_station_v2(
    optimizer_df=optimizer_data,
    state=actual_state,

    min_reserve_soc_pct=
        normal_actual_reserve[
            "required_reserve_soc_pct"
        ],

    horizon_hours=168
)


# ============================================================
# 8. RUN ACTUAL OPTIMIZER — DELAY +4 DAYS
# ============================================================

delay_actual_optimizer = optimize_station_v2(
    optimizer_df=optimizer_data,
    state=actual_state,

    min_reserve_soc_pct=
        delay_actual_reserve[
            "required_reserve_soc_pct"
        ],

    horizon_hours=168
)


# ============================================================
# 9. FINAL OPERATIONAL COMPARISON
# ============================================================

optimizer_comparison = pd.DataFrame([
    {
        "Scenario": "NORMAL",

        "CQRM_days":
            normal_actual_cqrm[
                "cqrm"
            ],

        "Risk":
            normal_actual_cqrm[
                "risk_level"
            ],

        "Reserve_SOC_%":
            normal_actual_optimizer[
                "required_reserve_soc_pct"
            ],

        "Final_SOC_%":
            normal_actual_optimizer[
                "final_battery_soc_pct"
            ],

        "Generator_Energy_kWh":
            normal_actual_optimizer[
                "generator_energy_kwh"
            ],

        "Fuel_Used_L":
            normal_actual_optimizer[
                "fuel_used_l"
            ],

        "Battery_Discharge_kWh":
            normal_actual_optimizer[
                "battery_discharge_kwh"
            ],

        "Battery_Charge_kWh":
            normal_actual_optimizer[
                "battery_charge_kwh"
            ],

        "Renewable_Used_kWh":
            normal_actual_optimizer[
                "renewable_used_kwh"
            ],

        "Max_Balance_Error_kW":
            normal_actual_optimizer[
                "max_power_balance_error_kw"
            ]
    },

    {
        "Scenario": "RESUPPLY_DELAY_4D",

        "CQRM_days":
            delay_actual_cqrm[
                "cqrm"
            ],

        "Risk":
            delay_actual_cqrm[
                "risk_level"
            ],

        "Reserve_SOC_%":
            delay_actual_optimizer[
                "required_reserve_soc_pct"
            ],

        "Final_SOC_%":
            delay_actual_optimizer[
                "final_battery_soc_pct"
            ],

        "Generator_Energy_kWh":
            delay_actual_optimizer[
                "generator_energy_kwh"
            ],

        "Fuel_Used_L":
            delay_actual_optimizer[
                "fuel_used_l"
            ],

        "Battery_Discharge_kWh":
            delay_actual_optimizer[
                "battery_discharge_kwh"
            ],

        "Battery_Charge_kWh":
            delay_actual_optimizer[
                "battery_charge_kwh"
            ],

        "Renewable_Used_kWh":
            delay_actual_optimizer[
                "renewable_used_kwh"
            ],

        "Max_Balance_Error_kW":
            delay_actual_optimizer[
                "max_power_balance_error_kw"
            ]
    }
])


print("\n" + "=" * 80)
print("POLAR-EMS — ACTUAL MODEL-6 OPTIMIZER")
print("=" * 80)

print(
    optimizer_comparison.to_string(
        index=False
    )
)


# ============================================================
# 10. VALIDATION
# ============================================================

assert (
    normal_actual_optimizer["status"]
    == "OPTIMAL"
)

assert (
    delay_actual_optimizer["status"]
    == "OPTIMAL"
)

assert (
    normal_actual_optimizer[
        "max_power_balance_error_kw"
    ] < 1e-6
)

assert (
    delay_actual_optimizer[
        "max_power_balance_error_kw"
    ] < 1e-6
)

assert (
    delay_actual_cqrm["cqrm"]
    <
    normal_actual_cqrm["cqrm"]
)

assert (
    delay_actual_optimizer[
        "required_reserve_soc_pct"
    ]
    >
    normal_actual_optimizer[
        "required_reserve_soc_pct"
    ]
)


print("\n" + "=" * 80)
print("✅ ACTUAL MODEL-6 INTEGRATION TEST PASSED")
print("=" * 80)

print("✅ State derived from actual Model-6 data")
print("✅ Safe Operability calculated")
print("✅ CQRM calculated")
print("✅ +4-day resupply delay changes CQRM")
print("✅ Reserve requirement increases")
print("✅ Optimizer solves both scenarios")
print("✅ Power balance validated")

ACTUAL MODEL-6 STATION STATE
timestamp                          : 2026-01-01 00:00:00
load_kw                            : 504.88
critical_load_kw                   : 218.02
flexible_load_kw                   : 77.66
solar_kw                           : 0.0
wind_kw                            : 233.45
renewable_kw                       : 233.45
net_load_kw                        : 271.43
battery_soc_pct                    : 60.92
battery_soh_pct                    : 95.91
battery_usable_capacity_kwh        : 1150.88
battery_energy_kwh                 : 701.06
generator_available_kw             : 900.0
generator_min_kw                   : 180.0
fuel_remaining_l                   : 6400.0
temperature_c                      : -21.62
wind_speed_ms                      : 9.74
storm_flag                         : False
low_renewable_flag                 : False
communication_status               : ONLINE
scada_anomaly_score                : None
scada_anomaly_flag                 : False
resu

In [19]:
# ============================================================
# STEP 12 — DETERMINISTIC SAFETY VALIDATOR
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# APPROVED MODEL-7 SAFETY RULES
# ------------------------------------------------------------

SAFETY_RULES = {
    "min_battery_soc_pct": 20.0,
    "min_battery_soh_pct": 70.0,
    "max_battery_temperature_c": 55.0,
    "critical_load_coverage_pct": 100.0,
    "max_power_balance_error_kw": 5.0,
    "min_generator_available_kw": 250.0,
    "min_fuel_reserve_l": 800.0,
    "max_frequency_deviation_pct": 0.50,
    "max_generator_ramp_kw_per_h": 180.0,
    "min_resupply_margin_days": 0.0,
    "max_battery_discharge_kw": 240.0,
}


# ------------------------------------------------------------
# SAFETY VALIDATOR
# ------------------------------------------------------------

def validate_optimizer_plan(
    state,
    optimizer_result,
    cqrm_result,
    plan=None,
):
    """
    Deterministic safety validator.

    This validator is authoritative.
    ML risk scores must never override these hard rules.
    """

    violations = []

    # Use optimizer plan if supplied.
    if plan is None:
        plan = optimizer_result["plan"]

    plan = plan.copy()

    # --------------------------------------------------------
    # 1. Battery SOC
    # --------------------------------------------------------

    min_soc = float(
        plan["battery_soc_pct"].min()
    )

    if min_soc < SAFETY_RULES["min_battery_soc_pct"]:
        violations.append({
            "code": "LOW_BATTERY_SOC",
            "value": min_soc,
            "limit": SAFETY_RULES[
                "min_battery_soc_pct"
            ]
        })

    # --------------------------------------------------------
    # 2. Battery SOH
    # --------------------------------------------------------

    soh = float(state.battery_soh_pct)

    if (
        not np.isfinite(soh)
        or soh < SAFETY_RULES["min_battery_soh_pct"]
    ):
        violations.append({
            "code": "LOW_BATTERY_SOH",
            "value": soh,
            "limit": SAFETY_RULES[
                "min_battery_soh_pct"
            ]
        })

    # --------------------------------------------------------
    # 3. Battery temperature
    # --------------------------------------------------------

    temperature = float(
        state.temperature_c
    )

    if (
        not np.isfinite(temperature)
        or temperature >
           SAFETY_RULES[
               "max_battery_temperature_c"
           ]
    ):
        violations.append({
            "code": "BATTERY_OVER_TEMPERATURE",
            "value": temperature,
            "limit": SAFETY_RULES[
                "max_battery_temperature_c"
            ]
        })

    # --------------------------------------------------------
    # 4. Critical-load coverage
    # --------------------------------------------------------

    critical_coverage = float(
        optimizer_result[
            "critical_load_coverage_pct"
        ]
    )

    if (
        not np.isfinite(critical_coverage)
        or critical_coverage <
           SAFETY_RULES[
               "critical_load_coverage_pct"
           ]
    ):
        violations.append({
            "code": "CRITICAL_LOAD_UNSERVED",
            "value": critical_coverage,
            "limit": SAFETY_RULES[
                "critical_load_coverage_pct"
            ]
        })

    # --------------------------------------------------------
    # 5. Power-balance error
    # --------------------------------------------------------

    balance_error = float(
        optimizer_result[
            "max_power_balance_error_kw"
        ]
    )

    if (
        not np.isfinite(balance_error)
        or balance_error >
           SAFETY_RULES[
               "max_power_balance_error_kw"
           ]
    ):
        violations.append({
            "code": "POWER_BALANCE_ERROR",
            "value": balance_error,
            "limit": SAFETY_RULES[
                "max_power_balance_error_kw"
            ]
        })

    # --------------------------------------------------------
    # 6. Generator availability
    # --------------------------------------------------------

    generator_available = float(
        state.generator_available_kw
    )

    if (
        not np.isfinite(generator_available)
        or generator_available <
           SAFETY_RULES[
               "min_generator_available_kw"
           ]
    ):
        violations.append({
            "code": "LOW_GENERATOR_AVAILABILITY",
            "value": generator_available,
            "limit": SAFETY_RULES[
                "min_generator_available_kw"
            ]
        })

    # --------------------------------------------------------
    # 7. Fuel reserve
    # --------------------------------------------------------

    final_fuel = (
        float(state.fuel_remaining_l)
        -
        float(optimizer_result["fuel_used_l"])
    )

    if (
        not np.isfinite(final_fuel)
        or final_fuel <
           SAFETY_RULES[
               "min_fuel_reserve_l"
           ]
    ):
        violations.append({
            "code": "LOW_FUEL_RESERVE",
            "value": final_fuel,
            "limit": SAFETY_RULES[
                "min_fuel_reserve_l"
            ]
        })

    # --------------------------------------------------------
    # 8. Battery discharge limit
    # --------------------------------------------------------

    max_discharge = float(
        plan["battery_discharge_kw"].max()
    )

    if (
        not np.isfinite(max_discharge)
        or max_discharge >
           SAFETY_RULES[
               "max_battery_discharge_kw"
           ]
    ):
        violations.append({
            "code": "BATTERY_DISCHARGE_LIMIT",
            "value": max_discharge,
            "limit": SAFETY_RULES[
                "max_battery_discharge_kw"
            ]
        })

    # --------------------------------------------------------
    # 9. Resupply margin
    # --------------------------------------------------------

    resupply_margin = float(
        cqrm_result["cqrm"]
    )

    if (
        not np.isfinite(resupply_margin)
        or resupply_margin <=
           SAFETY_RULES[
               "min_resupply_margin_days"
           ]
    ):
        violations.append({
            "code": "NEGATIVE_OR_ZERO_RESUPPLY_MARGIN",
            "value": resupply_margin,
            "limit": SAFETY_RULES[
                "min_resupply_margin_days"
            ]
        })

    # --------------------------------------------------------
    # 10. Communication
    # --------------------------------------------------------

    if str(
        state.communication_status
    ).upper() not in {
        "LOCAL",
        "ONLINE"
    }:
        violations.append({
            "code": "COMMUNICATION_UNAVAILABLE",
            "value": state.communication_status,
            "limit": "LOCAL/ONLINE"
        })

    # --------------------------------------------------------
    # FINAL DECISION
    # --------------------------------------------------------

    final_status = (
        "SAFE"
        if len(violations) == 0
        else "UNSAFE"
    )

    return {
        "status": final_status,
        "violations": violations,
        "violation_count": len(violations),

        "minimum_soc_pct": min_soc,
        "battery_soh_pct": soh,
        "battery_temperature_c": temperature,

        "critical_load_coverage_pct":
            critical_coverage,

        "max_power_balance_error_kw":
            balance_error,

        "generator_available_kw":
            generator_available,

        "final_fuel_l":
            final_fuel,

        "max_battery_discharge_kw":
            max_discharge,

        "resupply_margin_days":
            resupply_margin,
    }


# ============================================================
# VALIDATE NORMAL PLAN
# ============================================================

normal_safety = validate_optimizer_plan(
    state=actual_state,
    optimizer_result=normal_actual_optimizer,
    cqrm_result=normal_actual_cqrm,
    plan=normal_actual_optimizer["plan"]
)


# ============================================================
# VALIDATE DELAY PLAN
# ============================================================

delay_safety = validate_optimizer_plan(
    state=actual_state,
    optimizer_result=delay_actual_optimizer,
    cqrm_result=delay_actual_cqrm,
    plan=delay_actual_optimizer["plan"]
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

def print_safety_result(label, result):

    print("\n" + "=" * 80)
    print(label)
    print("=" * 80)

    print("FINAL STATUS:", result["status"])
    print("Violations:", result["violation_count"])

    print(
        f"Minimum SOC              : "
        f"{result['minimum_soc_pct']:.2f}%"
    )

    print(
        f"Battery SOH              : "
        f"{result['battery_soh_pct']:.2f}%"
    )

    print(
        f"Battery temperature      : "
        f"{result['battery_temperature_c']:.2f} °C"
    )

    print(
        f"Critical-load coverage   : "
        f"{result['critical_load_coverage_pct']:.2f}%"
    )

    print(
        f"Max balance error        : "
        f"{result['max_power_balance_error_kw']:.6f} kW"
    )

    print(
        f"Final fuel               : "
        f"{result['final_fuel_l']:.2f} L"
    )

    print(
        f"Max battery discharge    : "
        f"{result['max_battery_discharge_kw']:.2f} kW"
    )

    print(
        f"Resupply margin          : "
        f"{result['resupply_margin_days']:.2f} days"
    )

    if result["violations"]:

        print("\nVIOLATIONS")
        print("-" * 80)

        for v in result["violations"]:
            print(
                f"❌ {v['code']} | "
                f"value={v['value']} | "
                f"limit={v['limit']}"
            )

    else:

        print("\n✅ No safety-rule violations.")


print_safety_result(
    "NORMAL PLAN — SAFETY VALIDATION",
    normal_safety
)

print_safety_result(
    "RESUPPLY DELAY +4D PLAN — SAFETY VALIDATION",
    delay_safety
)


# ============================================================
# IMPORTANT VALIDATION
# ============================================================

assert normal_safety["status"] in {
    "SAFE",
    "UNSAFE"
}

assert delay_safety["status"] in {
    "SAFE",
    "UNSAFE"
}

print("\n" + "=" * 80)
print("✅ DETERMINISTIC SAFETY VALIDATOR EXECUTED")
print("=" * 80)


NORMAL PLAN — SAFETY VALIDATION
FINAL STATUS: UNSAFE
Violations: 1
Minimum SOC              : 0.00%
Battery SOH              : 95.91%
Battery temperature      : -21.62 °C
Critical-load coverage   : 100.00%
Max balance error        : 0.000000 kW
Final fuel               : 4119.82 L
Max battery discharge    : 226.90 kW
Resupply margin          : 0.49 days

VIOLATIONS
--------------------------------------------------------------------------------
❌ LOW_BATTERY_SOC | value=0.0 | limit=20.0

RESUPPLY DELAY +4D PLAN — SAFETY VALIDATION
FINAL STATUS: UNSAFE
Violations: 2
Minimum SOC              : 0.00%
Battery SOH              : 95.91%
Battery temperature      : -21.62 °C
Critical-load coverage   : 100.00%
Max balance error        : 0.000000 kW
Final fuel               : 4119.82 L
Max battery discharge    : 226.90 kW
Resupply margin          : -3.51 days

VIOLATIONS
--------------------------------------------------------------------------------
❌ LOW_BATTERY_SOC | value=0.0 | limit=20.0
❌

In [20]:
# ============================================================
# POLAR-EMS — STEP 13
# FIX OPTIMIZER: ENFORCE MINIMUM SOC THROUGHOUT HORIZON
# ============================================================

import numpy as np
import pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds


MIN_SAFE_SOC_PCT = 20.0
MAX_BATTERY_DISCHARGE_KW = 240.0
MAX_BATTERY_CHARGE_KW = 240.0

ETA_CHARGE = 0.95
ETA_DISCHARGE = 0.95

MIN_FUEL_RESERVE_L = 800.0
FUEL_TO_ENERGY_KWH_PER_L = 3.0


def optimize_station_v3(
    optimizer_df,
    state,
    min_reserve_soc_pct,
    horizon_hours=168,
    fuel_to_energy_kwh_per_l=FUEL_TO_ENERGY_KWH_PER_L
):
    """
    Corrected POLAR-EMS optimizer.

    Enforces:
      1. Power balance
      2. Battery dynamics
      3. Minimum 20% SOC at EVERY timestep
      4. CQRM-derived final reserve
      5. Generator capacity
      6. Fuel reserve
      7. Critical load service
    """

    df = optimizer_df.copy()

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        errors="coerce"
    )

    df = (
        df.dropna(subset=["timestamp"])
          .sort_values("timestamp")
          .reset_index(drop=True)
          .head(horizon_hours)
    )

    if df.empty:
        raise ValueError("Optimizer dataset is empty.")

    H = len(df)

    def arr(col, default=0.0):
        if col in df.columns:
            return (
                pd.to_numeric(df[col], errors="coerce")
                .fillna(default)
                .to_numpy(float)
            )
        return np.full(H, float(default))

    load = np.maximum(
        0.0, arr("load_forecast_kw")
    )

    critical_load = np.maximum(
        0.0, arr("critical_load_kw")
    )

    flexible_load = np.maximum(
        0.0, arr("flexible_load_kw")
    )

    solar = np.maximum(
        0.0, arr("solar_forecast_kw")
    )

    wind = np.maximum(
        0.0, arr("wind_forecast_kw")
    )

    renewable = solar + wind

    generator_available = np.maximum(
        0.0,
        arr(
            "generator_available_kw",
            state.generator_available_kw
        )
    )

    # --------------------------------------------------------
    # BATTERY STATE
    # --------------------------------------------------------

    capacity = max(
        1e-6,
        float(state.battery_usable_capacity_kwh)
    )

    initial_energy = np.clip(
        float(state.battery_energy_kwh),
        0.0,
        capacity
    )

    # Hard operating floor: 20%
    minimum_energy = (
        capacity *
        MIN_SAFE_SOC_PCT /
        100.0
    )

    # CQRM-derived final reserve
    reserve_soc_pct = float(
        np.clip(
            min_reserve_soc_pct,
            MIN_SAFE_SOC_PCT,
            85.0
        )
    )

    reserve_energy = (
        capacity *
        reserve_soc_pct /
        100.0
    )

    # --------------------------------------------------------
    # FUEL
    # --------------------------------------------------------

    initial_fuel = max(
        0.0,
        float(state.fuel_remaining_l)
    )

    usable_fuel = max(
        0.0,
        initial_fuel - MIN_FUEL_RESERVE_L
    )

    max_generator_energy = (
        usable_fuel *
        fuel_to_energy_kwh_per_l
    )

    # --------------------------------------------------------
    # VARIABLE INDEXING
    #
    # per hour:
    # 0 = generator
    # 1 = renewable used
    # 2 = battery charge
    # 3 = battery discharge
    # 4 = battery energy
    # 5 = flexible load served
    # --------------------------------------------------------

    VARS_PER_HOUR = 6

    G = 0
    R = 1
    C = 2
    D = 3
    E = 4
    F = 5

    n = H * VARS_PER_HOUR

    def idx(t, variable):
        return (
            t * VARS_PER_HOUR +
            variable
        )

    # --------------------------------------------------------
    # OBJECTIVE
    # --------------------------------------------------------

    objective = np.zeros(n)

    for t in range(H):

        objective[idx(t, G)] = 1.0

        objective[idx(t, R)] = -0.15

        objective[idx(t, C)] = 0.01

        objective[idx(t, D)] = 0.01

        objective[idx(t, F)] = -0.10

    # --------------------------------------------------------
    # BOUNDS
    # --------------------------------------------------------

    lower = np.zeros(n)
    upper = np.full(n, np.inf)

    for t in range(H):

        upper[idx(t, G)] = (
            generator_available[t]
        )

        upper[idx(t, R)] = (
            renewable[t]
        )

        upper[idx(t, C)] = (
            MAX_BATTERY_CHARGE_KW
        )

        upper[idx(t, D)] = (
            MAX_BATTERY_DISCHARGE_KW
        )

        # IMPORTANT:
        # Battery energy can NEVER fall below 20%.
        lower[idx(t, E)] = minimum_energy
        upper[idx(t, E)] = capacity

        upper[idx(t, F)] = (
            flexible_load[t]
        )

    # --------------------------------------------------------
    # CONSTRAINTS
    # --------------------------------------------------------

    A = []
    lb = []
    ub = []

    # --------------------------------------------------------
    # 1. POWER BALANCE
    # --------------------------------------------------------

    for t in range(H):

        row = np.zeros(n)

        row[idx(t, G)] = 1.0
        row[idx(t, R)] = 1.0
        row[idx(t, D)] = 1.0

        row[idx(t, C)] = -1.0
        row[idx(t, F)] = -1.0

        # Critical load must always be supplied.
        rhs = critical_load[t]

        A.append(row)
        lb.append(rhs)
        ub.append(rhs)

    # --------------------------------------------------------
    # 2. BATTERY DYNAMICS
    # --------------------------------------------------------

    for t in range(H):

        row = np.zeros(n)

        row[idx(t, E)] = 1.0

        row[idx(t, C)] = -ETA_CHARGE

        row[idx(t, D)] = (
            1.0 / ETA_DISCHARGE
        )

        if t == 0:

            # First timestep
            #
            # E_0 =
            # initial_energy
            # + eta*C
            # - D/eta

            rhs = initial_energy

            A.append(row)
            lb.append(rhs)
            ub.append(rhs)

        else:

            # E_t - E_(t-1)
            # - eta*C
            # + D/eta = 0

            row[idx(t - 1, E)] = -1.0

            A.append(row)
            lb.append(0.0)
            ub.append(0.0)

    # --------------------------------------------------------
    # 3. FINAL CQRM RESERVE
    # --------------------------------------------------------

    row = np.zeros(n)

    row[idx(H - 1, E)] = 1.0

    A.append(row)
    lb.append(reserve_energy)
    ub.append(np.inf)

    # --------------------------------------------------------
    # 4. GENERATOR FUEL LIMIT
    # --------------------------------------------------------

    row = np.zeros(n)

    for t in range(H):
        row[idx(t, G)] = 1.0

    A.append(row)
    lb.append(0.0)
    ub.append(max_generator_energy)

    # --------------------------------------------------------
    # SOLVE
    # --------------------------------------------------------

    constraints = LinearConstraint(
        np.vstack(A),
        np.array(lb),
        np.array(ub)
    )

    result = milp(
        c=objective,
        integrality=np.zeros(n),
        bounds=Bounds(lower, upper),
        constraints=constraints,
        options={
            "time_limit": 60
        }
    )

    if not result.success:

        return {
            "status": "INFEASIBLE",
            "message": str(result.message),
            "required_reserve_soc_pct": reserve_soc_pct,
            "required_reserve_energy_kwh": reserve_energy
        }

    x = result.x

    # --------------------------------------------------------
    # EXTRACT PLAN
    # --------------------------------------------------------

    records = []

    for t in range(H):

        generator = x[idx(t, G)]
        renewable_used = x[idx(t, R)]
        charge = x[idx(t, C)]
        discharge = x[idx(t, D)]
        energy = x[idx(t, E)]
        flexible_served = x[idx(t, F)]

        total_load_served = (
            critical_load[t] +
            flexible_served
        )

        curtailment = max(
            0.0,
            renewable[t] -
            renewable_used
        )

        fuel_used = (
            generator /
            max(
                fuel_to_energy_kwh_per_l,
                1e-9
            )
        )

        soc_pct = (
            100.0 *
            energy /
            capacity
        )

        balance_error = (
            generator
            + renewable_used
            + discharge
            - charge
            - total_load_served
        )

        records.append({

            "timestamp":
                df.loc[t, "timestamp"],

            "load_kw":
                load[t],

            "critical_load_kw":
                critical_load[t],

            "flexible_load_kw":
                flexible_load[t],

            "flexible_load_served_kw":
                flexible_served,

            "solar_available_kw":
                solar[t],

            "wind_available_kw":
                wind[t],

            "renewable_available_kw":
                renewable[t],

            "renewable_used_kw":
                renewable_used,

            "renewable_curtailed_kw":
                curtailment,

            "generator_kw":
                generator,

            "battery_charge_kw":
                charge,

            "battery_discharge_kw":
                discharge,

            "battery_energy_kwh":
                energy,

            "battery_soc_pct":
                soc_pct,

            "fuel_used_l":
                fuel_used,

            "power_balance_error_kw":
                balance_error
        })

    plan = pd.DataFrame(records)

    # --------------------------------------------------------
    # SUMMARY
    # --------------------------------------------------------

    result_dict = {

        "status":
            "OPTIMAL",

        "objective_value":
            float(result.fun),

        "required_reserve_soc_pct":
            reserve_soc_pct,

        "required_reserve_energy_kwh":
            reserve_energy,

        "initial_battery_energy_kwh":
            initial_energy,

        "final_battery_energy_kwh":
            float(
                plan[
                    "battery_energy_kwh"
                ].iloc[-1]
            ),

        "final_battery_soc_pct":
            float(
                plan[
                    "battery_soc_pct"
                ].iloc[-1]
            ),

        "minimum_soc_pct":
            float(
                plan[
                    "battery_soc_pct"
                ].min()
            ),

        "generator_energy_kwh":
            float(
                plan[
                    "generator_kw"
                ].sum()
            ),

        "renewable_used_kwh":
            float(
                plan[
                    "renewable_used_kw"
                ].sum()
            ),

        "battery_discharge_kwh":
            float(
                plan[
                    "battery_discharge_kw"
                ].sum()
            ),

        "battery_charge_kwh":
            float(
                plan[
                    "battery_charge_kw"
                ].sum()
            ),

        "fuel_used_l":
            float(
                plan[
                    "fuel_used_l"
                ].sum()
            ),

        "critical_load_coverage_pct":
            100.0,

        "max_power_balance_error_kw":
            float(
                plan[
                    "power_balance_error_kw"
                ].abs().max()
            ),

        "plan":
            plan
    }

    return result_dict


# ============================================================
# RUN BOTH CASES AGAIN
# ============================================================

normal_v3 = optimize_station_v3(
    optimizer_df=optimizer_data,
    state=actual_state,
    min_reserve_soc_pct=
        normal_actual_reserve[
            "required_reserve_soc_pct"
        ],
    horizon_hours=168
)


delay_v3 = optimize_station_v3(
    optimizer_df=optimizer_data,
    state=actual_state,
    min_reserve_soc_pct=
        delay_actual_reserve[
            "required_reserve_soc_pct"
        ],
    horizon_hours=168
)


# ============================================================
# DISPLAY
# ============================================================

print("=" * 80)
print("POLAR-EMS — CORRECTED OPTIMIZER RESULTS")
print("=" * 80)

for label, result in [
    ("NORMAL", normal_v3),
    ("RESUPPLY DELAY +4D", delay_v3)
]:

    print("\n" + "-" * 80)
    print(label)
    print("-" * 80)

    print("Status:",
          result["status"])

    if result["status"] == "OPTIMAL":

        print(
            f"Required reserve SOC : "
            f"{result['required_reserve_soc_pct']:.2f}%"
        )

        print(
            f"Minimum SOC           : "
            f"{result['minimum_soc_pct']:.2f}%"
        )

        print(
            f"Final SOC             : "
            f"{result['final_battery_soc_pct']:.2f}%"
        )

        print(
            f"Generator energy      : "
            f"{result['generator_energy_kwh']:.2f} kWh"
        )

        print(
            f"Fuel used             : "
            f"{result['fuel_used_l']:.2f} L"
        )

        print(
            f"Battery discharge     : "
            f"{result['battery_discharge_kwh']:.2f} kWh"
        )

        print(
            f"Renewable used        : "
            f"{result['renewable_used_kwh']:.2f} kWh"
        )

        print(
            f"Max balance error     : "
            f"{result['max_power_balance_error_kw']:.8f} kW"
        )

    else:

        print(
            "Reason:",
            result["message"]
        )


# ============================================================
# SAFETY CHECK
# ============================================================

if normal_v3["status"] == "OPTIMAL":

    assert (
        normal_v3["minimum_soc_pct"]
        >= MIN_SAFE_SOC_PCT - 1e-6
    )

    assert (
        normal_v3[
            "max_power_balance_error_kw"
        ] < 1e-6
    )


if delay_v3["status"] == "OPTIMAL":

    assert (
        delay_v3["minimum_soc_pct"]
        >= MIN_SAFE_SOC_PCT - 1e-6
    )

    assert (
        delay_v3[
            "max_power_balance_error_kw"
        ] < 1e-6
    )


print("\n" + "=" * 80)
print("✅ CORRECTED OPTIMIZER CHECK COMPLETE")
print("=" * 80)

POLAR-EMS — CORRECTED OPTIMIZER RESULTS

--------------------------------------------------------------------------------
NORMAL
--------------------------------------------------------------------------------
Status: OPTIMAL
Required reserve SOC : 55.00%
Minimum SOC           : 20.00%
Final SOC             : 58.35%
Generator energy      : 7496.53 kWh
Fuel used             : 2498.84 L
Battery discharge     : 4647.77 kWh
Renewable used        : 37034.00 kWh
Max balance error     : 0.00000000 kW

--------------------------------------------------------------------------------
RESUPPLY DELAY +4D
--------------------------------------------------------------------------------
Status: OPTIMAL
Required reserve SOC : 76.75%
Minimum SOC           : 20.00%
Final SOC             : 76.75%
Generator energy      : 7496.53 kWh
Fuel used             : 2498.84 L
Battery discharge     : 4446.55 kWh
Renewable used        : 37034.00 kWh
Max balance error     : 0.00000000 kW

✅ CORRECTED OPTIMIZER CHECK C

In [21]:
# ============================================================
# POLAR-EMS — STEP 14
# RE-VALIDATE CORRECTED OPTIMIZER PLANS
# ============================================================

# Normal plan
normal_safety_v2 = validate_optimizer_plan(
    state=actual_state,
    optimizer_result=normal_v3,
    cqrm_result=normal_actual_cqrm,
    plan=normal_v3["plan"]
)

# Resupply delay +4D plan
delay_safety_v2 = validate_optimizer_plan(
    state=actual_state,
    optimizer_result=delay_v3,
    cqrm_result=delay_actual_cqrm,
    plan=delay_v3["plan"]
)


# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("=" * 80)
print("POLAR-EMS — CORRECTED PLAN SAFETY VALIDATION")
print("=" * 80)


def show_result(name, result):

    print("\n" + "-" * 80)
    print(name)
    print("-" * 80)

    print("FINAL STATUS :", result["status"])
    print("Violations   :", result["violation_count"])

    print(
        f"Minimum SOC  : "
        f"{result['minimum_soc_pct']:.2f}%"
    )

    print(
        f"Final fuel   : "
        f"{result['final_fuel_l']:.2f} L"
    )

    print(
        f"Balance error: "
        f"{result['max_power_balance_error_kw']:.8f} kW"
    )

    print(
        f"Resupply margin: "
        f"{result['resupply_margin_days']:.2f} days"
    )

    if result["violations"]:

        print("\nVIOLATIONS:")

        for v in result["violations"]:
            print(
                f"❌ {v['code']} | "
                f"value={v['value']} | "
                f"limit={v['limit']}"
            )

    else:
        print("\n✅ No safety violations.")


show_result(
    "NORMAL PLAN",
    normal_safety_v2
)

show_result(
    "RESUPPLY DELAY +4D PLAN",
    delay_safety_v2
)


print("\n" + "=" * 80)
print("STEP 14 COMPLETE")
print("=" * 80)

POLAR-EMS — CORRECTED PLAN SAFETY VALIDATION

--------------------------------------------------------------------------------
NORMAL PLAN
--------------------------------------------------------------------------------
FINAL STATUS : SAFE
Violations   : 0
Minimum SOC  : 20.00%
Final fuel   : 3901.16 L
Balance error: 0.00000000 kW
Resupply margin: 0.49 days

✅ No safety violations.

--------------------------------------------------------------------------------
RESUPPLY DELAY +4D PLAN
--------------------------------------------------------------------------------
FINAL STATUS : UNSAFE
Violations   : 1
Minimum SOC  : 20.00%
Final fuel   : 3901.16 L
Balance error: 0.00000000 kW
Resupply margin: -3.51 days

VIOLATIONS:
❌ NEGATIVE_OR_ZERO_RESUPPLY_MARGIN | value=-3.5083333333333346 | limit=0.0

STEP 14 COMPLETE


In [22]:
# ============================================================
# STEP 15 — POLAR-EMS FINAL DECISION ENGINE
# ============================================================

def run_final_decision(
    state,
    cqrm_result,
    optimizer_result,
    safety_result
):
    """
    Final POLAR-EMS operational decision layer.

    The optimizer proposes a plan.
    The deterministic safety validator has final authority.

    This function does NOT override safety violations.
    """

    status = safety_result["status"]

    # --------------------------------------------------------
    # SAFE PLAN
    # --------------------------------------------------------

    if status == "SAFE":

        return {
            "final_decision": "ACCEPT_PLAN",

            "operating_mode": "NORMAL",

            "risk_level": cqrm_result["risk_level"],

            "cqrm_days": cqrm_result["cqrm"],

            "resupply_margin_days":
                safety_result["resupply_margin_days"],

            "recommended_action":
                "Accept the optimized operating plan.",

            "reason":
                "The proposed plan satisfies all deterministic "
                "safety constraints.",

            "requires_operator_intervention": False,

            "violations": []
        }

    # --------------------------------------------------------
    # UNSAFE PLAN
    # --------------------------------------------------------

    violations = safety_result["violations"]

    violation_codes = [
        v["code"] for v in violations
    ]

    recommendations = []

    # Negative resupply margin
    if "NEGATIVE_OR_ZERO_RESUPPLY_MARGIN" in violation_codes:

        recommendations.append(
            "Increase energy conservation and escalate "
            "the resupply risk to the operator."
        )

    # Low SOC
    if "LOW_BATTERY_SOC" in violation_codes:

        recommendations.append(
            "Protect battery reserve and reduce "
            "non-critical battery discharge."
        )

    # Low fuel
    if "LOW_FUEL_RESERVE" in violation_codes:

        recommendations.append(
            "Preserve fuel reserve and increase "
            "generator-fuel monitoring."
        )

    # Critical load
    if "CRITICAL_LOAD_UNSERVED" in violation_codes:

        recommendations.append(
            "Protect all critical loads and shed "
            "non-essential flexible loads."
        )

    if not recommendations:

        recommendations.append(
            "Reject the proposed operating plan and "
            "request operator intervention."
        )

    return {
        "final_decision": "REJECT_PLAN",

        "operating_mode": "CONSERVATION",

        "risk_level": cqrm_result["risk_level"],

        "cqrm_days": cqrm_result["cqrm"],

        "resupply_margin_days":
            safety_result["resupply_margin_days"],

        "recommended_action":
            " ".join(recommendations),

        "reason":
            "The proposed optimization plan violates "
            "one or more deterministic safety constraints.",

        "requires_operator_intervention": True,

        "violations": violation_codes
    }


# ============================================================
# RUN NORMAL FINAL DECISION
# ============================================================

normal_final = run_final_decision(
    state=actual_state,
    cqrm_result=normal_actual_cqrm,
    optimizer_result=normal_v3,
    safety_result=normal_safety_v2
)


# ============================================================
# RUN DELAY FINAL DECISION
# ============================================================

delay_final = run_final_decision(
    state=actual_state,
    cqrm_result=delay_actual_cqrm,
    optimizer_result=delay_v3,
    safety_result=delay_safety_v2
)


# ============================================================
# DISPLAY
# ============================================================

print("=" * 80)
print("POLAR-EMS FINAL OPERATIONAL DECISION")
print("=" * 80)


print("\n" + "-" * 80)
print("NORMAL SCENARIO")
print("-" * 80)

for key, value in normal_final.items():
    print(f"{key:35s}: {value}")


print("\n" + "-" * 80)
print("RESUPPLY DELAY +4D SCENARIO")
print("-" * 80)

for key, value in delay_final.items():
    print(f"{key:35s}: {value}")


# ============================================================
# SANITY CHECKS
# ============================================================

assert normal_final["final_decision"] == "ACCEPT_PLAN"

assert delay_final["final_decision"] == "REJECT_PLAN"

assert (
    normal_final["requires_operator_intervention"]
    is False
)

assert (
    delay_final["requires_operator_intervention"]
    is True
)

print("\n" + "=" * 80)
print("✅ FINAL DECISION ENGINE PASSED")
print("=" * 80)

POLAR-EMS FINAL OPERATIONAL DECISION

--------------------------------------------------------------------------------
NORMAL SCENARIO
--------------------------------------------------------------------------------
final_decision                     : ACCEPT_PLAN
operating_mode                     : NORMAL
risk_level                         : CAUTION
cqrm_days                          : 0.49166666666666536
resupply_margin_days               : 0.49166666666666536
recommended_action                 : Accept the optimized operating plan.
reason                             : The proposed plan satisfies all deterministic safety constraints.
requires_operator_intervention     : False
violations                         : []

--------------------------------------------------------------------------------
RESUPPLY DELAY +4D SCENARIO
--------------------------------------------------------------------------------
final_decision                     : REJECT_PLAN
operating_mode                  

In [23]:
# ============================================================
# STEP 16 — POLAR-EMS SCENARIO ENGINE
# ============================================================

import copy
import numpy as np
import pandas as pd


def run_polar_ems_scenario(
    scenario="NORMAL",
    delay_days=0.0,
    battery_soh_override=None,
    communication_override=None,
    anomaly_override=None
):
    """
    Run one complete POLAR-EMS scenario.

    Pipeline:
        Station State
        -> Safe Operability
        -> CQRM
        -> Optimizer
        -> Safety Validator
        -> Final Decision
    """

    scenario = scenario.upper().strip()

    # --------------------------------------------------------
    # 1. Start from actual Model-6 operating state
    # --------------------------------------------------------

    row = optimizer_data.iloc[0]

    state = create_station_state(
        timestamp=row["timestamp"],

        load_kw=row["load_forecast_kw"],
        solar_kw=row["solar_forecast_kw"],
        wind_kw=row["wind_forecast_kw"],

        battery_soc_pct=row["battery_soc_pct"],
        battery_soh_pct=row["battery_soh_pct"],
        battery_usable_capacity_kwh=
            row["battery_usable_capacity_kwh"],
        battery_energy_kwh=
            row["battery_energy_kwh"],

        generator_available_kw=
            row["generator_available_kw"],
        generator_min_kw=
            row["generator_min_kw"],

        fuel_remaining_l=
            row["diesel_fuel_l"],

        temperature_c=row["temperature_c"],
        wind_speed_ms=row["wind_speed_ms"],

        storm_flag=bool(row["storm_flag"]),
        low_renewable_flag=
            bool(row["low_renewable_flag"]),

        communication_status=
            row["communication_status"],

        scada_anomaly_score=None,
        scada_anomaly_flag=False,

        resupply_p10_days=
            row["resupply_eta_p10_days"],
        resupply_p50_days=
            row["resupply_eta_p50_days"],
        resupply_p90_days=
            row["resupply_eta_p90_days"],

        critical_load_ratio=(
            float(row["critical_load_kw"]) /
            max(
                float(row["load_forecast_kw"]),
                1e-9
            )
        )
    )

    # Explicit actual load values
    state.critical_load_kw = float(
        row["critical_load_kw"]
    )

    state.flexible_load_kw = float(
        row["flexible_load_kw"]
    )

    # --------------------------------------------------------
    # 2. Apply scenario modifications
    # --------------------------------------------------------

    if scenario == "STORM":

        state.storm_flag = True
        state.low_renewable_flag = True

    elif scenario == "LOW_RENEWABLE":

        state.low_renewable_flag = True

    elif scenario == "RESUPPLY_DELAY_4D":

        delay_days = 4.0

    elif scenario == "BATTERY_DEGRADATION":

        if battery_soh_override is None:
            battery_soh_override = 75.0

        state.battery_soh_pct = float(
            battery_soh_override
        )

        # Conservative capacity reduction.
        state.battery_usable_capacity_kwh = (
            state.battery_usable_capacity_kwh
            *
            state.battery_soh_pct
            / 100.0
        )

        state.battery_energy_kwh = min(
            state.battery_energy_kwh,
            state.battery_usable_capacity_kwh
        )

    elif scenario == "SCADA_ANOMALY":

        if anomaly_override is None:
            anomaly_override = 0.90

        state.scada_anomaly_score = float(
            anomaly_override
        )

        state.scada_anomaly_flag = True

    elif scenario == "COMMUNICATION_LOSS":

        state.communication_status = "LOCAL"

    elif scenario == "NORMAL":

        pass

    else:

        raise ValueError(
            f"Unknown scenario: {scenario}"
        )

    # Explicit communication override
    if communication_override is not None:
        state.communication_status = (
            communication_override
        )

    # Explicit anomaly override
    if anomaly_override is not None:
        state.scada_anomaly_score = float(
            anomaly_override
        )

        state.scada_anomaly_flag = (
            anomaly_override >= 0.5
        )

    # Explicit SOH override
    if battery_soh_override is not None:
        state.battery_soh_pct = float(
            battery_soh_override
        )

    # --------------------------------------------------------
    # 3. Resupply timing
    # --------------------------------------------------------

    p10 = (
        state.resupply_p10_days
        + delay_days
    )

    p50 = (
        state.resupply_p50_days
        + delay_days
    )

    p90 = (
        state.resupply_p90_days
        + delay_days
    )

    # --------------------------------------------------------
    # 4. Safe Operability
    # --------------------------------------------------------

    safe_result = calculate_safe_operability(
        state=state,
        optimizer_df=optimizer_data,
        start_timestamp=state.timestamp,
        horizon_hours=30 * 24
    )

    # --------------------------------------------------------
    # 5. CQRM
    # --------------------------------------------------------

    cqrm_result = calculate_cqrm(
        safe_operability_days=
            safe_result["safe_operability_days"],

        resupply_p10_days=p10,
        resupply_p50_days=p50,
        resupply_p90_days=p90
    )

    # --------------------------------------------------------
    # 6. CQRM reserve policy
    # --------------------------------------------------------

    reserve_policy = (
        calculate_cqrm_reserve_policy(
            cqrm_result=cqrm_result,
            battery_capacity_kwh=
                state.battery_usable_capacity_kwh
        )
    )

    # --------------------------------------------------------
    # 7. Optimizer
    # --------------------------------------------------------

    optimizer_result = optimize_station_v3(
        optimizer_df=optimizer_data,
        state=state,

        min_reserve_soc_pct=
            reserve_policy[
                "required_reserve_soc_pct"
            ],

        horizon_hours=168
    )

    # --------------------------------------------------------
    # 8. Safety validation
    # --------------------------------------------------------

    if optimizer_result["status"] == "OPTIMAL":

        safety_result = validate_optimizer_plan(
            state=state,
            optimizer_result=optimizer_result,
            cqrm_result=cqrm_result,
            plan=optimizer_result["plan"]
        )

    else:

        safety_result = {
            "status": "UNSAFE",

            "violations": [
                {
                    "code": "OPTIMIZER_INFEASIBLE",
                    "value": optimizer_result.get(
                        "message",
                        "infeasible"
                    ),
                    "limit": "feasible operating plan"
                }
            ],

            "violation_count": 1,

            "minimum_soc_pct":
                np.nan,

            "battery_soh_pct":
                state.battery_soh_pct,

            "battery_temperature_c":
                state.temperature_c,

            "critical_load_coverage_pct":
                0.0,

            "max_power_balance_error_kw":
                np.nan,

            "generator_available_kw":
                state.generator_available_kw,

            "final_fuel_l":
                state.fuel_remaining_l,

            "max_battery_discharge_kw":
                np.nan,

            "resupply_margin_days":
                cqrm_result["cqrm"]
        }

    # --------------------------------------------------------
    # 9. Final decision
    # --------------------------------------------------------

    if optimizer_result["status"] == "OPTIMAL":

        final_decision = run_final_decision(
            state=state,
            cqrm_result=cqrm_result,
            optimizer_result=optimizer_result,
            safety_result=safety_result
        )

    else:

        final_decision = {
            "final_decision":
                "REJECT_PLAN",

            "operating_mode":
                "CONSERVATION",

            "risk_level":
                cqrm_result["risk_level"],

            "cqrm_days":
                cqrm_result["cqrm"],

            "resupply_margin_days":
                cqrm_result["cqrm"],

            "recommended_action":
                "No feasible optimization plan was found. "
                "Enter conservation mode and escalate to "
                "the operator.",

            "reason":
                "Optimizer returned infeasible.",

            "requires_operator_intervention":
                True,

            "violations":
                ["OPTIMIZER_INFEASIBLE"]
        }

    # --------------------------------------------------------
    # 10. Unified output
    # --------------------------------------------------------

    return {
        "scenario": scenario,

        "state": state,

        "safe_operability":
            safe_result,

        "cqrm":
            cqrm_result,

        "reserve_policy":
            reserve_policy,

        "optimizer":
            optimizer_result,

        "safety":
            safety_result,

        "final_decision":
            final_decision
    }


# ============================================================
# TEST NORMAL SCENARIO
# ============================================================

normal_result = run_polar_ems_scenario(
    "NORMAL"
)

print("=" * 80)
print("POLAR-EMS SCENARIO ENGINE — NORMAL")
print("=" * 80)

print(
    "Safe operability:",
    round(
        normal_result[
            "safe_operability"
        ]["safe_operability_days"],
        3
    ),
    "days"
)

print(
    "CQRM:",
    round(
        normal_result[
            "cqrm"
        ]["cqrm"],
        3
    ),
    "days"
)

print(
    "Risk:",
    normal_result[
        "cqrm"
    ]["risk_level"]
)

print(
    "Optimizer:",
    normal_result[
        "optimizer"
    ]["status"]
)

print(
    "Safety:",
    normal_result[
        "safety"
    ]["status"]
)

print(
    "Final decision:",
    normal_result[
        "final_decision"
    ]["final_decision"]
)

print("\n" + "=" * 80)
print("✅ SCENARIO ENGINE NORMAL TEST COMPLETE")
print("=" * 80)

POLAR-EMS SCENARIO ENGINE — NORMAL
Safe operability: 10.792 days
CQRM: 0.492 days
Risk: CAUTION
Optimizer: OPTIMAL
Safety: SAFE
Final decision: ACCEPT_PLAN

✅ SCENARIO ENGINE NORMAL TEST COMPLETE


In [24]:
# ============================================================
# STEP 17 — POLAR-EMS FULL SCENARIO EVALUATION
# ============================================================

scenarios = [
    ("NORMAL", {}),
    ("STORM", {}),
    ("LOW_RENEWABLE", {}),
    ("RESUPPLY_DELAY_4D", {}),
    ("BATTERY_DEGRADATION", {
        "battery_soh_override": 75.0
    }),
    ("SCADA_ANOMALY", {
        "anomaly_override": 0.90
    }),
    ("COMMUNICATION_LOSS", {
        "communication_override": "LOCAL"
    }),
]


results = []


for scenario_name, kwargs in scenarios:

    print(
        f"\nRunning scenario: {scenario_name}"
    )

    try:

        result = run_polar_ems_scenario(
            scenario=scenario_name,
            **kwargs
        )

        safe = result["safe_operability"]
        cqrm = result["cqrm"]
        opt = result["optimizer"]
        safety = result["safety"]
        final = result["final_decision"]

        results.append({

            "Scenario":
                scenario_name,

            "Safe_Operability_days":
                round(
                    safe["safe_operability_days"],
                    3
                ),

            "CQRM_days":
                round(
                    cqrm["cqrm"],
                    3
                ),

            "Risk":
                cqrm["risk_level"],

            "Reserve_SOC_pct":
                round(
                    result[
                        "reserve_policy"
                    ]["required_reserve_soc_pct"],
                    2
                ),

            "Optimizer":
                opt["status"],

            "Safety":
                safety["status"],

            "Final_Decision":
                final["final_decision"],

            "Operating_Mode":
                final["operating_mode"],

            "Violations":
                safety["violation_count"],

            "Operator_Intervention":
                final[
                    "requires_operator_intervention"
                ]
        })

    except Exception as e:

        results.append({

            "Scenario":
                scenario_name,

            "Safe_Operability_days":
                np.nan,

            "CQRM_days":
                np.nan,

            "Risk":
                "ERROR",

            "Reserve_SOC_pct":
                np.nan,

            "Optimizer":
                "ERROR",

            "Safety":
                "ERROR",

            "Final_Decision":
                "ERROR",

            "Operating_Mode":
                "ERROR",

            "Violations":
                np.nan,

            "Operator_Intervention":
                True,

            "Error":
                str(e)
        })


# ============================================================
# RESULTS TABLE
# ============================================================

scenario_results_df = pd.DataFrame(results)

print("\n")
print("=" * 100)
print("POLAR-EMS — FULL SCENARIO EVALUATION")
print("=" * 100)

display(
    scenario_results_df
)


# ============================================================
# SAVE RESULTS
# ============================================================

scenario_results_df.to_csv(
    "/content/polar_ems_scenario_results.csv",
    index=False
)

print(
    "\n✅ Results saved to:"
    "\n/content/polar_ems_scenario_results.csv"
)


# ============================================================
# BASIC BEHAVIOR CHECKS
# ============================================================

normal_row = scenario_results_df[
    scenario_results_df["Scenario"] == "NORMAL"
].iloc[0]

delay_row = scenario_results_df[
    scenario_results_df["Scenario"] == "RESUPPLY_DELAY_4D"
].iloc[0]


assert delay_row["CQRM_days"] < normal_row["CQRM_days"]

assert (
    delay_row["Reserve_SOC_pct"]
    >
    normal_row["Reserve_SOC_pct"]
)

print("\n✅ Resupply-delay CQRM check passed.")
print("✅ Resupply-delay reserve check passed.")

print("\n" + "=" * 100)
print("STEP 17 COMPLETE")
print("=" * 100)


Running scenario: NORMAL

Running scenario: STORM

Running scenario: LOW_RENEWABLE

Running scenario: RESUPPLY_DELAY_4D

Running scenario: BATTERY_DEGRADATION

Running scenario: SCADA_ANOMALY

Running scenario: COMMUNICATION_LOSS


POLAR-EMS — FULL SCENARIO EVALUATION


,Scenario,Safe_Operability_days,CQRM_days,Risk,Reserve_SOC_pct,Optimizer,Safety,Final_Decision,Operating_Mode,Violations,Operator_Intervention
0,NORMAL,10.792,0.492,CAUTION,55.00,OPTIMAL,SAFE,ACCEPT_PLAN,NORMAL,0,False
1,STORM,10.792,0.492,CAUTION,55.00,OPTIMAL,SAFE,ACCEPT_PLAN,NORMAL,0,False
2,LOW_RENEWABLE,10.792,0.492,CAUTION,55.00,OPTIMAL,SAFE,ACCEPT_PLAN,NORMAL,0,False
3,RESUPPLY_DELAY_4D,10.792,-3.508,CRITICAL,76.75,OPTIMAL,UNSAFE,REJECT_PLAN,CONSERVATION,1,True
4,BATTERY_DEGRADATION,10.792,0.492,CAUTION,55.00,OPTIMAL,SAFE,ACCEPT_PLAN,NORMAL,0,False
5,SCADA_ANOMALY,10.792,0.492,CAUTION,55.00,OPTIMAL,SAFE,ACCEPT_PLAN,NORMAL,0,False
6,COMMUNICATION_LOSS,10.792,0.492,CAUTION,55.00,OPTIMAL,SAFE,ACCEPT_PLAN,NORMAL,0,False



✅ Results saved to:
/content/polar_ems_scenario_results.csv

✅ Resupply-delay CQRM check passed.
✅ Resupply-delay reserve check passed.

STEP 17 COMPLETE


In [25]:
# ============================================================
# STEP 18 — CORRECTED SCENARIO ENGINE
# ============================================================

import numpy as np
import pandas as pd


def build_scenario_profile(
    base_df,
    scenario,
    delay_days=0.0
):
    """
    Create a scenario-specific future operating profile.

    IMPORTANT:
    The original base dataset is never modified.
    Each scenario receives its own copy.
    """

    df = base_df.copy()

    scenario = scenario.upper().strip()

    # --------------------------------------------------------
    # STORM
    # --------------------------------------------------------
    #
    # Reduce renewable availability during storm conditions.
    # This is a prototype stress factor, not a measured
    # Antarctic weather relationship.
    # --------------------------------------------------------

    if scenario == "STORM":

        wind_factor = 0.45
        solar_factor = 0.65

        df["wind_forecast_kw"] *= wind_factor
        df["solar_forecast_kw"] *= solar_factor

        df["renewable_forecast_kw"] = (
            df["solar_forecast_kw"]
            + df["wind_forecast_kw"]
        )

        df["net_load_kw"] = np.maximum(
            0.0,
            df["load_forecast_kw"]
            -
            df["renewable_forecast_kw"]
        )

        df["storm_flag"] = 1
        df["low_renewable_flag"] = 1


    # --------------------------------------------------------
    # LOW RENEWABLE
    # --------------------------------------------------------

    elif scenario == "LOW_RENEWABLE":

        wind_factor = 0.60
        solar_factor = 0.50

        df["wind_forecast_kw"] *= wind_factor
        df["solar_forecast_kw"] *= solar_factor

        df["renewable_forecast_kw"] = (
            df["solar_forecast_kw"]
            + df["wind_forecast_kw"]
        )

        df["net_load_kw"] = np.maximum(
            0.0,
            df["load_forecast_kw"]
            -
            df["renewable_forecast_kw"]
        )

        df["low_renewable_flag"] = 1


    # --------------------------------------------------------
    # BATTERY DEGRADATION
    # --------------------------------------------------------
    #
    # The future energy profile itself remains the same.
    # Battery capacity is modified separately in the state.
    # --------------------------------------------------------

    elif scenario == "BATTERY_DEGRADATION":

        pass


    # --------------------------------------------------------
    # SCADA ANOMALY
    # --------------------------------------------------------
    #
    # For now, treat detected abnormality as generator
    # derating to make the operational impact explicit.
    # This is a prototype scenario assumption.
    # --------------------------------------------------------

    elif scenario == "SCADA_ANOMALY":

        if "generator_available_kw" in df.columns:

            df["generator_available_kw"] *= 0.75


    # --------------------------------------------------------
    # COMMUNICATION LOSS
    # --------------------------------------------------------
    #
    # No physical generation change.
    # Core decision engine continues locally.
    # --------------------------------------------------------

    elif scenario == "COMMUNICATION_LOSS":

        pass


    # --------------------------------------------------------
    # RESUPPLY DELAY
    # --------------------------------------------------------
    #
    # Delay is handled through CQRM outside this profile.
    # --------------------------------------------------------

    elif scenario == "RESUPPLY_DELAY_4D":

        pass


    elif scenario == "NORMAL":

        pass


    else:

        raise ValueError(
            f"Unknown scenario: {scenario}"
        )


    return df


# ============================================================
# FULL CORRECTED PIPELINE
# ============================================================

def run_polar_ems_scenario_v2(
    scenario="NORMAL",
    delay_days=0.0,
    battery_soh_override=None,
    anomaly_override=None,
    communication_override=None
):
    """
    Corrected end-to-end POLAR-EMS scenario engine.

    Scenario effects are injected into the actual future
    operating profile before Safe Operability and Optimization.
    """

    scenario = scenario.upper().strip()

    # --------------------------------------------------------
    # Base station state
    # --------------------------------------------------------

    row = optimizer_data.iloc[0]

    state = create_station_state(
        timestamp=row["timestamp"],

        load_kw=row["load_forecast_kw"],
        solar_kw=row["solar_forecast_kw"],
        wind_kw=row["wind_forecast_kw"],

        battery_soc_pct=row["battery_soc_pct"],
        battery_soh_pct=row["battery_soh_pct"],

        battery_usable_capacity_kwh=
            row["battery_usable_capacity_kwh"],

        battery_energy_kwh=
            row["battery_energy_kwh"],

        generator_available_kw=
            row["generator_available_kw"],

        generator_min_kw=
            row["generator_min_kw"],

        fuel_remaining_l=
            row["diesel_fuel_l"],

        temperature_c=row["temperature_c"],

        wind_speed_ms=row["wind_speed_ms"],

        storm_flag=bool(row["storm_flag"]),

        low_renewable_flag=
            bool(row["low_renewable_flag"]),

        communication_status=
            row["communication_status"],

        scada_anomaly_score=None,
        scada_anomaly_flag=False,

        resupply_p10_days=
            row["resupply_eta_p10_days"],

        resupply_p50_days=
            row["resupply_eta_p50_days"],

        resupply_p90_days=
            row["resupply_eta_p90_days"],

        critical_load_ratio=(
            float(row["critical_load_kw"])
            /
            max(
                float(row["load_forecast_kw"]),
                1e-9
            )
        )
    )

    # Explicit critical/flexible loads
    state.critical_load_kw = float(
        row["critical_load_kw"]
    )

    state.flexible_load_kw = float(
        row["flexible_load_kw"]
    )

    # --------------------------------------------------------
    # Scenario-specific station changes
    # --------------------------------------------------------

    if scenario == "STORM":

        state.storm_flag = True
        state.low_renewable_flag = True

    elif scenario == "LOW_RENEWABLE":

        state.low_renewable_flag = True

    elif scenario == "RESUPPLY_DELAY_4D":

        delay_days = 4.0

    elif scenario == "BATTERY_DEGRADATION":

        if battery_soh_override is None:
            battery_soh_override = 75.0

        state.battery_soh_pct = float(
            battery_soh_override
        )

        state.battery_usable_capacity_kwh *= (
            state.battery_soh_pct / 100.0
        )

        state.battery_energy_kwh = min(
            state.battery_energy_kwh,
            state.battery_usable_capacity_kwh
        )

    elif scenario == "SCADA_ANOMALY":

        if anomaly_override is None:
            anomaly_override = 0.90

        state.scada_anomaly_score = float(
            anomaly_override
        )

        state.scada_anomaly_flag = True

    elif scenario == "COMMUNICATION_LOSS":

        # Local operation remains available.
        state.communication_status = "LOCAL"

    # Explicit overrides
    if communication_override is not None:

        state.communication_status = str(
            communication_override
        )

    if anomaly_override is not None:

        state.scada_anomaly_score = float(
            anomaly_override
        )

        state.scada_anomaly_flag = True

    # --------------------------------------------------------
    # Build scenario-specific future profile
    # --------------------------------------------------------

    scenario_profile = build_scenario_profile(
        optimizer_data,
        scenario=scenario,
        delay_days=delay_days
    )

    # --------------------------------------------------------
    # Resupply scenario
    # --------------------------------------------------------

    p10 = (
        state.resupply_p10_days
        + delay_days
    )

    p50 = (
        state.resupply_p50_days
        + delay_days
    )

    p90 = (
        state.resupply_p90_days
        + delay_days
    )

    # --------------------------------------------------------
    # Safe Operability
    # --------------------------------------------------------

    safe_result = calculate_safe_operability(
        state=state,
        optimizer_df=scenario_profile,
        start_timestamp=state.timestamp,
        horizon_hours=30 * 24
    )

    # --------------------------------------------------------
    # CQRM
    # --------------------------------------------------------

    cqrm_result = calculate_cqrm(
        safe_operability_days=
            safe_result["safe_operability_days"],

        resupply_p10_days=p10,
        resupply_p50_days=p50,
        resupply_p90_days=p90
    )

    # --------------------------------------------------------
    # CQRM reserve policy
    # --------------------------------------------------------

    reserve_policy = (
        calculate_cqrm_reserve_policy(
            cqrm_result=cqrm_result,

            battery_capacity_kwh=
                state.battery_usable_capacity_kwh
        )
    )

    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    optimizer_result = optimize_station_v3(
        optimizer_df=scenario_profile,

        state=state,

        min_reserve_soc_pct=
            reserve_policy[
                "required_reserve_soc_pct"
            ],

        horizon_hours=168
    )

    # --------------------------------------------------------
    # Safety
    # --------------------------------------------------------

    if optimizer_result["status"] == "OPTIMAL":

        safety_result = validate_optimizer_plan(
            state=state,

            optimizer_result=
                optimizer_result,

            cqrm_result=
                cqrm_result,

            plan=optimizer_result["plan"]
        )

    else:

        safety_result = {
            "status": "UNSAFE",

            "violations": [
                {
                    "code":
                        "OPTIMIZER_INFEASIBLE",

                    "value":
                        optimizer_result.get(
                            "message",
                            "infeasible"
                        ),

                    "limit":
                        "feasible operating plan"
                }
            ],

            "violation_count": 1,

            "minimum_soc_pct": np.nan,

            "battery_soh_pct":
                state.battery_soh_pct,

            "battery_temperature_c":
                state.temperature_c,

            "critical_load_coverage_pct":
                0.0,

            "max_power_balance_error_kw":
                np.nan,

            "generator_available_kw":
                state.generator_available_kw,

            "final_fuel_l":
                state.fuel_remaining_l,

            "max_battery_discharge_kw":
                np.nan,

            "resupply_margin_days":
                cqrm_result["cqrm"]
        }

    # --------------------------------------------------------
    # Final decision
    # --------------------------------------------------------

    if optimizer_result["status"] == "OPTIMAL":

        final_decision = run_final_decision(
            state=state,

            cqrm_result=cqrm_result,

            optimizer_result=
                optimizer_result,

            safety_result=
                safety_result
        )

    else:

        final_decision = {
            "final_decision":
                "REJECT_PLAN",

            "operating_mode":
                "CONSERVATION",

            "risk_level":
                cqrm_result["risk_level"],

            "cqrm_days":
                cqrm_result["cqrm"],

            "resupply_margin_days":
                cqrm_result["cqrm"],

            "recommended_action":
                "No feasible optimization plan was found. "
                "Enter conservation mode and escalate to "
                "the operator.",

            "reason":
                "Optimizer returned infeasible.",

            "requires_operator_intervention":
                True,

            "violations":
                ["OPTIMIZER_INFEASIBLE"]
        }

    return {
        "scenario": scenario,

        "state": state,

        "scenario_profile":
            scenario_profile,

        "safe_operability":
            safe_result,

        "cqrm":
            cqrm_result,

        "reserve_policy":
            reserve_policy,

        "optimizer":
            optimizer_result,

        "safety":
            safety_result,

        "final_decision":
            final_decision
    }


print("=" * 80)
print("✅ CORRECTED SCENARIO ENGINE READY")
print("=" * 80)

✅ CORRECTED SCENARIO ENGINE READY


In [26]:
# ============================================================
# STEP 18B — TEST SCENARIO DIFFERENTIATION
# ============================================================

test_scenarios = [
    "NORMAL",
    "STORM",
    "LOW_RENEWABLE"
]

for name in test_scenarios:

    result = run_polar_ems_scenario_v2(
        scenario=name
    )

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    print(
        "Renewable first hour:",
        result[
            "scenario_profile"
        ]["renewable_forecast_kw"].iloc[0]
    )

    print(
        "Safe operability:",
        round(
            result[
                "safe_operability"
            ]["safe_operability_days"],
            3
        ),
        "days"
    )

    print(
        "CQRM:",
        round(
            result[
                "cqrm"
            ]["cqrm"],
            3
        )
    )

    print(
        "Risk:",
        result[
            "cqrm"
        ]["risk_level"]
    )

    print(
        "Optimizer:",
        result[
            "optimizer"
        ]["status"]
    )

    print(
        "Safety:",
        result[
            "safety"
        ]["status"]
    )

print("\n" + "=" * 80)
print("✅ SCENARIO DIFFERENTIATION TEST COMPLETE")
print("=" * 80)


NORMAL
Renewable first hour: 233.45
Safe operability: 10.792 days
CQRM: 0.492
Risk: CAUTION
Optimizer: OPTIMAL
Safety: SAFE

STORM
Renewable first hour: 105.0525
Safe operability: 5.583 days
CQRM: -4.717
Risk: CRITICAL
Optimizer: INFEASIBLE
Safety: UNSAFE

LOW_RENEWABLE
Renewable first hour: 140.07
Safe operability: 6.125 days
CQRM: -4.175
Risk: CRITICAL
Optimizer: OPTIMAL
Safety: UNSAFE

✅ SCENARIO DIFFERENTIATION TEST COMPLETE


In [27]:
# ============================================================
# STEP 19 — FULL CORRECTED POLAR-EMS SCENARIO EVALUATION
# ============================================================

scenarios = [
    ("NORMAL", {}),
    ("STORM", {}),
    ("LOW_RENEWABLE", {}),
    ("RESUPPLY_DELAY_4D", {}),
    ("BATTERY_DEGRADATION", {
        "battery_soh_override": 75.0
    }),
    ("SCADA_ANOMALY", {
        "anomaly_override": 0.90
    }),
    ("COMMUNICATION_LOSS", {
        "communication_override": "LOCAL"
    }),
]

results_v2 = []

for scenario_name, kwargs in scenarios:

    print(f"\nRunning: {scenario_name}")

    try:

        result = run_polar_ems_scenario_v2(
            scenario=scenario_name,
            **kwargs
        )

        safe = result["safe_operability"]
        cqrm = result["cqrm"]
        reserve = result["reserve_policy"]
        optimizer = result["optimizer"]
        safety = result["safety"]
        final = result["final_decision"]
        state = result["state"]

        results_v2.append({
            "Scenario":
                scenario_name,

            "SOH_pct":
                round(
                    state.battery_soh_pct,
                    2
                ),

            "Renewable_First_Hour_kW":
                round(
                    result[
                        "scenario_profile"
                    ]["renewable_forecast_kw"].iloc[0],
                    2
                ),

            "Safe_Operability_days":
                round(
                    safe[
                        "safe_operability_days"
                    ],
                    3
                ),

            "CQRM_days":
                round(
                    cqrm["cqrm"],
                    3
                ),

            "Risk":
                cqrm["risk_level"],

            "Reserve_SOC_pct":
                round(
                    reserve[
                        "required_reserve_soc_pct"
                    ],
                    2
                ),

            "Optimizer":
                optimizer["status"],

            "Safety":
                safety["status"],

            "Final_Decision":
                final["final_decision"],

            "Operating_Mode":
                final["operating_mode"],

            "Violations":
                safety["violation_count"],

            "Operator_Intervention":
                final[
                    "requires_operator_intervention"
                ]
        })

    except Exception as e:

        results_v2.append({
            "Scenario":
                scenario_name,

            "SOH_pct":
                np.nan,

            "Renewable_First_Hour_kW":
                np.nan,

            "Safe_Operability_days":
                np.nan,

            "CQRM_days":
                np.nan,

            "Risk":
                "ERROR",

            "Reserve_SOC_pct":
                np.nan,

            "Optimizer":
                "ERROR",

            "Safety":
                "ERROR",

            "Final_Decision":
                "ERROR",

            "Operating_Mode":
                "ERROR",

            "Violations":
                np.nan,

            "Operator_Intervention":
                True,

            "Error":
                str(e)
        })


# ============================================================
# CREATE RESULTS TABLE
# ============================================================

scenario_results_v2 = pd.DataFrame(
    results_v2
)

print("\n")
print("=" * 110)
print("POLAR-EMS — CORRECTED FULL SCENARIO EVALUATION")
print("=" * 110)

display(
    scenario_results_v2
)


# ============================================================
# SAVE
# ============================================================

scenario_results_v2.to_csv(
    "/content/polar_ems_corrected_scenario_results.csv",
    index=False
)

print(
    "\n✅ Saved:"
    "\n/content/polar_ems_corrected_scenario_results.csv"
)


# ============================================================
# CORE BEHAVIOR CHECKS
# ============================================================

normal = scenario_results_v2[
    scenario_results_v2["Scenario"] == "NORMAL"
].iloc[0]

storm = scenario_results_v2[
    scenario_results_v2["Scenario"] == "STORM"
].iloc[0]

low_renewable = scenario_results_v2[
    scenario_results_v2["Scenario"] == "LOW_RENEWABLE"
].iloc[0]

delay = scenario_results_v2[
    scenario_results_v2["Scenario"] == "RESUPPLY_DELAY_4D"
].iloc[0]


# Storm should reduce renewable availability.
assert (
    storm["Renewable_First_Hour_kW"]
    <
    normal["Renewable_First_Hour_kW"]
)

# Low renewable should reduce renewable availability.
assert (
    low_renewable["Renewable_First_Hour_kW"]
    <
    normal["Renewable_First_Hour_kW"]
)

# Resupply delay must reduce CQRM.
assert (
    delay["CQRM_days"]
    <
    normal["CQRM_days"]
)

# Resupply delay must increase reserve.
assert (
    delay["Reserve_SOC_pct"]
    >
    normal["Reserve_SOC_pct"]
)


print("\n" + "=" * 110)
print("✅ CORE SCENARIO BEHAVIOR CHECKS PASSED")
print("=" * 110)


Running: NORMAL

Running: STORM

Running: LOW_RENEWABLE

Running: RESUPPLY_DELAY_4D

Running: BATTERY_DEGRADATION

Running: SCADA_ANOMALY

Running: COMMUNICATION_LOSS


POLAR-EMS — CORRECTED FULL SCENARIO EVALUATION


,Scenario,SOH_pct,Renewable_First_Hour_kW,Safe_Operability_days,CQRM_days,Risk,Reserve_SOC_pct,Optimizer,Safety,Final_Decision,Operating_Mode,Violations,Operator_Intervention
0,NORMAL,95.91,233.45,10.792,0.492,CAUTION,55.00,OPTIMAL,SAFE,ACCEPT_PLAN,NORMAL,0,False
1,STORM,95.91,105.05,5.583,-4.717,CRITICAL,77.36,INFEASIBLE,UNSAFE,REJECT_PLAN,CONSERVATION,1,True
2,LOW_RENEWABLE,95.91,140.07,6.125,-4.175,CRITICAL,77.09,OPTIMAL,UNSAFE,REJECT_PLAN,CONSERVATION,1,True
3,RESUPPLY_DELAY_4D,95.91,233.45,10.792,-3.508,CRITICAL,76.75,OPTIMAL,UNSAFE,REJECT_PLAN,CONSERVATION,1,True
4,BATTERY_DEGRADATION,75.00,233.45,10.792,0.492,CAUTION,55.00,OPTIMAL,SAFE,ACCEPT_PLAN,NORMAL,0,False
5,SCADA_ANOMALY,95.91,233.45,10.792,0.492,CAUTION,55.00,OPTIMAL,SAFE,ACCEPT_PLAN,NORMAL,0,False
6,COMMUNICATION_LOSS,95.91,233.45,10.792,0.492,CAUTION,55.00,OPTIMAL,SAFE,ACCEPT_PLAN,NORMAL,0,False



✅ Saved:
/content/polar_ems_corrected_scenario_results.csv

✅ CORE SCENARIO BEHAVIOR CHECKS PASSED


In [28]:
# ============================================================
# STEP 20 — SCENARIO SENSITIVITY / OPERATIONAL IMPACT TEST
# ============================================================

scenario_names = [
    "NORMAL",
    "STORM",
    "LOW_RENEWABLE",
    "RESUPPLY_DELAY_4D",
    "BATTERY_DEGRADATION",
    "SCADA_ANOMALY",
    "COMMUNICATION_LOSS"
]

sensitivity_rows = []

for name in scenario_names:

    kwargs = {}

    if name == "BATTERY_DEGRADATION":
        kwargs["battery_soh_override"] = 75.0

    elif name == "SCADA_ANOMALY":
        kwargs["anomaly_override"] = 0.90

    elif name == "COMMUNICATION_LOSS":
        kwargs["communication_override"] = "LOCAL"

    result = run_polar_ems_scenario_v2(
        scenario=name,
        **kwargs
    )

    state = result["state"]
    profile = result["scenario_profile"]
    safe = result["safe_operability"]
    cqrm = result["cqrm"]
    opt = result["optimizer"]
    safety = result["safety"]

    row = {
        "Scenario": name,

        "SOH_pct":
            state.battery_soh_pct,

        "Battery_Capacity_kWh":
            state.battery_usable_capacity_kwh,

        "Initial_Battery_Energy_kWh":
            state.battery_energy_kwh,

        "Fuel_L":
            state.fuel_remaining_l,

        "Initial_Renewable_kW":
            float(
                profile[
                    "renewable_forecast_kw"
                ].iloc[0]
            ),

        "Mean_Renewable_kW":
            float(
                profile[
                    "renewable_forecast_kw"
                ].mean()
            ),

        "Generator_Available_kW":
            float(
                profile[
                    "generator_available_kw"
                ].iloc[0]
            ),

        "Safe_Operability_days":
            safe["safe_operability_days"],

        "CQRM_days":
            cqrm["cqrm"],

        "Risk":
            cqrm["risk_level"],

        "Reserve_SOC_pct":
            result[
                "reserve_policy"
            ]["required_reserve_soc_pct"],

        "Optimizer_Status":
            opt["status"],

        "Min_SOC_pct":
            (
                opt["minimum_soc_pct"]
                if opt["status"] == "OPTIMAL"
                else np.nan
            ),

        "Final_SOC_pct":
            (
                opt["final_battery_soc_pct"]
                if opt["status"] == "OPTIMAL"
                else np.nan
            ),

        "Generator_Energy_kWh":
            (
                opt["generator_energy_kwh"]
                if opt["status"] == "OPTIMAL"
                else np.nan
            ),

        "Battery_Discharge_kWh":
            (
                opt["battery_discharge_kwh"]
                if opt["status"] == "OPTIMAL"
                else np.nan
            ),

        "Renewable_Used_kWh":
            (
                opt["renewable_used_kwh"]
                if opt["status"] == "OPTIMAL"
                else np.nan
            ),

        "Safety":
            safety["status"],

        "Violations":
            safety["violation_count"]
    }

    sensitivity_rows.append(row)


sensitivity_df = pd.DataFrame(
    sensitivity_rows
)


print("=" * 120)
print("POLAR-EMS — SCENARIO SENSITIVITY ANALYSIS")
print("=" * 120)

display(sensitivity_df)


# ------------------------------------------------------------
# Compare each scenario against NORMAL
# ------------------------------------------------------------

normal = sensitivity_df[
    sensitivity_df["Scenario"] == "NORMAL"
].iloc[0]


delta_df = sensitivity_df.copy()

delta_df["Δ Safe_Operability_days"] = (
    delta_df["Safe_Operability_days"]
    - normal["Safe_Operability_days"]
)

delta_df["Δ CQRM_days"] = (
    delta_df["CQRM_days"]
    - normal["CQRM_days"]
)

delta_df["Δ Reserve_SOC_pct"] = (
    delta_df["Reserve_SOC_pct"]
    - normal["Reserve_SOC_pct"]
)

delta_df["Δ Generator_Energy_kWh"] = (
    delta_df["Generator_Energy_kWh"]
    - normal["Generator_Energy_kWh"]
)

delta_df["Δ Battery_Discharge_kWh"] = (
    delta_df["Battery_Discharge_kWh"]
    - normal["Battery_Discharge_kWh"]
)


print("\n")
print("=" * 120)
print("CHANGE FROM NORMAL")
print("=" * 120)

display(
    delta_df[
        [
            "Scenario",
            "Δ Safe_Operability_days",
            "Δ CQRM_days",
            "Δ Reserve_SOC_pct",
            "Δ Generator_Energy_kWh",
            "Δ Battery_Discharge_kWh",
            "Optimizer_Status",
            "Safety"
        ]
    ]
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

sensitivity_df.to_csv(
    "/content/polar_ems_scenario_sensitivity.csv",
    index=False
)

delta_df.to_csv(
    "/content/polar_ems_scenario_sensitivity_delta.csv",
    index=False
)


print("\n✅ Sensitivity results saved.")
print("   /content/polar_ems_scenario_sensitivity.csv")
print("   /content/polar_ems_scenario_sensitivity_delta.csv")

print("\n" + "=" * 120)
print("✅ STEP 20 COMPLETE")
print("=" * 120)

POLAR-EMS — SCENARIO SENSITIVITY ANALYSIS


,Scenario,SOH_pct,Battery_Capacity_kWh,Initial_Battery_Energy_kWh,Fuel_L,Initial_Renewable_kW,Mean_Renewable_kW,Generator_Available_kW,Safe_Operability_days,CQRM_days,Risk,Reserve_SOC_pct,Optimizer_Status,Min_SOC_pct,Final_SOC_pct,Generator_Energy_kWh,Battery_Discharge_kWh,Renewable_Used_kWh,Safety,Violations
0,NORMAL,95.91,1150.88,701.06,6400.0,233.4500,213.871588,900.0,10.791667,0.491667,CAUTION,55.000000,OPTIMAL,20.0,58.350002,7496.527625,4647.767675,37034.002465,SAFE,0
1,STORM,95.91,1150.88,701.06,6400.0,105.0525,104.329550,900.0,5.583333,-4.716667,CRITICAL,77.358333,INFEASIBLE,NaN,NaN,NaN,NaN,NaN,UNSAFE,1
2,LOW_RENEWABLE,95.91,1150.88,701.06,6400.0,140.0700,124.279205,900.0,6.125000,-4.175000,CRITICAL,77.087500,OPTIMAL,20.0,77.087500,16546.641464,1731.039536,22598.809000,UNSAFE,1
3,RESUPPLY_DELAY_4D,95.91,1150.88,701.06,6400.0,233.4500,213.871588,900.0,10.791667,-3.508333,CRITICAL,76.754167,OPTIMAL,20.0,76.754167,7496.527625,4446.548314,37034.002465,UNSAFE,1
4,BATTERY_DEGRADATION,75.00,863.16,701.06,6400.0,233.4500,213.871588,900.0,10.791667,0.491667,CAUTION,55.000000,OPTIMAL,20.0,55.000000,7879.195225,4178.726550,36307.130886,SAFE,0
5,SCADA_ANOMALY,95.91,1150.88,701.06,6400.0,233.4500,213.871588,675.0,10.791667,0.491667,CAUTION,55.000000,OPTIMAL,20.0,58.350002,7496.527625,4647.767675,37034.002465,SAFE,0
6,COMMUNICATION_LOSS,95.91,1150.88,701.06,6400.0,233.4500,213.871588,900.0,10.791667,0.491667,CAUTION,55.000000,OPTIMAL,20.0,58.350002,7496.527625,4647.767675,37034.002465,SAFE,0




CHANGE FROM NORMAL


,Scenario,Δ Safe_Operability_days,Δ CQRM_days,Δ Reserve_SOC_pct,Δ Generator_Energy_kWh,Δ Battery_Discharge_kWh,Optimizer_Status,Safety
0,NORMAL,0.000000,0.000000,0.000000,0.000000,0.000000,OPTIMAL,SAFE
1,STORM,-5.208333,-5.208333,22.358333,NaN,NaN,INFEASIBLE,UNSAFE
2,LOW_RENEWABLE,-4.666667,-4.666667,22.087500,9050.113839,-2916.728139,OPTIMAL,UNSAFE
3,RESUPPLY_DELAY_4D,0.000000,-4.000000,21.754167,0.000000,-201.219361,OPTIMAL,UNSAFE
4,BATTERY_DEGRADATION,0.000000,0.000000,0.000000,382.667600,-469.041125,OPTIMAL,SAFE
5,SCADA_ANOMALY,0.000000,0.000000,0.000000,0.000000,0.000000,OPTIMAL,SAFE
6,COMMUNICATION_LOSS,0.000000,0.000000,0.000000,0.000000,0.000000,OPTIMAL,SAFE



✅ Sensitivity results saved.
   /content/polar_ems_scenario_sensitivity.csv
   /content/polar_ems_scenario_sensitivity_delta.csv

✅ STEP 20 COMPLETE


In [29]:
# ============================================================
# STEP 21 — BASELINE vs POLAR-EMS
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# SIMPLE RULE-BASED BASELINE
# ============================================================

def run_rule_based_baseline(
    scenario_profile,
    state,
    horizon_hours=168,
    fixed_reserve_soc_pct=20.0,
    fuel_to_energy_kwh_per_l=3.0
):
    """
    Simple conventional baseline.

    Policy:
      1. Renewable energy first
      2. Critical load always prioritized
      3. Battery supplies remaining demand while staying
         above a fixed 20% SOC floor
      4. Generator supplies remaining demand
      5. No CQRM
      6. No dynamic resupply reserve

    This is intentionally simple so that the comparison
    measures the value of the POLAR-EMS decision layer.
    """

    df = (
        scenario_profile.copy()
        .sort_values("timestamp")
        .reset_index(drop=True)
        .head(horizon_hours)
    )

    H = len(df)

    capacity = float(
        state.battery_usable_capacity_kwh
    )

    battery_energy = float(
        state.battery_energy_kwh
    )

    min_energy = (
        capacity *
        fixed_reserve_soc_pct /
        100.0
    )

    fuel_remaining = float(
        state.fuel_remaining_l
    )

    generator_min = float(
        state.generator_min_kw
    )

    records = []

    for _, row in df.iterrows():

        load = max(
            0.0,
            float(row["load_forecast_kw"])
        )

        critical = max(
            0.0,
            float(row["critical_load_kw"])
        )

        flexible = max(
            0.0,
            float(row["flexible_load_kw"])
        )

        solar = max(
            0.0,
            float(row["solar_forecast_kw"])
        )

        wind = max(
            0.0,
            float(row["wind_forecast_kw"])
        )

        renewable = solar + wind

        generator_available = max(
            0.0,
            float(row["generator_available_kw"])
        )

        # ----------------------------------------------------
        # Renewable first
        # ----------------------------------------------------

        renewable_used = min(
            renewable,
            load
        )

        remaining = max(
            0.0,
            load - renewable_used
        )

        # ----------------------------------------------------
        # Battery
        # ----------------------------------------------------

        battery_available = max(
            0.0,
            battery_energy - min_energy
        )

        battery_discharge = min(
            remaining,
            battery_available,
            MAX_BATTERY_DISCHARGE_KW
        )

        battery_energy -= (
            battery_discharge /
            ETA_DISCHARGE
        )

        battery_energy = max(
            min_energy,
            battery_energy
        )

        remaining -= battery_discharge

        # ----------------------------------------------------
        # Generator
        # ----------------------------------------------------

        generator = min(
            remaining,
            generator_available
        )

        # If generator has a minimum-load requirement,
        # avoid inventing generation below zero.
        if (
            0 < generator < generator_min
            and remaining > 0
        ):
            generator = min(
                generator_min,
                generator_available
            )

        remaining_after_generator = max(
            0.0,
            remaining - generator
        )

        # Critical load coverage is assessed separately.
        critical_remaining = max(
            0.0,
            critical - renewable_used
        )

        battery_for_critical = min(
            critical_remaining,
            battery_discharge
        )

        critical_remaining -= (
            battery_for_critical
        )

        generator_for_critical = min(
            critical_remaining,
            generator
        )

        critical_remaining -= (
            generator_for_critical
        )

        critical_coverage = (
            100.0
            if critical_remaining <= 1e-9
            else
            100.0 *
            max(
                0.0,
                critical -
                critical_remaining
            )
            /
            max(critical, 1e-9)
        )

        fuel_used = (
            generator /
            max(
                fuel_to_energy_kwh_per_l,
                1e-9
            )
        )

        fuel_remaining -= fuel_used

        fuel_remaining = max(
            0.0,
            fuel_remaining
        )

        soc_pct = (
            100.0 *
            battery_energy /
            max(capacity, 1e-9)
        )

        balance_error = (
            renewable_used
            + battery_discharge
            + generator
            - load
        )

        records.append({

            "timestamp":
                row["timestamp"],

            "load_kw":
                load,

            "critical_load_kw":
                critical,

            "renewable_available_kw":
                renewable,

            "renewable_used_kw":
                renewable_used,

            "battery_discharge_kw":
                battery_discharge,

            "battery_energy_kwh":
                battery_energy,

            "battery_soc_pct":
                soc_pct,

            "generator_kw":
                generator,

            "fuel_used_l":
                fuel_used,

            "fuel_remaining_l":
                fuel_remaining,

            "critical_load_coverage_pct":
                critical_coverage,

            "power_balance_error_kw":
                balance_error,

            "unserved_load_kw":
                remaining_after_generator
        })

    plan = pd.DataFrame(records)

    return {

        "status": "BASELINE",

        "plan": plan,

        "fuel_used_l":
            float(
                plan["fuel_used_l"].sum()
            ),

        "generator_energy_kwh":
            float(
                plan["generator_kw"].sum()
            ),

        "battery_discharge_kwh":
            float(
                plan["battery_discharge_kw"].sum()
            ),

        "renewable_used_kwh":
            float(
                plan["renewable_used_kw"].sum()
            ),

        "minimum_soc_pct":
            float(
                plan["battery_soc_pct"].min()
            ),

        "final_soc_pct":
            float(
                plan["battery_soc_pct"].iloc[-1]
            ),

        "minimum_critical_coverage_pct":
            float(
                plan[
                    "critical_load_coverage_pct"
                ].min()
            ),

        "max_power_balance_error_kw":
            float(
                plan[
                    "power_balance_error_kw"
                ].abs().max()
            ),

        "unserved_load_kwh":
            float(
                plan["unserved_load_kw"].sum()
            )
    }


# ============================================================
# RUN BASELINE + POLAR-EMS
# ============================================================

baseline_rows = []


comparison_scenarios = [
    "NORMAL",
    "LOW_RENEWABLE",
    "RESUPPLY_DELAY_4D",
    "BATTERY_DEGRADATION",
    "STORM"
]


for scenario_name in comparison_scenarios:

    # --------------------------------------------------------
    # Get POLAR-EMS result
    # --------------------------------------------------------

    kwargs = {}

    if scenario_name == "BATTERY_DEGRADATION":
        kwargs["battery_soh_override"] = 75.0

    result = run_polar_ems_scenario_v2(
        scenario=scenario_name,
        **kwargs
    )

    state = result["state"]
    profile = result["scenario_profile"]

    # --------------------------------------------------------
    # Baseline
    # --------------------------------------------------------

    baseline = run_rule_based_baseline(
        scenario_profile=profile,
        state=state,
        horizon_hours=168
    )

    # --------------------------------------------------------
    # POLAR-EMS
    # --------------------------------------------------------

    opt = result["optimizer"]

    if opt["status"] == "OPTIMAL":

        polar_fuel = opt[
            "fuel_used_l"
        ]

        polar_generator = opt[
            "generator_energy_kwh"
        ]

        polar_battery = opt[
            "battery_discharge_kwh"
        ]

        polar_renewable = opt[
            "renewable_used_kwh"
        ]

        polar_min_soc = opt[
            "minimum_soc_pct"
        ]

        polar_final_soc = opt[
            "final_battery_soc_pct"
        ]

        polar_balance = opt[
            "max_power_balance_error_kw"
        ]

        polar_critical = opt[
            "critical_load_coverage_pct"
        ]

        polar_status = "OPTIMAL"

    else:

        polar_fuel = np.nan
        polar_generator = np.nan
        polar_battery = np.nan
        polar_renewable = np.nan
        polar_min_soc = np.nan
        polar_final_soc = np.nan
        polar_balance = np.nan
        polar_critical = np.nan
        polar_status = opt["status"]

    baseline_rows.append({

        "Scenario":
            scenario_name,

        # -----------------------------
        # POLAR-EMS
        # -----------------------------

        "POLAR_Status":
            polar_status,

        "POLAR_Fuel_Used_L":
            polar_fuel,

        "POLAR_Generator_Energy_kWh":
            polar_generator,

        "POLAR_Battery_Discharge_kWh":
            polar_battery,

        "POLAR_Renewable_Used_kWh":
            polar_renewable,

        "POLAR_Min_SOC_pct":
            polar_min_soc,

        "POLAR_Final_SOC_pct":
            polar_final_soc,

        "POLAR_Critical_Coverage_pct":
            polar_critical,

        "POLAR_Balance_Error_kW":
            polar_balance,

        # -----------------------------
        # BASELINE
        # -----------------------------

        "BASELINE_Fuel_Used_L":
            baseline["fuel_used_l"],

        "BASELINE_Generator_Energy_kWh":
            baseline[
                "generator_energy_kwh"
            ],

        "BASELINE_Battery_Discharge_kWh":
            baseline[
                "battery_discharge_kwh"
            ],

        "BASELINE_Renewable_Used_kWh":
            baseline[
                "renewable_used_kwh"
            ],

        "BASELINE_Min_SOC_pct":
            baseline[
                "minimum_soc_pct"
            ],

        "BASELINE_Final_SOC_pct":
            baseline[
                "final_soc_pct"
            ],

        "BASELINE_Critical_Coverage_pct":
            baseline[
                "minimum_critical_coverage_pct"
            ],

        "BASELINE_Balance_Error_kW":
            baseline[
                "max_power_balance_error_kw"
            ],

        "BASELINE_Unserved_Load_kWh":
            baseline[
                "unserved_load_kwh"
            ]
    })


baseline_comparison_df = pd.DataFrame(
    baseline_rows
)


print("=" * 140)
print("POLAR-EMS vs RULE-BASED BASELINE")
print("=" * 140)

display(
    baseline_comparison_df
)


# ============================================================
# CALCULATE DELTAS
# ============================================================

delta = baseline_comparison_df.copy()

delta["Fuel_Difference_L"] = (
    delta["BASELINE_Fuel_Used_L"]
    -
    delta["POLAR_Fuel_Used_L"]
)

delta["Battery_Discharge_Difference_kWh"] = (
    delta[
        "BASELINE_Battery_Discharge_kWh"
    ]
    -
    delta[
        "POLAR_Battery_Discharge_kWh"
    ]
)

delta["Final_SOC_Difference_pct"] = (
    delta["POLAR_Final_SOC_pct"]
    -
    delta["BASELINE_Final_SOC_pct"]
)

delta["Critical_Coverage_Difference_pct"] = (
    delta["POLAR_Critical_Coverage_pct"]
    -
    delta[
        "BASELINE_Critical_Coverage_pct"
    ]
)


print("\n" + "=" * 140)
print("OPERATIONAL DELTAS")
print("=" * 140)

display(
    delta[
        [
            "Scenario",
            "POLAR_Status",
            "Fuel_Difference_L",
            "Battery_Discharge_Difference_kWh",
            "Final_SOC_Difference_pct",
            "Critical_Coverage_Difference_pct"
        ]
    ]
)


# ============================================================
# SAVE
# ============================================================

baseline_comparison_df.to_csv(
    "/content/polar_ems_baseline_comparison.csv",
    index=False
)

delta.to_csv(
    "/content/polar_ems_baseline_comparison_delta.csv",
    index=False
)


print("\n✅ Baseline comparison saved:")
print("/content/polar_ems_baseline_comparison.csv")
print("/content/polar_ems_baseline_comparison_delta.csv")


print("\n" + "=" * 140)
print("✅ STEP 21 COMPLETE")
print("=" * 140)

POLAR-EMS vs RULE-BASED BASELINE


,Scenario,POLAR_Status,POLAR_Fuel_Used_L,POLAR_Generator_Energy_kWh,POLAR_Battery_Discharge_kWh,POLAR_Renewable_Used_kWh,POLAR_Min_SOC_pct,POLAR_Final_SOC_pct,POLAR_Critical_Coverage_pct,POLAR_Balance_Error_kW,BASELINE_Fuel_Used_L,BASELINE_Generator_Energy_kWh,BASELINE_Battery_Discharge_kWh,BASELINE_Renewable_Used_kWh,BASELINE_Min_SOC_pct,BASELINE_Final_SOC_pct,BASELINE_Critical_Coverage_pct,BASELINE_Balance_Error_kW,BASELINE_Unserved_Load_kWh
0,NORMAL,OPTIMAL,2498.842542,7496.527625,4647.767675,37034.002465,20.0,58.350002,100.0,5.684342e-14,16377.460000,49132.380000,458.252421,38002.3900,20.0,20.0,100.0,158.1700,0.0
1,LOW_RENEWABLE,OPTIMAL,5515.547155,16546.641464,1731.039536,22598.809000,20.0,77.087500,100.0,2.842171e-14,20901.151193,62703.453579,458.252421,22598.8090,20.0,20.0,100.0,69.6700,0.0
2,RESUPPLY_DELAY_4D,OPTIMAL,2498.842542,7496.527625,4446.548314,37034.002465,20.0,76.754167,100.0,5.684342e-14,16377.460000,49132.380000,458.252421,38002.3900,20.0,20.0,100.0,158.1700,0.0
3,BATTERY_DEGRADATION,OPTIMAL,2626.398408,7879.195225,4178.726550,36307.130886,20.0,55.000000,100.0,5.684342e-14,16369.738386,49109.215158,503.164842,38002.3900,20.0,20.0,100.0,158.1700,0.0
4,STORM,INFEASIBLE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22091.881526,66275.644579,458.252421,18806.5255,20.0,20.0,100.0,20.1725,0.0



OPERATIONAL DELTAS


,Scenario,POLAR_Status,Fuel_Difference_L,Battery_Discharge_Difference_kWh,Final_SOC_Difference_pct,Critical_Coverage_Difference_pct
0,NORMAL,OPTIMAL,13878.617458,-4189.515254,38.350002,0.0
1,LOW_RENEWABLE,OPTIMAL,15385.604038,-1272.787115,57.087500,0.0
2,RESUPPLY_DELAY_4D,OPTIMAL,13878.617458,-3988.295893,56.754167,0.0
3,BATTERY_DEGRADATION,OPTIMAL,13743.339978,-3675.561708,35.000000,0.0
4,STORM,INFEASIBLE,NaN,NaN,NaN,NaN



✅ Baseline comparison saved:
/content/polar_ems_baseline_comparison.csv
/content/polar_ems_baseline_comparison_delta.csv

✅ STEP 21 COMPLETE


In [30]:
# ============================================================
# STEP 22 — FAIR BASELINE COMPARISON
# INCLUDE FLEXIBLE-LOAD SERVICE
# ============================================================

comparison_rows = []

comparison_scenarios = [
    "NORMAL",
    "LOW_RENEWABLE",
    "RESUPPLY_DELAY_4D",
    "BATTERY_DEGRADATION",
    "STORM"
]

for scenario_name in comparison_scenarios:

    kwargs = {}

    if scenario_name == "BATTERY_DEGRADATION":
        kwargs["battery_soh_override"] = 75.0

    result = run_polar_ems_scenario_v2(
        scenario=scenario_name,
        **kwargs
    )

    state = result["state"]
    profile = result["scenario_profile"]

    # --------------------------------------------------------
    # BASELINE
    # --------------------------------------------------------

    baseline = run_rule_based_baseline(
        scenario_profile=profile,
        state=state,
        horizon_hours=168
    )

    baseline_plan = baseline["plan"]

    baseline_flexible_available = (
        baseline_plan["flexible_load_kw"].sum()
    )

    baseline_flexible_served = (
        baseline_plan["load_kw"].sum()
        -
        baseline_plan["critical_load_kw"].sum()
        -
        baseline_plan["unserved_load_kw"].sum()
    )

    baseline_flexible_coverage = (
        100.0 *
        baseline_flexible_served /
        max(baseline_flexible_available, 1e-9)
    )

    # --------------------------------------------------------
    # POLAR-EMS
    # --------------------------------------------------------

    opt = result["optimizer"]

    if opt["status"] == "OPTIMAL":

        polar_plan = opt["plan"]

        polar_flexible_available = (
            polar_plan["flexible_load_kw"].sum()
        )

        polar_flexible_served = (
            polar_plan[
                "flexible_load_served_kw"
            ].sum()
        )

        polar_flexible_coverage = (
            100.0 *
            polar_flexible_served /
            max(
                polar_flexible_available,
                1e-9
            )
        )

        polar_total_load_served = (
            polar_plan[
                "critical_load_kw"
            ].sum()
            +
            polar_flexible_served
        )

        polar_total_load_available = (
            polar_plan[
                "critical_load_kw"
            ].sum()
            +
            polar_flexible_available
        )

        polar_total_load_coverage = (
            100.0 *
            polar_total_load_served /
            max(
                polar_total_load_available,
                1e-9
            )
        )

        polar_generator = (
            opt["generator_energy_kwh"]
        )

        polar_fuel = opt["fuel_used_l"]

        polar_battery = (
            opt["battery_discharge_kwh"]
        )

        polar_renewable = (
            opt["renewable_used_kwh"]
        )

    else:

        polar_flexible_available = np.nan
        polar_flexible_served = np.nan
        polar_flexible_coverage = np.nan
        polar_total_load_coverage = np.nan

        polar_generator = np.nan
        polar_fuel = np.nan
        polar_battery = np.nan
        polar_renewable = np.nan

    # --------------------------------------------------------
    # BASELINE TOTAL LOAD COVERAGE
    # --------------------------------------------------------

    baseline_total_available = (
        baseline_plan["load_kw"].sum()
    )

    baseline_total_unserved = (
        baseline_plan["unserved_load_kw"].sum()
    )

    baseline_total_coverage = (
        100.0 *
        (
            baseline_total_available
            -
            baseline_total_unserved
        )
        /
        max(
            baseline_total_available,
            1e-9
        )
    )

    # --------------------------------------------------------
    # STORE
    # --------------------------------------------------------

    comparison_rows.append({

        "Scenario":
            scenario_name,

        "POLAR_Status":
            opt["status"],

        "POLAR_Fuel_L":
            polar_fuel,

        "BASELINE_Fuel_L":
            baseline["fuel_used_l"],

        "POLAR_Battery_Discharge_kWh":
            polar_battery,

        "BASELINE_Battery_Discharge_kWh":
            baseline[
                "battery_discharge_kwh"
            ],

        "POLAR_Renewable_Used_kWh":
            polar_renewable,

        "BASELINE_Renewable_Used_kWh":
            baseline[
                "renewable_used_kwh"
            ],

        "POLAR_Flexible_Coverage_pct":
            polar_flexible_coverage,

        "BASELINE_Flexible_Coverage_pct":
            baseline_flexible_coverage,

        "POLAR_Total_Load_Coverage_pct":
            polar_total_load_coverage,

        "BASELINE_Total_Load_Coverage_pct":
            baseline_total_coverage,

        "POLAR_Min_SOC_pct":
            (
                opt["minimum_soc_pct"]
                if opt["status"] == "OPTIMAL"
                else np.nan
            ),

        "BASELINE_Min_SOC_pct":
            baseline["minimum_soc_pct"],

        "POLAR_Final_SOC_pct":
            (
                opt["final_battery_soc_pct"]
                if opt["status"] == "OPTIMAL"
                else np.nan
            ),

        "BASELINE_Final_SOC_pct":
            baseline["final_soc_pct"],

        "POLAR_Safety":
            result["safety"]["status"],

        "BASELINE_Unserved_Load_kWh":
            baseline["unserved_load_kwh"]
    })


fair_comparison_df = pd.DataFrame(
    comparison_rows
)

print("=" * 130)
print("POLAR-EMS vs BASELINE — FAIR OPERATIONAL COMPARISON")
print("=" * 130)

display(fair_comparison_df)


# ============================================================
# NORMAL-CASE INTERPRETATION METRICS
# ============================================================

normal = fair_comparison_df[
    fair_comparison_df["Scenario"] == "NORMAL"
].iloc[0]

if normal["POLAR_Status"] == "OPTIMAL":

    polar_flexible = (
        normal[
            "POLAR_Flexible_Coverage_pct"
        ]
    )

    baseline_flexible = (
        normal[
            "BASELINE_Flexible_Coverage_pct"
        ]
    )

    fuel_reduction_pct = (
        100.0 *
        (
            normal["BASELINE_Fuel_L"]
            -
            normal["POLAR_Fuel_L"]
        )
        /
        max(
            normal["BASELINE_Fuel_L"],
            1e-9
        )
    )

    print("\n" + "=" * 130)
    print("NORMAL SCENARIO — KEY METRICS")
    print("=" * 130)

    print(
        f"POLAR-EMS flexible-load coverage : "
        f"{polar_flexible:.2f}%"
    )

    print(
        f"Baseline flexible-load coverage  : "
        f"{baseline_flexible:.2f}%"
    )

    print(
        f"Fuel difference                   : "
        f"{fuel_reduction_pct:.2f}%"
    )

    print(
        "\n⚠️ Fuel reduction must be interpreted "
        "together with flexible-load service."
    )


# ============================================================
# SAVE
# ============================================================

fair_comparison_df.to_csv(
    "/content/polar_ems_fair_baseline_comparison.csv",
    index=False
)

print(
    "\n✅ Saved:"
    "\n/content/polar_ems_fair_baseline_comparison.csv"
)

print("\n" + "=" * 130)
print("✅ STEP 22 COMPLETE")
print("=" * 130)

KeyError: 'flexible_load_kw'

In [31]:
# ============================================================
# STEP 22 — FAIR BASELINE COMPARISON (FIXED)
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# FIXED RULE-BASED BASELINE
# ============================================================

def run_rule_based_baseline(
    scenario_profile,
    state,
    horizon_hours=168,
    fixed_reserve_soc_pct=20.0,
    fuel_to_energy_kwh_per_l=3.0
):
    """
    Simple rule-based baseline.

    Rules:
      1. Renewable energy first
      2. Critical load prioritized
      3. Battery may discharge down to fixed 20% SOC
      4. Generator supplies remaining demand
      5. No CQRM
      6. No dynamic resupply reserve
    """

    df = (
        scenario_profile.copy()
        .sort_values("timestamp")
        .reset_index(drop=True)
        .head(horizon_hours)
    )

    capacity = float(
        state.battery_usable_capacity_kwh
    )

    battery_energy = min(
        float(state.battery_energy_kwh),
        capacity
    )

    minimum_energy = (
        capacity *
        fixed_reserve_soc_pct /
        100.0
    )

    fuel_remaining = float(
        state.fuel_remaining_l
    )

    records = []

    for _, row in df.iterrows():

        load = max(
            0.0,
            float(row["load_forecast_kw"])
        )

        critical = max(
            0.0,
            float(row["critical_load_kw"])
        )

        flexible = max(
            0.0,
            float(row["flexible_load_kw"])
        )

        solar = max(
            0.0,
            float(row["solar_forecast_kw"])
        )

        wind = max(
            0.0,
            float(row["wind_forecast_kw"])
        )

        renewable = solar + wind

        generator_available = max(
            0.0,
            float(row["generator_available_kw"])
        )

        # ----------------------------------------------------
        # Renewable first
        # ----------------------------------------------------

        renewable_used = min(
            renewable,
            load
        )

        remaining_load = max(
            0.0,
            load - renewable_used
        )

        # ----------------------------------------------------
        # Battery
        # ----------------------------------------------------

        battery_available = max(
            0.0,
            battery_energy - minimum_energy
        )

        battery_discharge = min(
            remaining_load,
            battery_available,
            MAX_BATTERY_DISCHARGE_KW
        )

        battery_energy -= (
            battery_discharge /
            ETA_DISCHARGE
        )

        battery_energy = max(
            minimum_energy,
            battery_energy
        )

        remaining_load -= battery_discharge

        # ----------------------------------------------------
        # Generator
        # ----------------------------------------------------

        generator = min(
            remaining_load,
            generator_available
        )

        # ----------------------------------------------------
        # Flexible load handling
        # ----------------------------------------------------
        #
        # Baseline attempts to serve all load.
        # Any remaining demand after generator is unserved.
        # ----------------------------------------------------

        unserved_load = max(
            0.0,
            remaining_load - generator
        )

        total_served = max(
            0.0,
            load - unserved_load
        )

        # Critical load coverage
        #
        # We calculate whether enough of the available supply
        # can cover the critical portion.
        # ----------------------------------------------------

        critical_unserved = max(
            0.0,
            critical - total_served
        )

        critical_coverage = (
            100.0
            if critical <= 1e-9
            else
            100.0 *
            (
                critical -
                critical_unserved
            )
            /
            critical
        )

        flexible_served = max(
            0.0,
            min(
                flexible,
                total_served - critical
            )
        )

        flexible_coverage = (
            100.0
            if flexible <= 1e-9
            else
            100.0 *
            flexible_served /
            flexible
        )

        # ----------------------------------------------------
        # Fuel
        # ----------------------------------------------------

        fuel_used = (
            generator /
            max(
                fuel_to_energy_kwh_per_l,
                1e-9
            )
        )

        fuel_remaining -= fuel_used

        fuel_remaining = max(
            0.0,
            fuel_remaining
        )

        # ----------------------------------------------------
        # State metrics
        # ----------------------------------------------------

        soc_pct = (
            100.0 *
            battery_energy /
            max(
                capacity,
                1e-9
            )
        )

        balance_error = (
            renewable_used
            + battery_discharge
            + generator
            - total_served
        )

        records.append({

            "timestamp":
                row["timestamp"],

            "load_kw":
                load,

            "critical_load_kw":
                critical,

            "flexible_load_kw":
                flexible,

            "flexible_load_served_kw":
                flexible_served,

            "renewable_available_kw":
                renewable,

            "renewable_used_kw":
                renewable_used,

            "battery_discharge_kw":
                battery_discharge,

            "battery_energy_kwh":
                battery_energy,

            "battery_soc_pct":
                soc_pct,

            "generator_kw":
                generator,

            "fuel_used_l":
                fuel_used,

            "fuel_remaining_l":
                fuel_remaining,

            "critical_load_coverage_pct":
                critical_coverage,

            "flexible_load_coverage_pct":
                flexible_coverage,

            "power_balance_error_kw":
                balance_error,

            "unserved_load_kw":
                unserved_load
        })

    plan = pd.DataFrame(records)

    return {

        "status":
            "BASELINE",

        "plan":
            plan,

        "fuel_used_l":
            float(
                plan["fuel_used_l"].sum()
            ),

        "generator_energy_kwh":
            float(
                plan["generator_kw"].sum()
            ),

        "battery_discharge_kwh":
            float(
                plan[
                    "battery_discharge_kw"
                ].sum()
            ),

        "renewable_used_kwh":
            float(
                plan[
                    "renewable_used_kw"
                ].sum()
            ),

        "minimum_soc_pct":
            float(
                plan[
                    "battery_soc_pct"
                ].min()
            ),

        "final_soc_pct":
            float(
                plan[
                    "battery_soc_pct"
                ].iloc[-1]
            ),

        "minimum_critical_coverage_pct":
            float(
                plan[
                    "critical_load_coverage_pct"
                ].min()
            ),

        "minimum_flexible_coverage_pct":
            float(
                plan[
                    "flexible_load_coverage_pct"
                ].min()
            ),

        "max_power_balance_error_kw":
            float(
                plan[
                    "power_balance_error_kw"
                ].abs()
                .max()
            ),

        "unserved_load_kwh":
            float(
                plan[
                    "unserved_load_kw"
                ].sum()
            )
    }


# ============================================================
# RUN FAIR COMPARISON
# ============================================================

comparison_rows = []

comparison_scenarios = [
    "NORMAL",
    "LOW_RENEWABLE",
    "RESUPPLY_DELAY_4D",
    "BATTERY_DEGRADATION",
    "STORM"
]


for scenario_name in comparison_scenarios:

    kwargs = {}

    if scenario_name == "BATTERY_DEGRADATION":
        kwargs["battery_soh_override"] = 75.0

    result = run_polar_ems_scenario_v2(
        scenario=scenario_name,
        **kwargs
    )

    state = result["state"]
    profile = result["scenario_profile"]

    # --------------------------------------------------------
    # BASELINE
    # --------------------------------------------------------

    baseline = run_rule_based_baseline(
        scenario_profile=profile,
        state=state,
        horizon_hours=168
    )

    baseline_plan = baseline["plan"]

    # --------------------------------------------------------
    # POLAR-EMS
    # --------------------------------------------------------

    opt = result["optimizer"]

    if opt["status"] == "OPTIMAL":

        polar_plan = opt["plan"]

        polar_flexible_available = (
            polar_plan[
                "flexible_load_kw"
            ].sum()
        )

        polar_flexible_served = (
            polar_plan[
                "flexible_load_served_kw"
            ].sum()
        )

        polar_flexible_coverage = (
            100.0 *
            polar_flexible_served /
            max(
                polar_flexible_available,
                1e-9
            )
        )

        polar_total_load_available = (
            polar_plan[
                "load_kw"
            ].sum()
        )

        polar_total_load_served = (
            polar_plan[
                "critical_load_kw"
            ].sum()
            +
            polar_flexible_served
        )

        polar_total_load_coverage = (
            100.0 *
            polar_total_load_served /
            max(
                polar_total_load_available,
                1e-9
            )
        )

        polar_fuel = opt[
            "fuel_used_l"
        ]

        polar_generator = opt[
            "generator_energy_kwh"
        ]

        polar_battery = opt[
            "battery_discharge_kwh"
        ]

        polar_renewable = opt[
            "renewable_used_kwh"
        ]

        polar_min_soc = opt[
            "minimum_soc_pct"
        ]

        polar_final_soc = opt[
            "final_battery_soc_pct"
        ]

        polar_critical = opt[
            "critical_load_coverage_pct"
        ]

        polar_balance = opt[
            "max_power_balance_error_kw"
        ]

        polar_status = "OPTIMAL"

    else:

        polar_fuel = np.nan
        polar_generator = np.nan
        polar_battery = np.nan
        polar_renewable = np.nan
        polar_min_soc = np.nan
        polar_final_soc = np.nan
        polar_critical = np.nan
        polar_flexible_coverage = np.nan
        polar_total_load_coverage = np.nan
        polar_balance = np.nan
        polar_status = opt["status"]

    # --------------------------------------------------------
    # BASELINE METRICS
    # --------------------------------------------------------

    baseline_flexible_coverage = float(
        baseline[
            "minimum_flexible_coverage_pct"
        ]
    )

    baseline_total_available = float(
        baseline_plan["load_kw"].sum()
    )

    baseline_total_unserved = float(
        baseline_plan["unserved_load_kw"].sum()
    )

    baseline_total_coverage = (
        100.0 *
        (
            baseline_total_available
            -
            baseline_total_unserved
        )
        /
        max(
            baseline_total_available,
            1e-9
        )
    )

    # --------------------------------------------------------
    # STORE
    # --------------------------------------------------------

    comparison_rows.append({

        "Scenario":
            scenario_name,

        "POLAR_Status":
            polar_status,

        "POLAR_Fuel_L":
            polar_fuel,

        "BASELINE_Fuel_L":
            baseline[
                "fuel_used_l"
            ],

        "POLAR_Generator_Energy_kWh":
            polar_generator,

        "BASELINE_Generator_Energy_kWh":
            baseline[
                "generator_energy_kwh"
            ],

        "POLAR_Battery_Discharge_kWh":
            polar_battery,

        "BASELINE_Battery_Discharge_kWh":
            baseline[
                "battery_discharge_kwh"
            ],

        "POLAR_Renewable_Used_kWh":
            polar_renewable,

        "BASELINE_Renewable_Used_kWh":
            baseline[
                "renewable_used_kwh"
            ],

        "POLAR_Flexible_Coverage_pct":
            polar_flexible_coverage,

        "BASELINE_Flexible_Coverage_pct":
            baseline_flexible_coverage,

        "POLAR_Total_Load_Coverage_pct":
            polar_total_load_coverage,

        "BASELINE_Total_Load_Coverage_pct":
            baseline_total_coverage,

        "POLAR_Min_SOC_pct":
            polar_min_soc,

        "BASELINE_Min_SOC_pct":
            baseline[
                "minimum_soc_pct"
            ],

        "POLAR_Final_SOC_pct":
            polar_final_soc,

        "BASELINE_Final_SOC_pct":
            baseline[
                "final_soc_pct"
            ],

        "POLAR_Critical_Coverage_pct":
            polar_critical,

        "BASELINE_Critical_Coverage_pct":
            baseline[
                "minimum_critical_coverage_pct"
            ],

        "POLAR_Balance_Error_kW":
            polar_balance,

        "BASELINE_Balance_Error_kW":
            baseline[
                "max_power_balance_error_kw"
            ],

        "BASELINE_Unserved_Load_kWh":
            baseline[
                "unserved_load_kwh"
            ],

        "POLAR_Safety":
            result[
                "safety"
            ]["status"]
    })


fair_comparison_df = pd.DataFrame(
    comparison_rows
)


print("=" * 140)
print("POLAR-EMS vs RULE-BASED BASELINE — FAIR COMPARISON")
print("=" * 140)

display(
    fair_comparison_df
)


# ============================================================
# DELTA TABLE
# ============================================================

delta_df = fair_comparison_df.copy()

delta_df["Fuel_Difference_L"] = (
    delta_df[
        "BASELINE_Fuel_L"
    ]
    -
    delta_df[
        "POLAR_Fuel_L"
    ]
)

delta_df["Battery_Discharge_Difference_kWh"] = (
    delta_df[
        "BASELINE_Battery_Discharge_kWh"
    ]
    -
    delta_df[
        "POLAR_Battery_Discharge_kWh"
    ]
)

delta_df["Final_SOC_Difference_pct"] = (
    delta_df[
        "POLAR_Final_SOC_pct"
    ]
    -
    delta_df[
        "BASELINE_Final_SOC_pct"
    ]
)

delta_df["Flexible_Coverage_Difference_pct"] = (
    delta_df[
        "POLAR_Flexible_Coverage_pct"
    ]
    -
    delta_df[
        "BASELINE_Flexible_Coverage_pct"
    ]
)

delta_df["Critical_Coverage_Difference_pct"] = (
    delta_df[
        "POLAR_Critical_Coverage_pct"
    ]
    -
    delta_df[
        "BASELINE_Critical_Coverage_pct"
    ]
)


print("\n" + "=" * 140)
print("OPERATIONAL DELTAS")
print("=" * 140)

display(
    delta_df[
        [
            "Scenario",
            "POLAR_Status",
            "Fuel_Difference_L",
            "Battery_Discharge_Difference_kWh",
            "Final_SOC_Difference_pct",
            "Flexible_Coverage_Difference_pct",
            "Critical_Coverage_Difference_pct",
            "POLAR_Safety"
        ]
    ]
)


# ============================================================
# SAVE
# ============================================================

fair_comparison_df.to_csv(
    "/content/polar_ems_fair_baseline_comparison.csv",
    index=False
)

delta_df.to_csv(
    "/content/polar_ems_fair_baseline_comparison_delta.csv",
    index=False
)

print("\n✅ Saved:")
print("/content/polar_ems_fair_baseline_comparison.csv")
print("/content/polar_ems_fair_baseline_comparison_delta.csv")

print("\n" + "=" * 140)
print("✅ STEP 22 COMPLETE")
print("=" * 140)

POLAR-EMS vs RULE-BASED BASELINE — FAIR COMPARISON


,Scenario,POLAR_Status,POLAR_Fuel_L,BASELINE_Fuel_L,POLAR_Generator_Energy_kWh,BASELINE_Generator_Energy_kWh,POLAR_Battery_Discharge_kWh,BASELINE_Battery_Discharge_kWh,POLAR_Renewable_Used_kWh,BASELINE_Renewable_Used_kWh,...,POLAR_Min_SOC_pct,BASELINE_Min_SOC_pct,POLAR_Final_SOC_pct,BASELINE_Final_SOC_pct,POLAR_Critical_Coverage_pct,BASELINE_Critical_Coverage_pct,POLAR_Balance_Error_kW,BASELINE_Balance_Error_kW,BASELINE_Unserved_Load_kWh,POLAR_Safety
0,NORMAL,OPTIMAL,2498.842542,15686.535860,7496.527625,47059.607579,4647.767675,458.252421,37034.002465,38002.3900,...,20.0,20.0,58.350002,20.0,100.0,100.0,5.684342e-14,5.684342e-14,0.0,SAFE
1,LOW_RENEWABLE,OPTIMAL,5515.547155,20821.062860,16546.641464,62463.188579,1731.039536,458.252421,22598.809000,22598.8090,...,20.0,20.0,77.087500,20.0,100.0,100.0,2.842171e-14,5.684342e-14,0.0,UNSAFE
2,RESUPPLY_DELAY_4D,OPTIMAL,2498.842542,15686.535860,7496.527625,47059.607579,4446.548314,458.252421,37034.002465,38002.3900,...,20.0,20.0,76.754167,20.0,100.0,100.0,5.684342e-14,5.684342e-14,0.0,UNSAFE
3,BATTERY_DEGRADATION,OPTIMAL,2626.398408,15671.565053,7879.195225,47014.695158,4178.726550,503.164842,36307.130886,38002.3900,...,20.0,20.0,55.000000,20.0,100.0,100.0,5.684342e-14,5.684342e-14,0.0,SAFE
4,STORM,INFEASIBLE,NaN,22085.157360,NaN,66255.472079,NaN,458.252421,NaN,18806.5255,...,NaN,20.0,NaN,20.0,NaN,100.0,NaN,1.136868e-13,0.0,UNSAFE



OPERATIONAL DELTAS


,Scenario,POLAR_Status,Fuel_Difference_L,Battery_Discharge_Difference_kWh,Final_SOC_Difference_pct,Flexible_Coverage_Difference_pct,Critical_Coverage_Difference_pct,POLAR_Safety
0,NORMAL,OPTIMAL,13187.693318,-4189.515254,38.350002,-62.754377,0.0,SAFE
1,LOW_RENEWABLE,OPTIMAL,15305.515705,-1272.787115,57.087500,-100.000000,0.0,UNSAFE
2,RESUPPLY_DELAY_4D,OPTIMAL,13187.693318,-3988.295893,56.754167,-64.169250,0.0,UNSAFE
3,BATTERY_DEGRADATION,OPTIMAL,13045.166644,-3675.561708,35.000000,-63.361712,0.0,SAFE
4,STORM,INFEASIBLE,NaN,NaN,NaN,NaN,NaN,UNSAFE



✅ Saved:
/content/polar_ems_fair_baseline_comparison.csv
/content/polar_ems_fair_baseline_comparison_delta.csv

✅ STEP 22 COMPLETE


In [32]:
# ============================================================
# STEP 23 — FAIR CONSTRAINED BASELINE
# ============================================================

import numpy as np
import pandas as pd


def run_fair_baseline(
    scenario_profile,
    state,
    horizon_hours=168,
    fixed_reserve_soc_pct=20.0,
    fuel_to_energy_kwh_per_l=3.0
):
    """
    Fair comparison baseline.

    Policy:
      - Renewable first
      - Critical load must be served
      - Battery can discharge only to fixed 20% SOC
      - Generator serves remaining demand
      - Flexible load can be curtailed when necessary
      - No CQRM
      - No dynamic resupply reserve

    This is deliberately simpler than POLAR-EMS.
    """

    df = (
        scenario_profile.copy()
        .sort_values("timestamp")
        .reset_index(drop=True)
        .head(horizon_hours)
    )

    capacity = float(
        state.battery_usable_capacity_kwh
    )

    battery_energy = min(
        float(state.battery_energy_kwh),
        capacity
    )

    minimum_energy = (
        capacity *
        fixed_reserve_soc_pct /
        100.0
    )

    fuel_remaining = float(
        state.fuel_remaining_l
    )

    records = []

    for _, row in df.iterrows():

        load = max(
            0.0,
            float(row["load_forecast_kw"])
        )

        critical = max(
            0.0,
            float(row["critical_load_kw"])
        )

        flexible = max(
            0.0,
            float(row["flexible_load_kw"])
        )

        renewable = max(
            0.0,
            float(row["renewable_forecast_kw"])
        )

        generator_available = max(
            0.0,
            float(row["generator_available_kw"])
        )

        # ----------------------------------------------------
        # Renewable first
        # ----------------------------------------------------

        renewable_used = min(
            renewable,
            load
        )

        remaining = max(
            0.0,
            load - renewable_used
        )

        # ----------------------------------------------------
        # Battery
        # ----------------------------------------------------

        battery_available = max(
            0.0,
            battery_energy - minimum_energy
        )

        battery_discharge = min(
            remaining,
            battery_available,
            MAX_BATTERY_DISCHARGE_KW
        )

        battery_energy -= (
            battery_discharge /
            ETA_DISCHARGE
        )

        battery_energy = max(
            minimum_energy,
            battery_energy
        )

        remaining -= battery_discharge

        # ----------------------------------------------------
        # Generator
        # ----------------------------------------------------

        generator = min(
            remaining,
            generator_available
        )

        remaining -= generator

        # ----------------------------------------------------
        # Whatever remains is flexible-load curtailment,
        # because critical load is protected first.
        # ----------------------------------------------------

        unserved = max(
            0.0,
            remaining
        )

        # Critical load check
        critical_supply_available = (
            renewable_used
            + battery_discharge
            + generator
        )

        critical_unserved = max(
            0.0,
            critical -
            critical_supply_available
        )

        critical_served = (
            critical -
            critical_unserved
        )

        critical_coverage = (
            100.0
            if critical <= 1e-9
            else
            100.0 *
            critical_served /
            critical
        )

        # Flexible load served only after critical load
        total_supply = (
            renewable_used
            + battery_discharge
            + generator
        )

        flexible_served = max(
            0.0,
            min(
                flexible,
                total_supply -
                critical
            )
        )

        flexible_coverage = (
            100.0
            if flexible <= 1e-9
            else
            100.0 *
            flexible_served /
            flexible
        )

        fuel_used = (
            generator /
            max(
                fuel_to_energy_kwh_per_l,
                1e-9
            )
        )

        fuel_remaining -= fuel_used

        soc_pct = (
            100.0 *
            battery_energy /
            max(capacity, 1e-9)
        )

        # Supply balance relative to actual served load
        total_served = (
            critical_served +
            flexible_served
        )

        balance_error = (
            renewable_used
            + battery_discharge
            + generator
            - total_served
        )

        records.append({

            "timestamp":
                row["timestamp"],

            "load_kw":
                load,

            "critical_load_kw":
                critical,

            "flexible_load_kw":
                flexible,

            "critical_load_served_kw":
                critical_served,

            "flexible_load_served_kw":
                flexible_served,

            "critical_load_coverage_pct":
                critical_coverage,

            "flexible_load_coverage_pct":
                flexible_coverage,

            "renewable_used_kw":
                renewable_used,

            "battery_discharge_kw":
                battery_discharge,

            "battery_soc_pct":
                soc_pct,

            "generator_kw":
                generator,

            "fuel_used_l":
                fuel_used,

            "fuel_remaining_l":
                fuel_remaining,

            "unserved_load_kw":
                unserved,

            "power_balance_error_kw":
                balance_error
        })

    plan = pd.DataFrame(records)

    return {

        "status": "BASELINE",

        "plan": plan,

        "fuel_used_l":
            float(
                plan["fuel_used_l"].sum()
            ),

        "generator_energy_kwh":
            float(
                plan["generator_kw"].sum()
            ),

        "battery_discharge_kwh":
            float(
                plan[
                    "battery_discharge_kw"
                ].sum()
            ),

        "renewable_used_kwh":
            float(
                plan[
                    "renewable_used_kw"
                ].sum()
            ),

        "minimum_soc_pct":
            float(
                plan[
                    "battery_soc_pct"
                ].min()
            ),

        "final_soc_pct":
            float(
                plan[
                    "battery_soc_pct"
                ].iloc[-1]
            ),

        "critical_coverage_pct":
            float(
                plan[
                    "critical_load_coverage_pct"
                ].min()
            ),

        "flexible_coverage_pct":
            float(
                plan[
                    "flexible_load_coverage_pct"
                ].mean()
            ),

        "max_power_balance_error_kw":
            float(
                plan[
                    "power_balance_error_kw"
                ].abs()
                .max()
            ),

        "unserved_load_kwh":
            float(
                plan["unserved_load_kw"].sum()
            )
    }


# ============================================================
# RUN FAIR COMPARISON
# ============================================================

fair_rows = []

fair_scenarios = [
    "NORMAL",
    "LOW_RENEWABLE",
    "RESUPPLY_DELAY_4D",
    "BATTERY_DEGRADATION"
]


for scenario_name in fair_scenarios:

    kwargs = {}

    if scenario_name == "BATTERY_DEGRADATION":
        kwargs["battery_soh_override"] = 75.0

    result = run_polar_ems_scenario_v2(
        scenario=scenario_name,
        **kwargs
    )

    baseline = run_fair_baseline(
        scenario_profile=
            result["scenario_profile"],

        state=result["state"],

        horizon_hours=168,

        fixed_reserve_soc_pct=20.0
    )

    opt = result["optimizer"]

    if opt["status"] == "OPTIMAL":

        polar_plan = opt["plan"]

        polar_flexible_coverage = (
            100.0 *
            polar_plan[
                "flexible_load_served_kw"
            ].sum()
            /
            max(
                polar_plan[
                    "flexible_load_kw"
                ].sum(),
                1e-9
            )
        )

        fair_rows.append({

            "Scenario":
                scenario_name,

            "POLAR_Fuel_L":
                opt["fuel_used_l"],

            "Baseline_Fuel_L":
                baseline["fuel_used_l"],

            "POLAR_Battery_Discharge_kWh":
                opt["battery_discharge_kwh"],

            "Baseline_Battery_Discharge_kWh":
                baseline[
                    "battery_discharge_kwh"
                ],

            "POLAR_Final_SOC_pct":
                opt["final_battery_soc_pct"],

            "Baseline_Final_SOC_pct":
                baseline["final_soc_pct"],

            "POLAR_Min_SOC_pct":
                opt["minimum_soc_pct"],

            "Baseline_Min_SOC_pct":
                baseline["minimum_soc_pct"],

            "POLAR_Critical_Coverage_pct":
                opt["critical_load_coverage_pct"],

            "Baseline_Critical_Coverage_pct":
                baseline["critical_coverage_pct"],

            "POLAR_Flexible_Coverage_pct":
                polar_flexible_coverage,

            "Baseline_Flexible_Coverage_pct":
                baseline["flexible_coverage_pct"],

            "POLAR_Renewable_Used_kWh":
                opt["renewable_used_kwh"],

            "Baseline_Renewable_Used_kWh":
                baseline["renewable_used_kwh"],

            "POLAR_Safety":
                result["safety"]["status"]
        })


fair_df = pd.DataFrame(
    fair_rows
)


# ============================================================
# DELTA / PERCENTAGE METRICS
# ============================================================

fair_df["Fuel_Saving_%"] = (
    100.0 *
    (
        fair_df["Baseline_Fuel_L"]
        -
        fair_df["POLAR_Fuel_L"]
    )
    /
    fair_df["Baseline_Fuel_L"]
)

fair_df["Battery_Discharge_Reduction_%"] = (
    100.0 *
    (
        fair_df[
            "Baseline_Battery_Discharge_kWh"
        ]
        -
        fair_df[
            "POLAR_Battery_Discharge_kWh"
        ]
    )
    /
    fair_df[
        "Baseline_Battery_Discharge_kWh"
    ]
)

fair_df["Final_SOC_Gain_pct_points"] = (
    fair_df[
        "POLAR_Final_SOC_pct"
    ]
    -
    fair_df[
        "Baseline_Final_SOC_pct"
    ]
)


print("=" * 140)
print("POLAR-EMS — FAIR CONSTRAINED BASELINE")
print("=" * 140)

display(fair_df)


# ============================================================
# SAVE
# ============================================================

fair_df.to_csv(
    "/content/polar_ems_fair_constrained_baseline.csv",
    index=False
)

print(
    "\n✅ Saved:"
    "\n/content/polar_ems_fair_constrained_baseline.csv"
)

print("\n" + "=" * 140)
print("✅ STEP 23 COMPLETE")
print("=" * 140)

POLAR-EMS — FAIR CONSTRAINED BASELINE


,Scenario,POLAR_Fuel_L,Baseline_Fuel_L,POLAR_Battery_Discharge_kWh,Baseline_Battery_Discharge_kWh,POLAR_Final_SOC_pct,Baseline_Final_SOC_pct,POLAR_Min_SOC_pct,Baseline_Min_SOC_pct,POLAR_Critical_Coverage_pct,Baseline_Critical_Coverage_pct,POLAR_Flexible_Coverage_pct,Baseline_Flexible_Coverage_pct,POLAR_Renewable_Used_kWh,Baseline_Renewable_Used_kWh,POLAR_Safety,Fuel_Saving_%,Battery_Discharge_Reduction_%,Final_SOC_Gain_pct_points
0,NORMAL,2498.842542,15686.535860,4647.767675,458.252421,58.350002,20.0,20.0,20.0,100.0,100.0,37.245623,100.0,37034.002465,38002.390,SAFE,84.070144,-914.237451,38.350002
1,LOW_RENEWABLE,5515.547155,20821.062860,1731.039536,458.252421,77.087500,20.0,20.0,20.0,100.0,100.0,0.000000,100.0,22598.809000,22598.809,UNSAFE,73.509771,-277.748039,57.087500
2,RESUPPLY_DELAY_4D,2498.842542,15686.535860,4446.548314,458.252421,76.754167,20.0,20.0,20.0,100.0,100.0,35.830750,100.0,37034.002465,38002.390,UNSAFE,84.070144,-870.327293,56.754167
3,BATTERY_DEGRADATION,2626.398408,15671.565053,4178.726550,503.164842,55.000000,20.0,20.0,20.0,100.0,100.0,36.638288,100.0,36307.130886,38002.390,SAFE,83.240995,-730.488580,35.000000



✅ Saved:
/content/polar_ems_fair_constrained_baseline.csv

✅ STEP 23 COMPLETE


In [33]:
# ============================================================
# STEP 24 — MATCHED ENERGY-SECURITY SCORECARD
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# SCENARIOS
# ------------------------------------------------------------

scorecard_scenarios = [
    "NORMAL",
    "LOW_RENEWABLE",
    "RESUPPLY_DELAY_4D",
    "BATTERY_DEGRADATION",
    "STORM"
]


scorecard_rows = []


# ------------------------------------------------------------
# HELPER
# ------------------------------------------------------------

def get_baseline_security_metrics(
    baseline_result,
    state,
    cqrm_result
):
    """
    Extract only security-relevant baseline metrics.

    We deliberately do NOT use flexible-load utilization as
    the primary comparison metric here.
    """

    return {
        "fuel_used_l":
            baseline_result["fuel_used_l"],

        "min_soc_pct":
            baseline_result["minimum_soc_pct"],

        "final_soc_pct":
            baseline_result["final_soc_pct"],

        "critical_coverage_pct":
            baseline_result[
                "critical_coverage_pct"
            ],

        "power_balance_error_kw":
            baseline_result[
                "max_power_balance_error_kw"
            ],

        "unserved_load_kwh":
            baseline_result[
                "unserved_load_kwh"
            ],

        # Baseline deliberately has NO CQRM awareness.
        "resupply_margin_days":
            np.nan
    }


# ------------------------------------------------------------
# RUN
# ------------------------------------------------------------

for scenario_name in scorecard_scenarios:

    kwargs = {}

    if scenario_name == "BATTERY_DEGRADATION":
        kwargs["battery_soh_override"] = 75.0

    result = run_polar_ems_scenario_v2(
        scenario=scenario_name,
        **kwargs
    )

    state = result["state"]
    profile = result["scenario_profile"]
    cqrm = result["cqrm"]
    opt = result["optimizer"]
    safety = result["safety"]
    final = result["final_decision"]


    # --------------------------------------------------------
    # BASELINE
    # --------------------------------------------------------

    baseline = run_fair_baseline(
        scenario_profile=profile,
        state=state,
        horizon_hours=168,
        fixed_reserve_soc_pct=20.0
    )

    base_metrics = get_baseline_security_metrics(
        baseline_result=baseline,
        state=state,
        cqrm_result=cqrm
    )


    # --------------------------------------------------------
    # POLAR-EMS
    # --------------------------------------------------------

    if opt["status"] == "OPTIMAL":

        polar_metrics = {
            "fuel_used_l":
                opt["fuel_used_l"],

            "min_soc_pct":
                opt["minimum_soc_pct"],

            "final_soc_pct":
                opt["final_battery_soc_pct"],

            "critical_coverage_pct":
                opt["critical_load_coverage_pct"],

            "power_balance_error_kw":
                opt["max_power_balance_error_kw"],

            "unserved_load_kwh":
                0.0,

            "resupply_margin_days":
                cqrm["cqrm"]
        }

    else:

        polar_metrics = {
            "fuel_used_l": np.nan,
            "min_soc_pct": np.nan,
            "final_soc_pct": np.nan,
            "critical_coverage_pct": np.nan,
            "power_balance_error_kw": np.nan,
            "unserved_load_kwh": np.nan,
            "resupply_margin_days": cqrm["cqrm"]
        }


    # --------------------------------------------------------
    # SECURITY DIFFERENCES
    # --------------------------------------------------------

    if np.isfinite(
        polar_metrics["fuel_used_l"]
    ):

        fuel_difference = (
            base_metrics["fuel_used_l"]
            -
            polar_metrics["fuel_used_l"]
        )

    else:

        fuel_difference = np.nan


    if np.isfinite(
        polar_metrics["final_soc_pct"]
    ):

        final_soc_gain = (
            polar_metrics["final_soc_pct"]
            -
            base_metrics["final_soc_pct"]
        )

    else:

        final_soc_gain = np.nan


    if np.isfinite(
        polar_metrics["min_soc_pct"]
    ):

        min_soc_gain = (
            polar_metrics["min_soc_pct"]
            -
            base_metrics["min_soc_pct"]
        )

    else:

        min_soc_gain = np.nan


    if np.isfinite(
        polar_metrics["critical_coverage_pct"]
    ):

        critical_coverage_difference = (
            polar_metrics[
                "critical_coverage_pct"
            ]
            -
            base_metrics[
                "critical_coverage_pct"
            ]
        )

    else:

        critical_coverage_difference = np.nan


    # --------------------------------------------------------
    # SECURITY FLAG
    # --------------------------------------------------------

    polar_safe = (
        safety["status"] == "SAFE"
        and
        opt["status"] == "OPTIMAL"
    )

    # --------------------------------------------------------
    # STORE
    # --------------------------------------------------------

    scorecard_rows.append({

        "Scenario":
            scenario_name,

        # --------------------------
        # CQRM
        # --------------------------

        "POLAR_CQRM_days":
            cqrm["cqrm"],

        "POLAR_Risk":
            cqrm["risk_level"],

        "POLAR_Reserve_SOC_pct":
            result[
                "reserve_policy"
            ][
                "required_reserve_soc_pct"
            ],

        # --------------------------
        # Critical Load
        # --------------------------

        "POLAR_Critical_Coverage_pct":
            polar_metrics[
                "critical_coverage_pct"
            ],

        "BASELINE_Critical_Coverage_pct":
            base_metrics[
                "critical_coverage_pct"
            ],

        "Critical_Coverage_Difference_pct":
            critical_coverage_difference,

        # --------------------------
        # Battery security
        # --------------------------

        "POLAR_Min_SOC_pct":
            polar_metrics[
                "min_soc_pct"
            ],

        "BASELINE_Min_SOC_pct":
            base_metrics[
                "min_soc_pct"
            ],

        "POLAR_Final_SOC_pct":
            polar_metrics[
                "final_soc_pct"
            ],

        "BASELINE_Final_SOC_pct":
            base_metrics[
                "final_soc_pct"
            ],

        "Final_SOC_Gain_pct_points":
            final_soc_gain,

        "Min_SOC_Gain_pct_points":
            min_soc_gain,

        # --------------------------
        # Fuel
        # --------------------------

        "POLAR_Fuel_L":
            polar_metrics[
                "fuel_used_l"
            ],

        "BASELINE_Fuel_L":
            base_metrics[
                "fuel_used_l"
            ],

        "Fuel_Difference_L":
            fuel_difference,

        # --------------------------
        # Physical validity
        # --------------------------

        "POLAR_Balance_Error_kW":
            polar_metrics[
                "power_balance_error_kw"
            ],

        "BASELINE_Balance_Error_kW":
            base_metrics[
                "power_balance_error_kw"
            ],

        # --------------------------
        # Resupply resilience
        # --------------------------

        "POLAR_Resupply_Margin_days":
            polar_metrics[
                "resupply_margin_days"
            ],

        # --------------------------
        # Decision status
        # --------------------------

        "POLAR_Optimizer":
            opt["status"],

        "POLAR_Safety":
            safety["status"],

        "POLAR_Final_Decision":
            final["final_decision"],

        "POLAR_Operationally_Acceptable":
            polar_safe
    })


# ------------------------------------------------------------
# DATAFRAME
# ------------------------------------------------------------

security_scorecard_df = pd.DataFrame(
    scorecard_rows
)


print("=" * 140)
print("POLAR-EMS — MATCHED ENERGY-SECURITY SCORECARD")
print("=" * 140)

display(
    security_scorecard_df
)


# ============================================================
# SECURITY SUMMARY
# ============================================================

print("\n" + "=" * 140)
print("SECURITY SUMMARY")
print("=" * 140)

print(
    "\nCritical-load coverage:"
)

print(
    security_scorecard_df[
        [
            "Scenario",
            "POLAR_Critical_Coverage_pct",
            "BASELINE_Critical_Coverage_pct"
        ]
    ].to_string(index=False)
)


print(
    "\nBattery reserve:"
)

print(
    security_scorecard_df[
        [
            "Scenario",
            "POLAR_Min_SOC_pct",
            "BASELINE_Min_SOC_pct",
            "POLAR_Final_SOC_pct",
            "BASELINE_Final_SOC_pct"
        ]
    ].to_string(index=False)
)


print(
    "\nResupply resilience:"
)

print(
    security_scorecard_df[
        [
            "Scenario",
            "POLAR_CQRM_days",
            "POLAR_Resupply_Margin_days",
            "POLAR_Risk",
            "POLAR_Final_Decision"
        ]
    ].to_string(index=False)
)


print(
    "\nFuel:"
)

print(
    security_scorecard_df[
        [
            "Scenario",
            "POLAR_Fuel_L",
            "BASELINE_Fuel_L",
            "Fuel_Difference_L"
        ]
    ].to_string(index=False)
)


# ============================================================
# HARD VALIDATION CHECKS
# ============================================================

# Every feasible POLAR-EMS plan must preserve the 20% SOC floor.
feasible_polar = security_scorecard_df[
    security_scorecard_df[
        "POLAR_Optimizer"
    ] == "OPTIMAL"
]

if len(feasible_polar) > 0:

    assert (
        feasible_polar[
            "POLAR_Min_SOC_pct"
        ]
        >= 20.0 - 1e-6
    ).all()

# Critical-load coverage for feasible optimizer plans.
if len(feasible_polar) > 0:

    assert (
        feasible_polar[
            "POLAR_Critical_Coverage_pct"
        ]
        >= 100.0 - 1e-6
    ).all()

# Physical power balance.
if len(feasible_polar) > 0:

    assert (
        feasible_polar[
            "POLAR_Balance_Error_kW"
        ].abs()
        < 1e-6
    ).all()


# Resupply delay should worsen CQRM.
normal_cqrm = security_scorecard_df[
    security_scorecard_df["Scenario"] == "NORMAL"
]["POLAR_CQRM_days"].iloc[0]

delay_cqrm = security_scorecard_df[
    security_scorecard_df["Scenario"] ==
    "RESUPPLY_DELAY_4D"
]["POLAR_CQRM_days"].iloc[0]

assert delay_cqrm < normal_cqrm


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

security_scorecard_df.to_csv(
    "/content/polar_ems_matched_energy_security_scorecard.csv",
    index=False
)

print(
    "\n✅ Saved:"
    "\n/content/polar_ems_matched_energy_security_scorecard.csv"
)

print("\n" + "=" * 140)
print("✅ STEP 24 COMPLETE")
print("=" * 140)

POLAR-EMS — MATCHED ENERGY-SECURITY SCORECARD


,Scenario,POLAR_CQRM_days,POLAR_Risk,POLAR_Reserve_SOC_pct,POLAR_Critical_Coverage_pct,BASELINE_Critical_Coverage_pct,Critical_Coverage_Difference_pct,POLAR_Min_SOC_pct,BASELINE_Min_SOC_pct,POLAR_Final_SOC_pct,...,POLAR_Fuel_L,BASELINE_Fuel_L,Fuel_Difference_L,POLAR_Balance_Error_kW,BASELINE_Balance_Error_kW,POLAR_Resupply_Margin_days,POLAR_Optimizer,POLAR_Safety,POLAR_Final_Decision,POLAR_Operationally_Acceptable
0,NORMAL,0.491667,CAUTION,55.000000,100.0,100.0,0.0,20.0,20.0,58.350002,...,2498.842542,15686.535860,13187.693318,5.684342e-14,302.97,0.491667,OPTIMAL,SAFE,ACCEPT_PLAN,True
1,LOW_RENEWABLE,-4.175000,CRITICAL,77.087500,100.0,100.0,0.0,20.0,20.0,77.087500,...,5515.547155,20821.062860,15305.515705,2.842171e-14,302.97,-4.175000,OPTIMAL,UNSAFE,REJECT_PLAN,False
2,RESUPPLY_DELAY_4D,-3.508333,CRITICAL,76.754167,100.0,100.0,0.0,20.0,20.0,76.754167,...,2498.842542,15686.535860,13187.693318,5.684342e-14,302.97,-3.508333,OPTIMAL,UNSAFE,REJECT_PLAN,False
3,BATTERY_DEGRADATION,0.491667,CAUTION,55.000000,100.0,100.0,0.0,20.0,20.0,55.000000,...,2626.398408,15671.565053,13045.166644,5.684342e-14,302.97,0.491667,OPTIMAL,SAFE,ACCEPT_PLAN,True
4,STORM,-4.716667,CRITICAL,77.358333,NaN,100.0,NaN,NaN,20.0,NaN,...,NaN,22085.157360,NaN,NaN,302.97,-4.716667,INFEASIBLE,UNSAFE,REJECT_PLAN,False



SECURITY SUMMARY

Critical-load coverage:
           Scenario  POLAR_Critical_Coverage_pct  BASELINE_Critical_Coverage_pct
             NORMAL                        100.0                           100.0
      LOW_RENEWABLE                        100.0                           100.0
  RESUPPLY_DELAY_4D                        100.0                           100.0
BATTERY_DEGRADATION                        100.0                           100.0
              STORM                          NaN                           100.0

Battery reserve:
           Scenario  POLAR_Min_SOC_pct  BASELINE_Min_SOC_pct  POLAR_Final_SOC_pct  BASELINE_Final_SOC_pct
             NORMAL               20.0                  20.0            58.350002                    20.0
      LOW_RENEWABLE               20.0                  20.0            77.087500                    20.0
  RESUPPLY_DELAY_4D               20.0                  20.0            76.754167                    20.0
BATTERY_DEGRADATION          